In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Setup, Imports, and Global Configuration                             ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

!pip install -q timm einops albumentations opencv-python-headless

import os, time, random, copy, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
import cv2
from tqdm.auto import tqdm
from scipy.ndimage import zoom

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2
from einops import rearrange

from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, roc_curve, 
                             confusion_matrix, classification_report, precision_recall_curve,
                             average_precision_score, matthews_corrcoef, cohen_kappa_score,
                             log_loss, brier_score_loss)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

# ─── Seeds ─────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ─── Device & Paths ───────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTDIR = Path('/kaggle/working')
CKDIR = OUTDIR / 'checkpoints'
PLTDIR = OUTDIR / 'plots'
for d in [CKDIR, PLTDIR]: 
    d.mkdir(exist_ok=True)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.JPG', '.JPEG', '.PNG'}

# ✅ DIABETIC RETINOPATHY CONFIG - 5 CLASSES
class_names = ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferate_DR']
NUM_CLASSES = 5
COLORS = ['#2ecc71', '#f1c40f', '#3498db', '#e67e22', '#e74c3c']

print(f"✅ Setup complete on {DEVICE}")
print(f"   Classes ({NUM_CLASSES}): {class_names}")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Hyperparameters                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

HPARAMS = {
    'img_size': 224,
    'batch_size': 32,
    'num_epochs': 40,
    'lr_backbone': 1e-5,
    'lr_head': 1e-4,
    'weight_decay': 1e-4,
    'warmup_epochs': 5,
    'freeze_epochs': 8,
    'patience': 10,
    'aug_level': 'medium',
    'd_model': 384,
    'nhead': 6,
    'n_layers': 4,
    'dim_ffn': 1536,
    'head_dropout': 0.3,
    'label_smoothing': 0.05,
    'mixup_alpha': 0.2,
    'cutmix_alpha': 1.0,
    'use_mixup': True,
    'use_cutmix': True,
    'mixup_prob': 0.3,
    'backbone': 'convnext_base',
    'num_classes': NUM_CLASSES,
    'class_names': class_names
}
print("✅ Hyperparameters ready (5-class DR)")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Data Augmentations (Safe for Retina Images)                          ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

def get_retina_transforms(split, img_size=224, level='medium'):
    norm = [A.Normalize(mean=MEAN, std=STD), ToTensorV2()]
    if split != 'train':
        return A.Compose([A.Resize(height=img_size, width=img_size)] + norm)
    
    # Retina-specific augmentations
    base = [
        A.Resize(height=img_size, width=img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5, 
                          border_mode=cv2.BORDER_CONSTANT, value=0),
    ]
    light_med = [
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.GaussNoise(var_limit=(10.0, 30.0), p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    ]
    
    return A.Compose(base + light_med + norm)

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Dataset Class                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

class RetinaDataset(Dataset):
    def __init__(self, samples, class_names, transform=None):
        self.samples = samples
        self.transform = transform
        self.class_names = class_names
        self.class_to_idx = {n: i for i, n in enumerate(class_names)}
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        p, l = self.samples[idx]
        l = self.class_to_idx.get(l, l)
        
        # Read image (retina images are RGB)
        img = cv2.imread(p)
        if img is None:
            img = np.array(Image.open(p).convert('RGB'))
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            img = self.transform(image=img)['image']
        return img.float(), l
    
    def class_weights(self):
        labels = [self.class_to_idx.get(l, l) for _, l in self.samples]
        counts = torch.bincount(torch.tensor(labels), minlength=len(self.class_names)).float()
        beta = 0.9999
        effective_num = 1.0 - torch.pow(beta, counts)
        weights = ((1.0 - beta) / (effective_num + 1e-8)) / len(self.class_names) * len(self.class_names)
        return weights

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Load Dataset                                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

def load_retina_dataset():
    print("📂 Loading Diabetic Retinopathy dataset...")
    samples = []
    
    # Try multiple possible paths
    possible_paths = [
        Path('/kaggle/input/datasets/nikitachaulagain/dataasets/Diabetic Retinopathy 224x224 (2019 Data)/Diabetic Retinopathy 224x224 (2019 Data)/colored_images'),
        Path('/kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-2019-data/colored_images'),
        Path('/kaggle/input/diabetic-retinopathy-224x224-2019-data/colored_images'),
    ]
    
    base = None
    for path in possible_paths:
        if path.exists():
            base = path
            print(f"✅ Found dataset at: {base}")
            break
    
    if base is None:
        print("❌ Dataset not found! Available directories:")
        input_dir = Path('/kaggle/input')
        if input_dir.exists():
            for item in input_dir.iterdir():
                print(f"  - {item.name}")
                if item.is_dir():
                    for sub in item.iterdir():
                        if sub.is_dir():
                            print(f"    └─ {sub.name}")
        return []
    
    # Map folder names to class names
    folder_to_class = {
        '0': 'No_DR',
        '1': 'Mild',
        '2': 'Moderate',
        '3': 'Severe',
        '4': 'Proliferate_DR',
        '0 - No_DR': 'No_DR',
        '1 - Mild': 'Mild',
        '2 - Moderate': 'Moderate',
        '3 - Severe': 'Severe',
        '4 - Proliferate_DR': 'Proliferate_DR',
        'No_DR': 'No_DR',
        'Mild': 'Mild',
        'Moderate': 'Moderate',
        'Severe': 'Severe',
        'Proliferate_DR': 'Proliferate_DR'
    }
    
    # Collect images
    for folder in base.iterdir():
        if not folder.is_dir():
            continue
        folder_name = folder.name.strip()
        class_name = folder_to_class.get(folder_name, None)
        
        if class_name is None:
            # Try to find class from folder name containing number
            for key in folder_to_class:
                if key in folder_name:
                    class_name = folder_to_class[key]
                    break
        
        if class_name is None:
            print(f"⚠️ Skipping unknown folder: {folder_name}")
            continue
        
        count = 0
        for ext in VALID_EXTS:
            for p in folder.rglob(f'*{ext}'):
                samples.append((str(p), class_name))
                count += 1
        if count > 0:
            print(f"   {class_name}: {count} images")
    
    print(f"✅ Total images: {len(samples)}")
    return samples

print("📦 Loading Diabetic Retinopathy dataset...")
samples = load_retina_dataset()

if not samples:
    raise ValueError("No samples loaded! Please check dataset path.")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Train/Val/Test Split                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

labels = [class_names.index(l) for _, l in samples]

# Stratified split
train_s, temp_s = train_test_split(samples, test_size=0.3, stratify=labels, random_state=SEED)
val_s, test_s = train_test_split(temp_s, test_size=0.5, 
                                 stratify=[class_names.index(l) for _, l in temp_s], 
                                 random_state=SEED)

df_all = pd.DataFrame(samples, columns=['path', 'label'])

print("\n" + "="*60)
print("📊 DIABETIC RETINOPATHY DATASET ANALYSIS")
print("="*60)
print(f"\n🔹 Total Images: {len(df_all)} | Classes: {len(class_names)}")
print(f"🔹 Split: 70% Train ({len(train_s)}) / 15% Val ({len(val_s)}) / 15% Test ({len(test_s)})")
print("\n🔹 Class Distribution (Overall):")
for cls in class_names:
    count = len(df_all[df_all['label'] == cls])
    pct = count/len(df_all)*100 if len(df_all) > 0 else 0
    print(f"   - {cls:15s}: {count:5d} images ({pct:.1f}%)")
print("="*60 + "\n")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Create Dataloaders                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

# Create datasets
train_ds = RetinaDataset(train_s, class_names, get_retina_transforms('train'))
val_ds = RetinaDataset(val_s, class_names, get_retina_transforms('val'))
test_ds = RetinaDataset(test_s, class_names, get_retina_transforms('test'))

# Class weights for imbalance
class_weights = train_ds.class_weights()
sample_weights = [class_weights[class_names.index(l)] for _, l in train_s]

loaders = {
    'train': DataLoader(train_ds, batch_size=HPARAMS['batch_size'],
                        sampler=WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True),
                        num_workers=2, pin_memory=True),
    'val': DataLoader(val_ds, batch_size=HPARAMS['batch_size'], shuffle=False, 
                      num_workers=2, pin_memory=True),
    'test': DataLoader(test_ds, batch_size=HPARAMS['batch_size'], shuffle=False,
                       num_workers=2, pin_memory=True)
}

print("✅ Dataloaders created")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Model Architecture (Multi-Class)                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

# ─── Model classes (same as before but with num_classes=5) ──────────────────────
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0): 
        super().__init__(); 
        self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training: 
            return x
        return x * (torch.rand((x.shape[0], 1, 1), dtype=torch.float32, device=x.device) * float(1.0 - self.drop_prob))

class WindowAttention(nn.Module):
    def __init__(self, d_model, nhead, window_size=7, dropout=0.1):
        super().__init__()
        self.nhead, self.window_size, self.head_dim = nhead, window_size, d_model // nhead
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.proj_drop, self.attn_weights = nn.Dropout(dropout), nn.Dropout(dropout), None
    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.reshape(B, H, W, C)
        pad_h, pad_w = (self.window_size - H % self.window_size) % self.window_size, (self.window_size - W % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0: 
            x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        Hp, Wp = x.shape[1], x.shape[2]
        x = x.reshape(B, Hp//self.window_size, self.window_size, Wp//self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(-1, self.window_size**2, C)
        qkv = self.qkv(x).reshape(-1, self.window_size**2, 3, self.nhead, self.head_dim).permute(2,0,3,1,4)
        attn = (qkv[0] @ qkv[1].transpose(-2,-1)) * self.scale
        attn = attn.softmax(dim=-1)
        self.attn_weights = attn.detach()
        x = self.proj_drop(self.proj((self.attn_drop(attn) @ qkv[2]).transpose(1,2).reshape(-1, self.window_size**2, C)))
        x = x.reshape(B, Hp//self.window_size, Wp//self.window_size, self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(B, Hp, Wp, C)
        return x[:, :H, :W, :].reshape(B, -1, C) if pad_h > 0 or pad_w > 0 else x.reshape(B, -1, C)

class TransformerBlockWithWindow(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim, window_size=7, attn_drop=0.1, ffn_drop=0.1, drop_path=0.0):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.global_attn = nn.MultiheadAttention(d_model, nhead, dropout=attn_drop, batch_first=True)
        self.window_attn = WindowAttention(d_model, nhead, window_size, attn_drop)
        self.gate = nn.Sequential(nn.Linear(d_model*2, d_model), nn.Sigmoid())
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(ffn_drop), nn.Linear(ffn_dim, d_model), nn.Dropout(ffn_drop))
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.attn_weights, self.window_attn_weights = None, None
    def forward(self, x, H, W):
        x_norm = self.norm1(x)
        global_out, global_attn = self.global_attn(x_norm, x_norm, x_norm, need_weights=True)
        self.attn_weights = global_attn.detach()
        window_out = self.window_attn(x_norm[:, 1:, :], H, W)
        self.window_attn_weights = self.window_attn.attn_weights
        gate = self.gate(torch.cat([global_out[:, 1:, :], window_out], dim=-1))
        combined = torch.cat([global_out[:, :1, :], gate * global_out[:, 1:, :] + (1-gate) * window_out], dim=1)
        x = x + self.drop_path(combined)
        return x + self.drop_path(self.ffn(self.norm2(x)))

class ImprovedBackbone(nn.Module):
    def __init__(self): 
        super().__init__()
        self.backbone = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
        self.feature_extractor = self.backbone.features
    def forward(self, x): 
        return self.feature_extractor(x)

class EnhancedHViT(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536):
        super().__init__()
        self.d_model = d_model
        self.img_size = img_size
        self.num_classes = num_classes
        self.backbone = ImprovedBackbone()
        with torch.no_grad():
            _, c, h, w = self.backbone(torch.zeros(1, 3, img_size, img_size)).shape
        self.spatial_h, self.spatial_w = h, w
        self.proj = nn.Sequential(
            nn.Conv2d(c, d_model, 1, bias=False),
            nn.BatchNorm2d(d_model),
            nn.GELU(),
            nn.Conv2d(d_model, d_model, 3, padding=1, bias=False),
            nn.BatchNorm2d(d_model),
            nn.GELU()
        )
        self.norm_proj = nn.LayerNorm(d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, h*w + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        dpr = [x.item() for x in torch.linspace(0, 0.15, n_layers)]
        self.encoder = nn.ModuleList([
            TransformerBlockWithWindow(d_model, nhead, dim_ffn, min(7, h, w), drop_path=dpr[i]) 
            for i in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model//2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(d_model//2, d_model//4),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(d_model//4, num_classes)
        )
    def forward(self, x):
        B = x.size(0)
        tokens = rearrange(self.proj(self.backbone(x)), 'b d h w -> b (h w) d')
        tokens = self.norm_proj(tokens)
        tokens = torch.cat([self.cls_token.expand(B,-1,-1), tokens], dim=1) + self.pos_embed
        for block in self.encoder:
            tokens = block(tokens, self.spatial_h, self.spatial_w)
        return self.head(self.norm(tokens)[:, 0])
    
    def get_attentions(self):
        g, w = [], []
        for b in self.encoder:
            if b.attn_weights is not None: 
                g.append(b.attn_weights)
            if b.window_attn_weights is not None: 
                w.append(b.window_attn_weights)
        return g, w
    
    def unfreeze_backbone_partial(self, n=3):
        for p in self.backbone.feature_extractor.parameters(): 
            p.requires_grad_(False)
        for child in list(self.backbone.feature_extractor.children())[-n:]:
            for p in child.parameters(): 
                p.requires_grad_(True)

print("✅ Model architecture ready (5-class DR)")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Training Functions                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

class SmoothCELoss(nn.Module):
    def __init__(self, label_smoothing=0.05):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    def forward(self, inputs, targets):
        return self.criterion(inputs, targets)

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    W, H = x.size(2), x.size(3)
    r = np.sqrt(1.0 - lam)
    cw, ch = int(W*r), int(H*r)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = np.clip(cx - cw//2, 0, W), np.clip(cy - ch//2, 0, H)
    x2, y2 = np.clip(cx + cw//2, 0, W), np.clip(cy + ch//2, 0, H)
    mixed = x.clone()
    mixed[:, :, x1:x2, y1:y2] = x[idx, :, x1:x2, y1:y2]
    return mixed, y, y[idx], 1 - (x2-x1)*(y2-y1)/(W*H)

def train_one_epoch(model, loader, optimizer, scaler, criterion, epoch):
    model.train()
    total_loss, preds, labs = 0, [], []
    
    for images, labels in tqdm(loader, desc=f'Epoch {epoch+1}', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        
        # Mixup/Cutmix
        mixed = False
        if np.random.random() < HPARAMS['mixup_prob']:
            if np.random.random() < 0.5:
                images, la, lb, lam = mixup_data(images, labels, HPARAMS['mixup_alpha'])
                mixed = True
            else:
                images, la, lb, lam = cutmix_data(images, labels, HPARAMS['cutmix_alpha'])
                mixed = True
        
        with autocast():
            out = model(images)
            if mixed:
                loss = lam * criterion(out, la) + (1-lam) * criterion(out, lb)
            else:
                loss = criterion(out, labels)
                preds.extend(out.argmax(1).cpu().numpy())
                labs.extend(labels.cpu().numpy())
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    
    acc = accuracy_score(labs, preds) if labs else 0
    f1 = f1_score(labs, preds, average='weighted', zero_division=0) if labs else 0
    return total_loss/len(loader), acc, f1

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, preds, labs, probs = 0, [], [], []
    
    for images, labels in tqdm(loader, desc='Validating', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with autocast():
            out = model(images)
            loss = criterion(out, labels)
        total_loss += loss.item()
        p = F.softmax(out, dim=1)
        preds.extend(out.argmax(1).cpu().numpy())
        labs.extend(labels.cpu().numpy())
        probs.extend(p.cpu().numpy())
    
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='weighted', zero_division=0)
    return total_loss/len(loader), acc, f1, preds, labs, probs

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Main Training Loop                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

print("\n🚀 Starting Training for Diabetic Retinopathy...")
print("="*60)

# Initialize model
model = EnhancedHViT(num_classes=NUM_CLASSES).to(DEVICE)

# Loss
criterion = SmoothCELoss(label_smoothing=HPARAMS['label_smoothing'])

# Optimizer with separate learning rates
backbone_params = [p for n, p in model.named_parameters() if 'backbone' in n]
head_params = [p for n, p in model.named_parameters() if 'backbone' not in n]

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': HPARAMS['lr_backbone']},
    {'params': head_params, 'lr': HPARAMS['lr_head']}
], weight_decay=HPARAMS['weight_decay'])

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=HPARAMS['num_epochs']//3, T_mult=2, eta_min=1e-7
)
scaler = GradScaler()

# Training loop
best_f1 = 0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': [], 'lr': []}

for epoch in range(HPARAMS['num_epochs']):
    # Unfreeze backbone gradually
    if epoch == HPARAMS['freeze_epochs']:
        print("\n🔄 Unfreezing backbone...")
        model.unfreeze_backbone_partial(3)
        optimizer.param_groups[0]['lr'] = HPARAMS['lr_backbone']
    
    # Train
    train_loss, train_acc, train_f1 = train_one_epoch(model, loaders['train'], optimizer, scaler, criterion, epoch)
    
    # Validate
    val_loss, val_acc, val_f1, _, _, _ = validate(model, loaders['val'], criterion)
    scheduler.step()
    
    # Log history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.3f}, Train F1={train_f1:.3f} | Val Acc={val_acc:.3f}, Val F1={val_f1:.3f}")
    
    # Save best model
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save({
            'model': model.state_dict(),
            'f1': val_f1,
            'acc': val_acc,
            'epoch': epoch + 1,
            'class_names': class_names
        }, CKDIR / 'best_model.pth')
        print(f"   ✅ Best model saved (Val F1={val_f1:.3f})")
    else:
        patience_counter += 1
        if patience_counter >= HPARAMS['patience']:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

print("\n✅ Training complete!")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Final Evaluation on Test Set                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

print("\n📊 Final Evaluation on Test Set...")
model.load_state_dict(torch.load(CKDIR/'best_model.pth', map_location=DEVICE, weights_only=False)['model'])
model.eval()

all_p, all_l, all_pr = [], [], []
with torch.no_grad():
    for im, l in tqdm(loaders['test'], desc='Testing'):
        im = im.to(DEVICE)
        out = model(im)
        p = F.softmax(out, dim=1)
        all_p.extend(out.argmax(1).cpu().numpy())
        all_l.extend(l.numpy())
        all_pr.extend(p.cpu().numpy())

all_p = np.array(all_p)
all_l = np.array(all_l)
all_pr = np.array(all_pr)

# ─── Metrics ──────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📊 COMPREHENSIVE EVALUATION METRICS")
print("="*60)

# Primary metrics
test_acc = accuracy_score(all_l, all_p)
test_f1_macro = f1_score(all_l, all_p, average='macro')
test_f1_weighted = f1_score(all_l, all_p, average='weighted')
test_f1_micro = f1_score(all_l, all_p, average='micro')

# ROC-AUC (multi-class)
try:
    test_auc = roc_auc_score(all_l, all_pr, multi_class='ovr', average='weighted')
except:
    test_auc = 0.0

# Additional metrics
mcc = matthews_corrcoef(all_l, all_p)
kappa = cohen_kappa_score(all_l, all_p)

# Confusion matrix
cm = confusion_matrix(all_l, all_p)

print(f"\n🎯 PRIMARY METRICS:")
print(f"   Accuracy:  {test_acc:.4f}")
print(f"   F1-Macro:  {test_f1_macro:.4f}")
print(f"   F1-Weighted: {test_f1_weighted:.4f}")
print(f"   F1-Micro:  {test_f1_micro:.4f}")
print(f"   AUC-ROC:   {test_auc:.4f}")

print(f"\n📈 ADDITIONAL METRICS:")
print(f"   MCC:       {mcc:.4f}")
print(f"   Cohen's Kappa: {kappa:.4f}")

print(f"\n📊 CONFUSION MATRIX:")
print(f"{'':18s}", end="")
for cls in class_names:
    print(f"{cls[:8]:>10s}", end="")
print()
for i, cls in enumerate(class_names):
    print(f"{cls:18s}", end="")
    for j in range(len(class_names)):
        print(f"{cm[i,j]:>10d}", end="")
    print()

print(f"\n📋 CLASSIFICATION REPORT:")
print(classification_report(all_l, all_p, target_names=class_names))

# ─── Per-class metrics ──────────────────────────────────────────────────────────
print(f"\n🔬 PER-CLASS DETAILS:")
for i, cls in enumerate(class_names):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    tn = cm.sum() - tp - fp - fn
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
    
    print(f"   {cls:15s}: Sens={sensitivity:.3f}, Spec={specificity:.3f}, Prec={precision:.3f}, F1={f1:.3f}")

print("="*60)

# ─── Save test predictions ──────────────────────────────────────────────────────
results_df = pd.DataFrame({
    'true_label': all_l,
    'predicted': all_p,
    **{f'prob_{cls}': all_pr[:, i] for i, cls in enumerate(class_names)}
})
results_df.to_csv(OUTDIR / 'test_predictions.csv', index=False)
print(f"\n💾 Saved test predictions to: {OUTDIR / 'test_predictions.csv'}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 13-15 — Interactive Prediction + Batch Predict + Quick Test               ║
# ║  (Diabetic Retinopathy - 5 Classes)                                             ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink
import io
from PIL import Image as PILImage

# ─── LOAD MODEL ──────────────────────────────────────────────────────────────────
print("📦 Loading Diabetic Retinopathy model from checkpoint...")

# Check if model exists, if not create it
try:
    model
except NameError:
    print("⚠️ Model not found. Creating new model...")
    model = EnhancedHViT(num_classes=5, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536).to(DEVICE)

# Load weights
checkpoint_path = CKDIR / 'best_model.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"✅ Model loaded from: {checkpoint_path}")
    if 'class_names' in checkpoint:
        CLASS_NAMES = checkpoint['class_names']
    else:
        CLASS_NAMES = ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferate_DR']
else:
    print(f"❌ Checkpoint not found at: {checkpoint_path}")
    print("Please train the model first or check the path.")
    CLASS_NAMES = ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferate_DR']

model.eval()
IMG_SIZE = 224

# ─── PREDICTION FUNCTION ─────────────────────────────────────────────────────────
def predict_single_image(image_array):
    """Run inference on a single numpy RGB image."""
    img_resized = cv2.resize(image_array, (IMG_SIZE, IMG_SIZE))
    img_tensor = torch.from_numpy(img_resized).float().permute(2, 0, 1) / 255.0
    img_tensor = (img_tensor - torch.tensor(MEAN).view(3, 1, 1)) / torch.tensor(STD).view(3, 1, 1)
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        with autocast():
            logits = model(img_tensor)
            probs = F.softmax(logits, dim=1)
            conf, pred_idx = probs.max(dim=1)
    
    return pred_idx.item(), conf.item(), probs.cpu().numpy()[0]

def generate_prediction_plot(image_rgb, pred_idx, confidence, probs):
    """Create a nice prediction visualization for retina images."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Left: Image with prediction
    axes[0].imshow(image_rgb)
    color = 'green' if confidence > 0.8 else ('orange' if confidence > 0.5 else 'red')
    
    # Get severity level
    severity_map = {
        'No_DR': '✅ Normal',
        'Mild': '⚠️ Mild DR',
        'Moderate': '⚠️ Moderate DR',
        'Severe': '🔴 Severe DR',
        'Proliferate_DR': '🚨 Proliferative DR'
    }
    status = severity_map.get(CLASS_NAMES[pred_idx], CLASS_NAMES[pred_idx])
    
    axes[0].set_title(f"{status}\nConfidence: {confidence:.2%}", 
                      fontsize=14, fontweight='bold', color=color)
    axes[0].axis('off')
    
    # Right: Bar chart of all class probabilities
    colors_bar = ['#2ecc71' if i == pred_idx else '#95a5a6' for i in range(len(CLASS_NAMES))]
    bars = axes[1].barh(CLASS_NAMES, probs * 100, color=colors_bar, edgecolor='black', linewidth=0.5)
    axes[1].set_xlabel('Probability (%)', fontsize=12)
    axes[1].set_title('Class Probabilities', fontsize=14, fontweight='bold')
    axes[1].set_xlim(0, 105)
    
    for bar, prob in zip(bars, probs):
        axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
                     f'{prob:.1f}%', va='center', fontsize=11, fontweight='bold')
    
    axes[1].grid(axis='x', alpha=0.3)
    plt.tight_layout()
    return fig

# ─── WIDGET SETUP ──────────────────────────────────────────────────────────────
upload_widget = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.bmp,.tiff',
    multiple=False,
    description='📤 Upload Retina Image',
    button_style='info',
    style={'button_width': '300px', 'description_width': '200px'}
)

path_input = widgets.Text(
    value='',
    placeholder='/kaggle/input/.../retina.jpg',
    description='📁 Or enter path:',
    layout={'width': '600px'}
)

predict_button = widgets.Button(
    description='🔍 Predict',
    button_style='success',
    layout={'width': '200px'}
)

output_area = widgets.Output()

def on_predict_clicked(b):
    with output_area:
        clear_output(wait=True)
        
        image_rgb = None
        source = ""
        
        # Try file upload first
        if upload_widget.value:
            uploaded_file = list(upload_widget.value.values())[0]
            content = uploaded_file['content']
            image_rgb = np.array(PILImage.open(io.BytesIO(content)).convert('RGB'))
            source = f"Uploaded: {uploaded_file['metadata']['name']}"
        
        # Try path input
        elif path_input.value.strip():
            p = Path(path_input.value.strip())
            if p.exists():
                img = cv2.imread(str(p))
                if img is not None:
                    image_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    source = f"Path: {p.name}"
            else:
                print(f"❌ File not found: {p}")
                return
        
        else:
            print("⚠️ Please upload an image OR enter a file path")
            return
        
        if image_rgb is None:
            print("❌ Could not read image")
            return
        
        # Run prediction
        print(f"🔮 Analyzing {source}...")
        pred_idx, confidence, probs = predict_single_image(image_rgb)
        
        # Display results
        severity_map = {
            'No_DR': '✅ NORMAL - No Retinopathy',
            'Mild': '⚠️ MILD - Early signs of DR',
            'Moderate': '⚠️ MODERATE - Progressive DR',
            'Severe': '🔴 SEVERE - Advanced DR',
            'Proliferate_DR': '🚨 PROLIFERATIVE - High risk'
        }
        
        print(f"\n{'='*50}")
        print(f"   PREDICTION: {CLASS_NAMES[pred_idx].upper()}")
        print(f"   {severity_map.get(CLASS_NAMES[pred_idx], '')}")
        print(f"   CONFIDENCE: {confidence:.2%}")
        print(f"{'='*50}")
        print("\nAll probabilities:")
        for name, prob in zip(CLASS_NAMES, probs):
            marker = " ◀" if name == CLASS_NAMES[pred_idx] else ""
            print(f"   {name:18s}: {prob:.4f} ({prob*100:.1f}%){marker}")
        
        # Show plot
        fig = generate_prediction_plot(image_rgb, pred_idx, confidence, probs)
        plt.show()
        
        # Save prediction plot
        pred_path = PLTDIR / f'prediction_{CLASS_NAMES[pred_idx]}_{int(confidence*100)}.png'
        fig.savefig(pred_path, dpi=150, bbox_inches='tight')
        print(f"\n💾 Saved prediction plot to: {pred_path}")

predict_button.on_click(on_predict_clicked)

# ─── DISPLAY INTERFACE ──────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════╗")
print("║        👁️ DIABETIC RETINOPATHY CLASSIFICATION - INFERENCE   ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  Upload a retina image or enter a file path, then click     ║")
print("║  'Predict' to see the DR severity classification result.    ║")
print("╚══════════════════════════════════════════════════════════════╝\n")

display(widgets.VBox([
    upload_widget,
    widgets.Label("── OR ──"),
    path_input,
    widgets.Label(""),
    predict_button,
    widgets.Label(""),
    output_area
]))

# ─── BATCH PREDICT ──────────────────────────────────────────────────────────────
def batch_predict_dr(folder_path, save_csv=True):
    """Predict on all images in a folder and optionally save results to CSV."""
    folder = Path(folder_path)
    if not folder.exists():
        print(f"❌ Folder not found: {folder}")
        return None
    
    image_files = []
    for ext in VALID_EXTS:
        image_files.extend(folder.rglob(f'*{ext}'))
    
    if not image_files:
        print(f"❌ No images found in {folder}")
        return None
    
    print(f"📁 Found {len(image_files)} images in {folder}")
    
    results = []
    correct = 0
    total = 0
    
    for img_path in tqdm(image_files, desc='Predicting'):
        # Try to infer true label from folder name
        true_label = None
        parent_name = img_path.parent.name.lower()
        for cls in CLASS_NAMES:
            if cls.lower() in parent_name or cls.replace('_', '').lower() in parent_name:
                true_label = cls
                break
        
        # Read and predict
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        pred_idx, confidence, probs = predict_single_image(img_rgb)
        pred_label = CLASS_NAMES[pred_idx]
        
        if true_label is not None:
            total += 1
            if true_label == pred_label:
                correct += 1
        
        result = {
            'path': str(img_path),
            'filename': img_path.name,
            'true_label': true_label,
            'predicted': pred_label,
            'confidence': confidence,
        }
        for i, cls in enumerate(CLASS_NAMES):
            result[f'prob_{cls}'] = probs[i]
        results.append(result)
    
    df = pd.DataFrame(results)
    
    if total > 0:
        acc = correct / total
        print(f"\n{'='*50}")
        print(f"📊 BATCH RESULTS")
        print(f"   Total: {total} images with known labels")
        print(f"   Correct: {correct}")
        print(f"   Accuracy: {acc:.2%}")
        print(f"{'='*50}")
        
        # Per-class accuracy
        print("\n📊 Per-class accuracy:")
        for cls in CLASS_NAMES:
            cls_df = df[df['true_label'] == cls]
            if len(cls_df) > 0:
                cls_acc = (cls_df['true_label'] == cls_df['predicted']).mean()
                print(f"   {cls:18s}: {cls_acc:.2%} ({len(cls_df)} images)")
        
        # Confusion Matrix
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(df['true_label'], df['predicted'], labels=CLASS_NAMES)
        print("\n📊 Confusion Matrix:")
        print(f"{'':18s}", end="")
        for cls in CLASS_NAMES:
            print(f"{cls[:8]:>10s}", end="")
        print()
        for i, cls in enumerate(CLASS_NAMES):
            print(f"{cls:18s}", end="")
            for j in range(len(CLASS_NAMES)):
                print(f"{cm[i,j]:>10d}", end="")
            print()
    
    if save_csv:
        csv_path = OUTDIR / 'dr_predictions.csv'
        df.to_csv(csv_path, index=False)
        print(f"\n💾 Saved predictions to: {csv_path}")
    
    return df

# ─── QUICK TEST ──────────────────────────────────────────────────────────────────
def quick_predict_dr(image_path):
    """Quickly predict a single retina image from path."""
    img = cv2.imread(str(image_path))
    if img is None:
        print(f"❌ Could not read image: {image_path}")
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pred_idx, confidence, probs = predict_single_image(img_rgb)
    
    severity_map = {
        'No_DR': '✅ NORMAL',
        'Mild': '⚠️ MILD',
        'Moderate': '⚠️ MODERATE',
        'Severe': '🔴 SEVERE',
        'Proliferate_DR': '🚨 PROLIFERATIVE'
    }
    
    print(f"\n{'='*50}")
    print(f"📁 Image: {Path(image_path).name}")
    print(f"{'='*50}")
    print(f"   PREDICTION: {CLASS_NAMES[pred_idx].upper()}")
    print(f"   {severity_map.get(CLASS_NAMES[pred_idx], '')}")
    print(f"   CONFIDENCE: {confidence:.2%}")
    print(f"{'='*50}")
    print("\nProbabilities:")
    for name, prob in zip(CLASS_NAMES, probs):
        print(f"   {name:18s}: {prob:.4f} ({prob*100:.1f}%)")
    
    # Show image with prediction
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(img_rgb)
    color = 'green' if confidence > 0.8 else ('orange' if confidence > 0.5 else 'red')
    ax[0].set_title(f"Prediction: {CLASS_NAMES[pred_idx]}\nConfidence: {confidence:.2%}", 
                    fontsize=14, fontweight='bold', color=color)
    ax[0].axis('off')
    
    colors_bar = ['#2ecc71' if i == pred_idx else '#95a5a6' for i in range(len(CLASS_NAMES))]
    bars = ax[1].barh(CLASS_NAMES, probs * 100, color=colors_bar, edgecolor='black')
    ax[1].set_xlabel('Probability (%)')
    ax[1].set_title('Class Probabilities')
    ax[1].set_xlim(0, 105)
    for bar, prob in zip(bars, probs):
        ax[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
                   f'{prob:.1f}%', va='center', fontweight='bold')
    ax[1].grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\n✅ Diabetic Retinopathy Interactive Prediction ready!")
print("\n💡 To use:")
print("   1. Upload a retina image or enter a path to predict")
print("   2. Run batch_predict_dr('/path/to/folder') for bulk prediction")
print("   3. Run quick_predict_dr('/path/to/image.jpg') for quick single test")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  STANDALONE HEATMAP GENERATOR - DIABETIC RETINOPATHY (FIXED)                   ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path
from einops import rearrange
from scipy.ndimage import zoom
import torchvision.models as models
import warnings
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import os
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTDIR = Path('/kaggle/working')
CKDIR = OUTDIR / 'checkpoints'
PLTDIR = OUTDIR / 'plots'
PLTDIR.mkdir(exist_ok=True)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
class_names = ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferate_DR']
colors = ['#2ecc71', '#f1c40f', '#3498db', '#e67e22', '#e74c3c']
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.JPG', '.JPEG', '.PNG'}

# ─── MODEL ARCHITECTURE ──────────────────────────────────────────────────────────
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0): super().__init__(); self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training: return x
        return x * (torch.rand((x.shape[0], 1, 1), dtype=torch.float32, device=x.device) * float(1.0 - self.drop_prob))

class WindowAttention(nn.Module):
    def __init__(self, d_model, nhead, window_size=7, dropout=0.1):
        super().__init__()
        self.nhead, self.window_size, self.head_dim = nhead, window_size, d_model // nhead
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3); self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.proj_drop, self.attn_weights = nn.Dropout(dropout), nn.Dropout(dropout), None
    def forward(self, x, H, W):
        B, N, C = x.shape; x = x.reshape(B, H, W, C)
        pad_h, pad_w = (self.window_size - H % self.window_size) % self.window_size, (self.window_size - W % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0: x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        Hp, Wp = x.shape[1], x.shape[2]
        x = x.reshape(B, Hp//self.window_size, self.window_size, Wp//self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(-1, self.window_size**2, C)
        qkv = self.qkv(x).reshape(-1, self.window_size**2, 3, self.nhead, self.head_dim).permute(2,0,3,1,4)
        attn = (qkv[0] @ qkv[1].transpose(-2,-1)) * self.scale; attn = attn.softmax(dim=-1); self.attn_weights = attn.detach()
        x = self.proj_drop(self.proj((self.attn_drop(attn) @ qkv[2]).transpose(1,2).reshape(-1, self.window_size**2, C)))
        x = x.reshape(B, Hp//self.window_size, Wp//self.window_size, self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(B, Hp, Wp, C)
        return x[:, :H, :W, :].reshape(B, -1, C) if pad_h > 0 or pad_w > 0 else x.reshape(B, -1, C)

class TransformerBlockWithWindow(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim, window_size=7, attn_drop=0.1, ffn_drop=0.1, drop_path=0.0):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.global_attn = nn.MultiheadAttention(d_model, nhead, dropout=attn_drop, batch_first=True)
        self.window_attn = WindowAttention(d_model, nhead, window_size, attn_drop)
        self.gate = nn.Sequential(nn.Linear(d_model*2, d_model), nn.Sigmoid())
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(ffn_drop), nn.Linear(ffn_dim, d_model), nn.Dropout(ffn_drop))
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.attn_weights, self.window_attn_weights = None, None
    def forward(self, x, H, W):
        x_norm = self.norm1(x)
        global_out, global_attn = self.global_attn(x_norm, x_norm, x_norm, need_weights=True)
        self.attn_weights = global_attn.detach()
        window_out = self.window_attn(x_norm[:, 1:, :], H, W)
        self.window_attn_weights = self.window_attn.attn_weights
        gate = self.gate(torch.cat([global_out[:, 1:, :], window_out], dim=-1))
        combined = torch.cat([global_out[:, :1, :], gate * global_out[:, 1:, :] + (1-gate) * window_out], dim=1)
        x = x + self.drop_path(combined)
        return x + self.drop_path(self.ffn(self.norm2(x)))

class ImprovedBackbone(nn.Module):
    def __init__(self): super().__init__(); self.backbone = models.convnext_base(weights=None); self.feature_extractor = self.backbone.features
    def forward(self, x): return self.feature_extractor(x)

class EnhancedHViT(nn.Module):
    def __init__(self, num_classes=5, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536):
        super().__init__(); self.d_model = d_model; self.img_size = img_size; self.backbone = ImprovedBackbone()
        with torch.no_grad(): _, c, h, w = self.backbone(torch.zeros(1, 3, img_size, img_size)).shape
        self.spatial_h, self.spatial_w = h, w
        self.proj = nn.Sequential(nn.Conv2d(c, d_model, 1, bias=False), nn.BatchNorm2d(d_model), nn.GELU(), nn.Conv2d(d_model, d_model, 3, padding=1, bias=False), nn.BatchNorm2d(d_model), nn.GELU())
        self.norm_proj = nn.LayerNorm(d_model); self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, h*w + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02); nn.init.trunc_normal_(self.pos_embed, std=0.02)
        dpr = [x.item() for x in torch.linspace(0, 0.15, n_layers)]
        self.encoder = nn.ModuleList([TransformerBlockWithWindow(d_model, nhead, dim_ffn, min(7, h, w), drop_path=dpr[i]) for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model//2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d_model//2, d_model//4), nn.GELU(), nn.Dropout(0.15), nn.Linear(d_model//4, num_classes))
    def forward(self, x):
        B = x.size(0); tokens = rearrange(self.proj(self.backbone(x)), 'b d h w -> b (h w) d')
        tokens = self.norm_proj(tokens); tokens = torch.cat([self.cls_token.expand(B,-1,-1), tokens], dim=1) + self.pos_embed
        for block in self.encoder: tokens = block(tokens, self.spatial_h, self.spatial_w)
        return self.head(self.norm(tokens)[:, 0])
    def get_attentions(self):
        g, w = [], []
        for b in self.encoder:
            if b.attn_weights is not None: g.append(b.attn_weights)
            if b.window_attn_weights is not None: w.append(b.window_attn_weights)
        return g, w

# ─── TENSOR FUNCTION ─────────────────────────────────────────────────────────────
def get_tensor(img_input):
    if len(img_input.shape) == 2:
        img_input = np.stack([img_input, img_input, img_input], axis=-1)
    # Ensure image is 3-channel
    if img_input.shape[2] == 4:
        img_input = img_input[:, :, :3]
    t = torch.from_numpy(cv2.resize(img_input, (224, 224))).float().permute(2,0,1)/255.0
    return ((t - torch.tensor(MEAN).view(3,1,1)) / torch.tensor(STD).view(3,1,1)).unsqueeze(0).to(DEVICE)

# ─── LOAD MODEL ──────────────────────────────────────────────────────────────────
print("📦 Rebuilding Diabetic Retinopathy model and loading weights...")
model = EnhancedHViT(num_classes=5, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536).to(DEVICE)

checkpoint_path = CKDIR / 'best_model.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    if 'class_names' in checkpoint:
        class_names = checkpoint['class_names']
    print(f"✅ Model loaded from: {checkpoint_path}")
else:
    print(f"❌ Checkpoint not found at: {checkpoint_path}")
    raise FileNotFoundError("Model checkpoint not found!")

model.eval()
print("✅ Model loaded successfully!")

# ─── HEATMAP GENERATOR ───────────────────────────────────────────────────────────
def isolate_region(heatmap, percentile=40):
    if heatmap is None: return None
    threshold = np.percentile(heatmap, percentile)
    masked = np.where(heatmap > threshold, heatmap, 0.0)
    if masked.max() > masked.min():
        return (masked - masked.min()) / (masked.max() - masked.min() + 1e-8)
    return np.zeros_like(heatmap)

class HeatmapGenerator:
    def __init__(self, model):
        self.model = model.to(DEVICE)
        self.model.eval()
        self.cnn_f, self.cnn_g = None, None
        self.img_size = getattr(model, 'img_size', 224)
        
        last_conv = None
        for m in self.model.backbone.feature_extractor.modules():
            if isinstance(m, nn.Conv2d):
                last_conv = m
        
        if last_conv:
            last_conv.register_forward_hook(lambda m, i, o: setattr(self, 'cnn_f', o.detach()))
            last_conv.register_backward_hook(lambda m, gi, go: setattr(self, 'cnn_g', (go[0] if isinstance(go, (tuple, list)) else go).detach()))
            print("✅ Grad-CAM hooks registered")
    
    def get_cnn(self, x, target):
        out = self.model(x)
        self.model.zero_grad()
        oh = torch.zeros_like(out)
        oh[0, target] = 1
        out.backward(gradient=oh, retain_graph=True)
        if self.cnn_f is None: return None
        weights = self.cnn_g.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.cnn_f).sum(1)).squeeze().cpu().numpy()
        if cam.max() > cam.min():
            return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam
    
    def get_attn(self, x):
        with torch.no_grad():
            self.model(x)
        g, _ = self.model.get_attentions()
        if not g: return None
        a = g[-1]
        if a.dim() == 3:
            a = a[0, 0, 1:].cpu().numpy()
        else:
            a = a[0, :, 0, 1:].mean(0).cpu().numpy()
        a = a.reshape(self.model.spatial_h, self.model.spatial_w)
        if a.max() > a.min():
            return (a - a.min()) / (a.max() - a.min() + 1e-8)
        return a
    
    def draw(self, img_path, target, names, save_path, target_class_name=None):
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"❌ Could not read image: {img_path}")
            return
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        sz = rgb.shape[:2]
        t = get_tensor(rgb)
        
        c = self.get_cnn(t.clone(), target)
        a = self.get_attn(t.clone())
        comb = None
        if c is not None and a is not None:
            if a.shape != c.shape:
                a = zoom(a, (c.shape[0]/a.shape[0], c.shape[1]/a.shape[1]), order=1)
            comb = 0.5 * c + 0.5 * a
            if comb.max() > comb.min():
                comb = (comb - comb.min()) / (comb.max() - comb.min() + 1e-8)
        
        c, a, comb = isolate_region(c, 40), isolate_region(a, 35), isolate_region(comb, 35)
        
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(rgb)
        status = f"True: {names[target]}"
        if target_class_name:
            status += f" ({target_class_name})"
        axes[0].set_title(status, fontsize=12, fontweight='bold')
        axes[0].axis('off')
        
        heatmaps = [c, a, comb]
        titles = ['CNN (Grad-CAM)', 'Transformer Attention', 'Combined']
        for i, (hmap, title) in enumerate(zip(heatmaps, titles)):
            if hmap is not None:
                hr = zoom(hmap, (sz[0]/hmap.shape[0], sz[1]/hmap.shape[1]), order=1)
                colored = (plt.cm.jet(hr)[:,:,:3] * 255).astype(np.uint8)
                overlay = cv2.addWeighted(rgb, 0.6, colored, 0.4, 0)
                axes[i+1].imshow(overlay)
            else:
                axes[i+1].imshow(rgb)
            axes[i+1].set_title(title, fontsize=12, fontweight='bold')
            axes[i+1].axis('off')
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"✅ Heatmap saved: {save_path}")

# ─── LOAD DATASET ────────────────────────────────────────────────────────────────
print("\n📂 Loading dataset and finding test images...")

samples = []
possible_paths = [
    Path('/kaggle/input/datasets/nikitachaulagain/dataasets/Diabetic Retinopathy 224x224 (2019 Data)/Diabetic Retinopathy 224x224 (2019 Data)/colored_images'),
    Path('/kaggle/input/diabetic-retinopathy-224x224-2019-data/colored_images'),
    Path('/kaggle/input/datasets/nikitachaulagain/dataasets/Diabetic Retinopathy 224x224 (2019 Data)/colored_images'),
]

base = None
for path in possible_paths:
    if path.exists():
        base = path
        print(f"✅ Found dataset at: {base}")
        break

if base is None:
    print("❌ Dataset not found! Checking available directories...")
    input_dir = Path('/kaggle/input')
    if input_dir.exists():
        for item in input_dir.iterdir():
            print(f"  - {item.name}")
            if item.is_dir():
                for sub in item.iterdir():
                    if sub.is_dir():
                        print(f"    └─ {sub.name}")
    exit()

# First, check what folders exist in the base directory
print(f"\n📁 Contents of {base}:")
for item in base.iterdir():
    if item.is_dir():
        print(f"  📁 {item.name}")
    else:
        print(f"  📄 {item.name}")

# Map folder names to class names (handle various naming conventions)
folder_to_class = {
    '0': 'No_DR',
    '1': 'Mild',
    '2': 'Moderate',
    '3': 'Severe',
    '4': 'Proliferate_DR',
    '0 - No_DR': 'No_DR',
    '1 - Mild': 'Mild',
    '2 - Moderate': 'Moderate',
    '3 - Severe': 'Severe',
    '4 - Proliferate_DR': 'Proliferate_DR',
    'No_DR': 'No_DR',
    'Mild': 'Mild',
    'Moderate': 'Moderate',
    'Severe': 'Severe',
    'Proliferate_DR': 'Proliferate_DR',
    '0_No_DR': 'No_DR',
    '1_Mild': 'Mild',
    '2_Moderate': 'Moderate',
    '3_Severe': 'Severe',
    '4_Proliferate_DR': 'Proliferate_DR',
}

print("\n📂 Scanning dataset...")
for folder in base.iterdir():
    if not folder.is_dir(): 
        continue
    
    folder_name = folder.name.strip()
    class_name = None
    
    # Try exact match first
    if folder_name in folder_to_class:
        class_name = folder_to_class[folder_name]
    else:
        # Try partial match
        for key in folder_to_class:
            if key in folder_name or folder_name in key:
                class_name = folder_to_class[key]
                break
    
    if class_name is None:
        # Try to extract class number from folder name
        import re
        numbers = re.findall(r'\d+', folder_name)
        if numbers:
            num = numbers[0]
            if num in ['0', '1', '2', '3', '4']:
                class_name = folder_to_class[num]
    
    if class_name is None:
        print(f"   ⚠️ Skipping unknown folder: {folder_name}")
        continue
    
    count = 0
    for ext in VALID_EXTS:
        for p in folder.rglob(f'*{ext}'):
            samples.append((str(p), class_name))
            count += 1
    if count > 0:
        print(f"   ✅ {class_name}: {count} images")

print(f"\n✅ Total images found: {len(samples)}")

if len(samples) == 0:
    print("\n❌ No images found! Trying to find images directly in subfolders...")
    # Try to find images in subdirectories
    for ext in VALID_EXTS:
        for p in base.rglob(f'*{ext}'):
            # Try to infer class from parent folder
            parent = p.parent.name
            class_name = None
            for key in folder_to_class:
                if key in parent:
                    class_name = folder_to_class[key]
                    break
            if class_name is None:
                # Try to find any number in the path
                import re
                numbers = re.findall(r'\d+', str(p.parent))
                if numbers:
                    num = numbers[0]
                    if num in ['0', '1', '2', '3', '4']:
                        class_name = folder_to_class[num]
            if class_name:
                samples.append((str(p), class_name))
    print(f"✅ Found {len(samples)} images in fallback search")

if len(samples) == 0:
    print("❌ Still no images found. Please check the dataset structure.")
    exit()

# Create test split
labels = [class_names.index(l) for l in [s[1] for s in samples]]
_, temp_s = train_test_split(samples, test_size=0.3, stratify=labels, random_state=42)
_, test_s = train_test_split(temp_s, test_size=0.5, stratify=[class_names.index(l) for l in [s[1] for s in temp_s]], random_state=42)
print(f"📊 Test set size: {len(test_s)} images")

# ─── RUN INFERENCE ──────────────────────────────────────────────────────────────
print("\n🔍 Finding correct predictions for heatmap...")
all_p, all_l = [], []
hm = HeatmapGenerator(model)

for p, l in tqdm(test_s, desc="Evaluating"):
    img = cv2.imread(p)
    if img is None: continue
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    t = get_tensor(rgb)
    with torch.no_grad():
        out = model(t)
    all_p.append(out.argmax(1).item())
    all_l.append(class_names.index(l))

if len(all_p) > 0:
    correct = sum(1 for p,l in zip(all_p, all_l) if p == l)
    print(f"\n📊 Test Accuracy: {correct/len(all_p)*100:.2f}%")
else:
    print("❌ No predictions made!")

# ─── GENERATE HEATMAPS ──────────────────────────────────────────────────────────
if len(all_p) > 0:
    print("\n🔥 Generating Heatmaps...")

    # For each class, find correct predictions and generate heatmaps
    for class_idx in range(len(class_names)):
        correct_indices = [i for i, (p, l) in enumerate(zip(all_p, all_l)) 
                           if p == l and l == class_idx]
        
        if correct_indices:
            print(f"\n📊 {class_names[class_idx]}: {len(correct_indices)} correct predictions")
            # Generate up to 2 heatmaps per class
            for idx in correct_indices[:2]:
                hm.draw(test_s[idx][0], class_idx, class_names, 
                        PLTDIR / f'heatmap_{class_names[class_idx]}_{idx}.png',
                        target_class_name=class_names[class_idx])
        else:
            print(f"\n⚠️ No correct predictions for {class_names[class_idx]}")

    print("\n✅ All heatmaps generated successfully!")
    print(f"📁 Heatmaps saved in: {PLTDIR}")
else:
    print("❌ No predictions available to generate heatmaps.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  ULTIMATE MEGA CELL: ALL 20 PLOTS (DIABETIC RETINOPATHY - 5 CLASSES)           ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

import os, time, random, copy, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, cv2
from tqdm.auto import tqdm
from scipy.ndimage import zoom
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from sklearn.metrics import (accuracy_score, f1_score, roc_curve, auc, confusion_matrix, 
                             classification_report, precision_recall_curve, average_precision_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from torch.cuda.amp import autocast
from einops import rearrange

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTDIR = Path('/kaggle/working')
PLTDIR = OUTDIR / 'plots/dr'; PLTDIR.mkdir(exist_ok=True, parents=True)
CKDIR = OUTDIR / 'checkpoints'
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

# ✅ DIABETIC RETINOPATHY CONFIG - 5 CLASSES
class_names = ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferate_DR']
colors = ['#2ecc71', '#f1c40f', '#3498db', '#e67e22', '#e74c3c']
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.JPG', '.JPEG', '.PNG'}

# ─── MODEL ARCHITECTURE ──────────────────────────────────────────────────────────
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0): super().__init__(); self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training: return x
        return x * (torch.rand((x.shape[0], 1, 1), dtype=torch.float32, device=x.device) * float(1.0 - self.drop_prob))

class WindowAttention(nn.Module):
    def __init__(self, d_model, nhead, window_size=7, dropout=0.1):
        super().__init__()
        self.nhead, self.window_size, self.head_dim = nhead, window_size, d_model // nhead
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3); self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.proj_drop, self.attn_weights = nn.Dropout(dropout), nn.Dropout(dropout), None
    def forward(self, x, H, W):
        B, N, C = x.shape; x = x.reshape(B, H, W, C)
        pad_h, pad_w = (self.window_size - H % self.window_size) % self.window_size, (self.window_size - W % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0: x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        Hp, Wp = x.shape[1], x.shape[2]
        x = x.reshape(B, Hp//self.window_size, self.window_size, Wp//self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(-1, self.window_size**2, C)
        qkv = self.qkv(x).reshape(-1, self.window_size**2, 3, self.nhead, self.head_dim).permute(2,0,3,1,4)
        attn = (qkv[0] @ qkv[1].transpose(-2,-1)) * self.scale; attn = attn.softmax(dim=-1); self.attn_weights = attn.detach()
        x = self.proj_drop(self.proj((self.attn_drop(attn) @ qkv[2]).transpose(1,2).reshape(-1, self.window_size**2, C)))
        x = x.reshape(B, Hp//self.window_size, Wp//self.window_size, self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(B, Hp, Wp, C)
        return x[:, :H, :W, :].reshape(B, -1, C) if pad_h > 0 or pad_w > 0 else x.reshape(B, -1, C)

class TransformerBlockWithWindow(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim, window_size=7, attn_drop=0.1, ffn_drop=0.1, drop_path=0.0):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.global_attn = nn.MultiheadAttention(d_model, nhead, dropout=attn_drop, batch_first=True)
        self.window_attn = WindowAttention(d_model, nhead, window_size, attn_drop)
        self.gate = nn.Sequential(nn.Linear(d_model*2, d_model), nn.Sigmoid())
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(ffn_drop), nn.Linear(ffn_dim, d_model), nn.Dropout(ffn_drop))
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.attn_weights, self.window_attn_weights = None, None
    def forward(self, x, H, W):
        x_norm = self.norm1(x)
        global_out, global_attn = self.global_attn(x_norm, x_norm, x_norm, need_weights=True)
        self.attn_weights = global_attn.detach()
        window_out = self.window_attn(x_norm[:, 1:, :], H, W); self.window_attn_weights = self.window_attn.attn_weights
        gate = self.gate(torch.cat([global_out[:, 1:, :], window_out], dim=-1))
        combined = torch.cat([global_out[:, :1, :], gate * global_out[:, 1:, :] + (1-gate) * window_out], dim=1)
        x = x + self.drop_path(combined)
        return x + self.drop_path(self.ffn(self.norm2(x)))

class ImprovedBackbone(nn.Module):
    def __init__(self): super().__init__(); self.backbone = models.convnext_base(weights=None); self.feature_extractor = self.backbone.features
    def forward(self, x): return self.feature_extractor(x)

class EnhancedHViT(nn.Module):
    def __init__(self, num_classes=5, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536):
        super().__init__(); self.d_model = d_model; self.img_size = img_size; self.backbone = ImprovedBackbone()
        with torch.no_grad(): _, c, h, w = self.backbone(torch.zeros(1, 3, img_size, img_size)).shape
        self.spatial_h, self.spatial_w = h, w
        self.proj = nn.Sequential(nn.Conv2d(c, d_model, 1, bias=False), nn.BatchNorm2d(d_model), nn.GELU(), nn.Conv2d(d_model, d_model, 3, padding=1, bias=False), nn.BatchNorm2d(d_model), nn.GELU())
        self.norm_proj = nn.LayerNorm(d_model); self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, h*w + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02); nn.init.trunc_normal_(self.pos_embed, std=0.02)
        dpr = [x.item() for x in torch.linspace(0, 0.15, n_layers)]
        self.encoder = nn.ModuleList([TransformerBlockWithWindow(d_model, nhead, dim_ffn, min(7, h, w), drop_path=dpr[i]) for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model//2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d_model//2, d_model//4), nn.GELU(), nn.Dropout(0.15), nn.Linear(d_model//4, num_classes))
    def forward(self, x):
        B = x.size(0); tokens = rearrange(self.proj(self.backbone(x)), 'b d h w -> b (h w) d')
        tokens = self.norm_proj(tokens); tokens = torch.cat([self.cls_token.expand(B,-1,-1), tokens], dim=1) + self.pos_embed
        for block in self.encoder: tokens = block(tokens, self.spatial_h, self.spatial_w)
        return self.head(self.norm(tokens)[:, 0])
    def get_attentions(self):
        g, w = [], []
        for b in self.encoder:
            if b.attn_weights is not None: g.append(b.attn_weights)
            if b.window_attn_weights is not None: w.append(b.window_attn_weights)
        return g, w

# ─── TENSOR FUNCTION ─────────────────────────────────────────────────────────────
def get_tensor(img_input):
    if len(img_input.shape) == 2:
        img_input = np.stack([img_input, img_input, img_input], axis=-1)
    t = torch.from_numpy(cv2.resize(img_input, (224, 224))).float().permute(2,0,1)/255.0
    return ((t - torch.tensor(MEAN).view(3,1,1)) / torch.tensor(STD).view(3,1,1)).unsqueeze(0).to(DEVICE)

# ─── LOAD MODEL & DATA ──────────────────────────────────────────────────────────
print("📦 Rebuilding Diabetic Retinopathy model and loading weights...")
model = EnhancedHViT(num_classes=5, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536).to(DEVICE)

checkpoint_path = CKDIR / 'best_model.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    if 'class_names' in checkpoint:
        class_names = checkpoint['class_names']
    print(f"✅ Model loaded from: {checkpoint_path}")
else:
    print(f"❌ Checkpoint not found! Please train the model first.")
    raise FileNotFoundError("Model checkpoint not found!")

model.eval()

print("\n📂 Loading dataset and finding test images...")

samples = []
possible_paths = [
    Path('/kaggle/input/datasets/nikitachaulagain/dataasets/Diabetic Retinopathy 224x224 (2019 Data)/Diabetic Retinopathy 224x224 (2019 Data)/colored_images'),
    Path('/kaggle/input/diabetic-retinopathy-224x224-2019-data/colored_images'),
    Path('/kaggle/input/datasets/nikitachaulagain/dataasets/Diabetic Retinopathy 224x224 (2019 Data)/colored_images'),
]

base = None
for path in possible_paths:
    if path.exists():
        base = path
        print(f"✅ Found dataset at: {base}")
        break

if base is None:
    print("❌ Dataset not found! Checking available directories...")
    input_dir = Path('/kaggle/input')
    if input_dir.exists():
        for item in input_dir.iterdir():
            print(f"  - {item.name}")
            if item.is_dir():
                for sub in item.iterdir():
                    if sub.is_dir():
                        print(f"    └─ {sub.name}")
    exit()

# First, check what folders exist in the base directory
print(f"\n📁 Contents of {base}:")
for item in base.iterdir():
    if item.is_dir():
        print(f"  📁 {item.name}")
    else:
        print(f"  📄 {item.name}")

# Map folder names to class names (handle various naming conventions)
folder_to_class = {
    '0': 'No_DR',
    '1': 'Mild',
    '2': 'Moderate',
    '3': 'Severe',
    '4': 'Proliferate_DR',
    '0 - No_DR': 'No_DR',
    '1 - Mild': 'Mild',
    '2 - Moderate': 'Moderate',
    '3 - Severe': 'Severe',
    '4 - Proliferate_DR': 'Proliferate_DR',
    'No_DR': 'No_DR',
    'Mild': 'Mild',
    'Moderate': 'Moderate',
    'Severe': 'Severe',
    'Proliferate_DR': 'Proliferate_DR',
    '0_No_DR': 'No_DR',
    '1_Mild': 'Mild',
    '2_Moderate': 'Moderate',
    '3_Severe': 'Severe',
    '4_Proliferate_DR': 'Proliferate_DR',
}

print("\n📂 Scanning dataset...")
for folder in base.iterdir():
    if not folder.is_dir(): 
        continue
    
    folder_name = folder.name.strip()
    class_name = None
    
    # Try exact match first
    if folder_name in folder_to_class:
        class_name = folder_to_class[folder_name]
    else:
        # Try partial match
        for key in folder_to_class:
            if key in folder_name or folder_name in key:
                class_name = folder_to_class[key]
                break
    
    if class_name is None:
        # Try to extract class number from folder name
        import re
        numbers = re.findall(r'\d+', folder_name)
        if numbers:
            num = numbers[0]
            if num in ['0', '1', '2', '3', '4']:
                class_name = folder_to_class[num]
    
    if class_name is None:
        print(f"   ⚠️ Skipping unknown folder: {folder_name}")
        continue
    
    count = 0
    for ext in VALID_EXTS:
        for p in folder.rglob(f'*{ext}'):
            samples.append((str(p), class_name))
            count += 1
    if count > 0:
        print(f"   ✅ {class_name}: {count} images")

print(f"\n✅ Total images found: {len(samples)}")

if len(samples) == 0:
    print("\n❌ No images found! Trying to find images directly in subfolders...")
    # Try to find images in subdirectories
    for ext in VALID_EXTS:
        for p in base.rglob(f'*{ext}'):
            # Try to infer class from parent folder
            parent = p.parent.name
            class_name = None
            for key in folder_to_class:
                if key in parent:
                    class_name = folder_to_class[key]
                    break
            if class_name is None:
                # Try to find any number in the path
                import re
                numbers = re.findall(r'\d+', str(p.parent))
                if numbers:
                    num = numbers[0]
                    if num in ['0', '1', '2', '3', '4']:
                        class_name = folder_to_class[num]
            if class_name:
                samples.append((str(p), class_name))
    print(f"✅ Found {len(samples)} images in fallback search")

if len(samples) == 0:
    print("❌ Still no images found. Please check the dataset structure.")
    exit()
    
labels = [class_names.index(l) for l in [s[1] for s in samples]]
train_s, temp_s = train_test_split(samples, test_size=0.3, stratify=labels, random_state=42)
val_s, test_s = train_test_split(temp_s, test_size=0.5, stratify=[class_names.index(l) for l in [s[1] for s in temp_s]], random_state=42)
df_all = pd.DataFrame(samples, columns=['path', 'label'])

print("\n" + "="*60)
print("📊 DIABETIC RETINOPATHY DATASET ANALYSIS")
print("="*60)
print(f"\n🔹 Total Images: {len(df_all)} | Classes: {len(class_names)}")
print(f"🔹 Split: 70% Train ({len(train_s)}) / 15% Val ({len(val_s)}) / 15% Test ({len(test_s)})")
for cls in class_names:
    count = len(df_all[df_all['label'] == cls])
    print(f"   - {cls:18s}: {count:5d} images ({count/len(df_all)*100:.1f}%)")
print("="*60 + "\n")

# ─── RUN INFERENCE ──────────────────────────────────────────────────────────────
print("🧠 Running inference on Test Set...")
all_p, all_l, all_pr = [], [], []
for p, l in tqdm(test_s, desc="Predicting"):
    img = cv2.imread(p)
    if img is None: continue
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        out = model(get_tensor(rgb))
        pr = F.softmax(out, dim=1)
    all_p.append(out.argmax(1).item())
    all_l.append(class_names.index(l))
    all_pr.append(pr.cpu().numpy()[0])

all_p = np.array(all_p)
all_l = np.array(all_l)
all_pr = np.array(all_pr)

# Ensure all_pr has correct shape
if all_pr.shape[1] == 1:
    # For binary case, but we have 5 classes
    pass

max_probs = all_pr.max(axis=1)
correct_mask = (all_p == all_l)
labels_bin = label_binarize(all_l, classes=list(range(len(class_names))))

# ─── EVALUATION METRICS ─────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📊 COMPREHENSIVE EVALUATION METRICS")
print("="*60)

test_acc = accuracy_score(all_l, all_p)
test_f1_macro = f1_score(all_l, all_p, average='macro')
test_f1_weighted = f1_score(all_l, all_p, average='weighted')
test_f1_micro = f1_score(all_l, all_p, average='micro')

try:
    test_auc = roc_auc_score(all_l, all_pr, multi_class='ovr', average='weighted')
except:
    test_auc = 0.0

mcc = matthews_corrcoef(all_l, all_p)
kappa = cohen_kappa_score(all_l, all_p)

cm = confusion_matrix(all_l, all_p)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

print(f"\n🎯 PRIMARY METRICS:")
print(f"   Accuracy:       {test_acc:.4f}")
print(f"   F1-Macro:       {test_f1_macro:.4f}")
print(f"   F1-Weighted:    {test_f1_weighted:.4f}")
print(f"   F1-Micro:       {test_f1_micro:.4f}")
print(f"   AUC-ROC (OVR):  {test_auc:.4f}")

print(f"\n📈 ADDITIONAL METRICS:")
print(f"   MCC:            {mcc:.4f}")
print(f"   Cohen's Kappa:  {kappa:.4f}")

print(f"\n📊 CONFUSION MATRIX:")
print(f"{'':18s}", end="")
for cls in class_names:
    print(f"{cls[:8]:>10s}", end="")
print()
for i, cls in enumerate(class_names):
    print(f"{cls:18s}", end="")
    for j in range(len(class_names)):
        print(f"{cm[i,j]:>10d}", end="")
    print()

print(f"\n📋 CLASSIFICATION REPORT:")
print(classification_report(all_l, all_p, target_names=class_names))

print("="*60)

# ─── PLOT 1: Training History ──────────────────────────────────────────────────
try:
    history
except NameError:
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': [], 'lr': []}

try:
    if len(history.get('train_loss', [])) > 0:
        fig, ax = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('DR Model Training Dynamics', fontsize=16, fontweight='bold', y=1.02)
        ax[0,0].plot(history['train_loss'], label='Train', color='#e74c3c', lw=2)
        ax[0,0].plot(history['val_loss'], label='Val', color='#3498db', lw=2)
        ax[0,0].set_title('Loss'); ax[0,0].legend(); ax[0,0].grid(True, alpha=0.3)
        ax[0,1].plot(history['train_acc'], label='Train', color='#e74c3c', lw=2)
        ax[0,1].plot(history['val_acc'], label='Val', color='#3498db', lw=2)
        ax[0,1].set_title('Accuracy'); ax[0,1].set_ylim(0,1.05)
        ax[0,1].legend(); ax[0,1].grid(True, alpha=0.3)
        ax[1,0].plot(history['val_f1'], label='Val F1', color='#2ecc71', lw=2)
        ax[1,0].set_title('F1 Score'); ax[1,0].set_ylim(0,1.05)
        ax[1,0].legend(); ax[1,0].grid(True, alpha=0.3)
        ax[1,1].plot(history['lr'], color='#9b59b6', lw=2)
        ax[1,1].set_title('Learning Rate'); ax[1,1].set_yscale('log')
        ax[1,1].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(PLTDIR / '1_training_history.png', dpi=300, bbox_inches='tight')
        plt.show()
except:
    pass

# ─── PLOT 2: Confusion Matrix ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Diabetic Retinopathy Confusion Matrix', fontsize=16, fontweight='bold')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0], cbar=False)
axes[0].set_title('Absolute Counts'); axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=axes[1], cbar=False)
axes[1].set_title('Normalized (Recall)'); axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(PLTDIR / '2_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 3: ROC Curves ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], all_pr[:, i])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{class_names[i]} (AUC = {roc_auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k:', lw=1, alpha=0.5)
ax.set_title('ROC Curves - Diabetic Retinopathy Detection', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLTDIR / '3_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 4: Precision-Recall Curves ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
for i in range(len(class_names)):
    prec, rec, _ = precision_recall_curve(labels_bin[:, i], all_pr[:, i])
    ap = average_precision_score(labels_bin[:, i], all_pr[:, i])
    ax.plot(rec, prec, color=colors[i], lw=2, label=f'{class_names[i]} (AP = {ap:.3f})')
ax.set_title('Precision-Recall Curves - DR Detection', fontsize=14, fontweight='bold')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
plt.tight_layout()
plt.savefig(PLTDIR / '4_precision_recall.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 5: Per-Class Metrics ─────────────────────────────────────────────────
report = classification_report(all_l, all_p, target_names=class_names, output_dict=True)
metrics_df = pd.DataFrame(report).T.iloc[:-3, :3]
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(class_names)); width = 0.25
bars1 = ax.bar(x - width, metrics_df['precision'], width, label='Precision', color='#3498db', edgecolor='black')
bars2 = ax.bar(x, metrics_df['recall'], width, label='Recall', color='#2ecc71', edgecolor='black')
bars3 = ax.bar(x + width, metrics_df['f1-score'], width, label='F1-Score', color='#e74c3c', edgecolor='black')
ax.set_title('Per-Class Performance Metrics - DR', fontsize=14, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(class_names, rotation=15, ha='right')
ax.legend(); ax.set_ylim(0,1.15); ax.grid(axis='y', alpha=0.3)
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, 
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(PLTDIR / '5_per_class_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 6: Confidence Distribution ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(max_probs[correct_mask], bins=20, alpha=0.7, label='Correct', color='#2ecc71', edgecolor='black')
ax.hist(max_probs[~correct_mask], bins=20, alpha=0.7, label='Incorrect', color='#e74c3c', edgecolor='black')
ax.set_title('Model Confidence Distribution - DR', fontsize=14, fontweight='bold')
ax.set_xlabel('Confidence'); ax.set_ylabel('Count')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLTDIR / '6_confidence_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 7: Calibration Curve ─────────────────────────────────────────────────
def calibration_curve(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    accs, confs = [], []
    for i in range(n_bins):
        mask = (y_prob > bins[i]) & (y_prob <= bins[i+1])
        if i == n_bins - 1:
            mask = (y_prob >= bins[i]) & (y_prob <= bins[i+1])
        if mask.sum() > 0:
            accs.append(y_true[mask].mean())
            confs.append(y_prob[mask].mean())
    return np.array(confs), np.array(accs)

fig, ax = plt.subplots(figsize=(8, 8))
for i, cls in enumerate(class_names):
    conf, acc = calibration_curve((all_l == i).astype(int), all_pr[:, i])
    if len(conf) > 0:
        ax.plot(conf, acc, marker='o', color=colors[i], lw=2, label=cls, markersize=5)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfectly Calibrated')
ax.set_title('Reliability Diagram - DR', fontsize=14, fontweight='bold')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlim([0,1]); ax.set_ylim([0,1])
plt.tight_layout()
plt.savefig(PLTDIR / '7_calibration_curve.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 8: Sample Predictions ─────────────────────────────────────────────────
correct_idxs, incorrect_idxs = np.where(correct_mask)[0], np.where(~correct_mask)[0]
np.random.seed(42)
samp_corr = np.random.choice(correct_idxs, min(9, len(correct_idxs)), replace=False)
samp_incorr = np.random.choice(incorrect_idxs, min(9, len(incorrect_idxs)), replace=False)
fig, axes = plt.subplots(2, 9, figsize=(25, 6))
for i, idx in enumerate(samp_corr):
    img = cv2.imread(test_s[idx][0])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"T:{class_names[all_l[idx]]}\nP:{class_names[all_p[idx]]}\n{max_probs[idx]:.1%}", 
                         fontsize=8, color='green')
    axes[0, i].axis('off')
for i, idx in enumerate(samp_incorr):
    img = cv2.imread(test_s[idx][0])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"T:{class_names[all_l[idx]]}\nP:{class_names[all_p[idx]]}\n{max_probs[idx]:.1%}", 
                         fontsize=8, color='red')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('CORRECT', fontsize=12, fontweight='bold', color='green', rotation=0, labelpad=60, va='center')
axes[1, 0].set_ylabel('INCORRECT', fontsize=12, fontweight='bold', color='red', rotation=0, labelpad=60, va='center')
fig.suptitle('Visual Inspection - DR (T=True, P=Predicted)', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(PLTDIR / '8_sample_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 9: Violin Plot ───────────────────────────────────────────────────────
plot_data = [{'True Class': class_names[i], 'Assigned Probability': prob} 
             for i in range(len(class_names)) for prob in all_pr[all_l == i, i]]
df_probs = pd.DataFrame(plot_data)
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df_probs, x='True Class', y='Assigned Probability', palette=colors, inner='quartile', ax=ax)
ax.set_title('Probability Density for True Class - DR', fontsize=14, fontweight='bold')
ax.set_ylim(0,1.05); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(PLTDIR / '9_probability_violin.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 10: t-SNE ────────────────────────────────────────────────────────────
print("⏳ Extracting t-SNE features...")
feature_tensor = None
def hook_fn(module, inp, out):
    global feature_tensor
    feature_tensor = out
handle = model.norm.register_forward_hook(hook_fn)
tsne_indices = np.random.choice(len(test_s), min(600, len(test_s)), replace=False)
features_list, labels_tsne = [], []
for idx in tsne_indices:
    p, l = test_s[idx]
    img = cv2.imread(p)
    if img is None: continue
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        _ = model(get_tensor(rgb))
    features_list.append(feature_tensor[:, 0, :].cpu().numpy()[0])
    labels_tsne.append(class_names.index(l))
handle.remove()
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
tsne_results = tsne.fit_transform(np.array(features_list))
fig, ax = plt.subplots(figsize=(10, 8))
for i, cls in enumerate(class_names):
    mask = np.array(labels_tsne) == i
    ax.scatter(tsne_results[mask, 0], tsne_results[mask, 1], c=colors[i], label=cls, 
               alpha=0.6, s=20, edgecolors='white', linewidth=0.5)
ax.set_title('t-SNE Visualization - DR Features', fontsize=14, fontweight='bold')
ax.legend(markerscale=2)
ax.grid(True, alpha=0.2)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(PLTDIR / '10_tsne_features.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 11: Error Analysis ──────────────────────────────────────────────────
errors = [{'True': class_names[all_l[i]], 'Predicted': class_names[all_p[i]]} 
          for i in range(len(all_l)) if all_l[i] != all_p[i]]
if errors:
    error_df = pd.DataFrame(errors)
    error_counts = error_df.groupby(['True', 'Predicted']).size().reset_index(name='Count').sort_values(by='Count', ascending=False)
    fig, ax = plt.subplots(figsize=(10, 6))
    error_counts['Pair'] = error_counts['True'] + ' → ' + error_counts['Predicted']
    sns.barplot(data=error_counts, x='Count', y='Pair', palette='Reds_r', ax=ax)
    ax.set_title('Top Misclassification Pairs - DR', fontsize=14, fontweight='bold')
    for i, v in enumerate(error_counts['Count']):
        ax.text(v + 0.2, i, str(v), color='black', va='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(PLTDIR / '11_error_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

# ─── PLOT 12: Class Distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('DR Dataset Class Distribution', fontsize=16, fontweight='bold')
counts = df_all['label'].value_counts().reindex(class_names)
sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette=colors, edgecolor='black')
axes[0].set_title('Image Count per Class', fontsize=13)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold', fontsize=11)
axes[1].pie(counts.values, labels=class_names, autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'black', 'linewidth': 1.5}, textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title('Percentage Distribution', fontsize=13)
plt.tight_layout()
plt.savefig(PLTDIR / '12_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 13: Split Distribution ──────────────────────────────────────────────
train_s_df = pd.DataFrame(train_s, columns=['path', 'label']); train_s_df['split'] = 'Train'
val_s_df = pd.DataFrame(val_s, columns=['path', 'label']); val_s_df['split'] = 'Validation'
test_s_df = pd.DataFrame(test_s, columns=['path', 'label']); test_s_df['split'] = 'Test'
df_splits = pd.concat([train_s_df, val_s_df, test_s_df])
split_counts = df_splits.groupby(['split', 'label']).size().unstack(fill_value=0)[class_names]
fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Class Distribution Across Data Splits - DR', fontsize=16, fontweight='bold')
split_counts.plot(kind='barh', stacked=True, ax=ax, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Number of Images'); ax.set_ylabel('')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=12)
ax.legend(title='Class', bbox_to_anchor=(1.05, 1))
ax.grid(axis='x', alpha=0.3)
for i, split in enumerate(split_counts.index):
    ax.text(split_counts.loc[split].sum() + 10, i, f'Total: {split_counts.loc[split].sum()}', 
            va='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig(PLTDIR / '13_split_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 14: Average Images ──────────────────────────────────────────────────
print("⏳ Computing average images per class...")
fig, axes = plt.subplots(1, 5, figsize=(15, 4))
fig.suptitle('Average Retina Image per Class', fontsize=14, fontweight='bold')
for i, cls in enumerate(class_names):
    cls_paths = df_all[df_all['label'] == cls]['path'].sample(n=min(200, len(df_all[df_all['label']==cls])), random_state=42).tolist()
    avg_img = np.zeros((224, 224, 3), dtype=np.float32)
    for p in cls_paths:
        img = cv2.imread(p)
        if img is not None:
            avg_img += cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (224, 224))
    axes[i].imshow((avg_img / len(cls_paths)).astype(np.uint8))
    axes[i].set_title(f'{cls}'); axes[i].axis('off')
plt.tight_layout()
plt.savefig(PLTDIR / '14_average_images.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 15: Pixel Intensity ──────────────────────────────────────────────────
print("⏳ Sampling pixel intensities...")
fig, ax = plt.subplots(figsize=(10, 6))
for i, cls in enumerate(class_names):
    cls_paths = df_all[df_all['label'] == cls]['path'].sample(n=50, random_state=42).tolist()
    pixels = []
    for p in cls_paths:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            pixels.extend(np.random.choice(img.flatten(), size=1000, replace=False))
    sns.kdeplot(pixels, fill=True, color=colors[i], label=cls, ax=ax, alpha=0.3, linewidth=2)
ax.set_title('Pixel Intensity Distribution by Class - DR', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLTDIR / '15_pixel_intensity.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 16: Image Dimensions (FIXED) ──────────────────────────────────────────
print("⏳ Checking image dimensions...")
dim_data = []
for p in df_all['path'].sample(n=500, random_state=42):
    img = cv2.imread(p)
    if img is not None:
        h, w = img.shape[:2]
        lbl = df_all.loc[df_all['path']==p, 'label'].values[0]
        dim_data.append({'Height': h, 'Width': w, 'Label': lbl})
dim_df = pd.DataFrame(dim_data)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Image Dimension Analysis - DR', fontsize=14, fontweight='bold')
sns.boxplot(data=dim_df, x='Label', y='Height', palette=colors, ax=axes[0])
axes[0].set_title('Image Height Distribution')
axes[0].tick_params(axis='x', rotation=15)  # ✅ FIXED
sns.boxplot(data=dim_df, x='Label', y='Width', palette=colors, ax=axes[1])
axes[1].set_title('Image Width Distribution')
axes[1].tick_params(axis='x', rotation=15)  # ✅ FIXED
plt.tight_layout()
plt.savefig(PLTDIR / '16_image_dimensions.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 17: GRAD-CAM + ATTENTION FOR 5 SAMPLE IMAGES ──────────────────────────
print("\n🔥 Generating Grad-CAM + Attention Overlays for 5 Sample Images...")

class AdvancedXAI:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.grads, self.activations = None, None
        
        # Find last conv layer for Grad-CAM
        last_conv = None
        for m in self.model.backbone.feature_extractor.modules():
            if isinstance(m, nn.Conv2d):
                last_conv = m
        if last_conv:
            last_conv.register_forward_hook(self._act_hook)
            last_conv.register_backward_hook(self._grad_hook)
            print("✅ Grad-CAM hooks registered")
        else:
            print("⚠️ No Conv2d found for Grad-CAM")

    def _act_hook(self, m, i, o): 
        self.activations = o.detach()
    
    def _grad_hook(self, m, gi, go): 
        self.grads = go[0].detach()

    def generate(self, img_path, target_class):
        """Generate Grad-CAM, Attention, and Combined overlays."""
        # Load image
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"❌ Could not read image: {img_path}")
            return None, None, None, None
        
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = rgb.shape[:2]
        
        # Get tensor
        x = get_tensor(rgb).float().requires_grad_(True)
        
        # ─── Grad-CAM ────────────────────────────────────────────────────────────
        out = self.model(x)
        self.model.zero_grad()
        oh = torch.zeros_like(out)
        oh[0, target_class] = 1
        out.backward(gradient=oh, retain_graph=True)
        
        if self.grads is not None and self.activations is not None:
            weights = self.grads.mean(dim=(2, 3), keepdim=True)
            cam = F.relu((weights * self.activations).sum(1)).squeeze().cpu().numpy()
            if cam.max() > cam.min():
                cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
            else:
                cam = np.zeros_like(cam)
        else:
            cam = np.zeros((self.model.spatial_h, self.model.spatial_w))
        
        # ─── Attention Map ──────────────────────────────────────────────────────
        with torch.no_grad():
            _ = self.model(x)
        
        g_attn, _ = self.model.get_attentions()
        
        if g_attn:
            attn_weights = g_attn[-1]  # Use last layer
            
            # Safely handle both 3D and 4D tensors
            try:
                if attn_weights.dim() == 4:
                    # Shape: (batch, num_heads, seq_len, seq_len)
                    attn = attn_weights[0, :, 0, 1:].mean(0).cpu().numpy()
                elif attn_weights.dim() == 3:
                    if attn_weights.shape[0] == 1:
                        # (batch, seq_len, seq_len)
                        attn = attn_weights[0, 0, 1:].cpu().numpy()
                    else:
                        # (num_heads, seq_len, seq_len)
                        attn = attn_weights[:, 0, 1:].mean(0).cpu().numpy()
                elif attn_weights.dim() == 2:
                    # (seq_len, seq_len)
                    attn = attn_weights[0, 1:].cpu().numpy()
                else:
                    print(f"⚠️ Unexpected attention shape: {attn_weights.shape}")
                    attn = None
            except Exception as e:
                print(f"⚠️ Error extracting attention: {e}")
                attn = None
            
            if attn is not None:
                # Reshape to spatial dimensions
                n_tokens = len(attn)
                spatial_h, spatial_w = self.model.spatial_h, self.model.spatial_w
                
                if n_tokens == spatial_h * spatial_w:
                    attn = attn.reshape(spatial_h, spatial_w)
                else:
                    # Try to find square grid
                    grid_size = int(np.sqrt(n_tokens))
                    if grid_size * grid_size == n_tokens:
                        attn = attn.reshape(grid_size, grid_size)
                        attn = cv2.resize(attn, (spatial_w, spatial_h))
                    else:
                        print(f"⚠️ Token count {n_tokens} doesn't match spatial {spatial_h}x{spatial_w}")
                        attn = None
                
                if attn is not None and attn.max() > attn.min():
                    attn = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
                else:
                    attn = np.zeros((spatial_h, spatial_w))
        else:
            print("⚠️ No attention weights captured")
            attn = np.zeros((self.model.spatial_h, self.model.spatial_w))
        
        # ─── Overlay Function ──────────────────────────────────────────────────
        def overlay_heatmap(hmap):
            if hmap is None or hmap.max() == 0:
                return rgb.copy()
            hmap_resized = cv2.resize(hmap, (orig_w, orig_h))
            heatmap_color = cv2.applyColorMap((hmap_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
            return cv2.addWeighted(rgb, 0.6, heatmap_color, 0.4, 0)
        
        # ─── Generate Overlays ──────────────────────────────────────────────────
        cam_rgb = overlay_heatmap(cam)
        attn_rgb = overlay_heatmap(attn)
        
        # Combined (50/50 blend)
        if cam is not None and attn is not None:
            if cam.shape == attn.shape:
                comb = 0.5 * cam + 0.5 * attn
            else:
                attn_resized = cv2.resize(attn, (cam.shape[1], cam.shape[0]))
                comb = 0.5 * cam + 0.5 * attn_resized
            if comb.max() > comb.min():
                comb = (comb - comb.min()) / (comb.max() - comb.min() + 1e-8)
            comb_rgb = overlay_heatmap(comb)
        else:
            comb_rgb = overlay_heatmap(cam if cam is not None else attn)
        
        return rgb, cam_rgb, attn_rgb, comb_rgb

# ─── Initialize XAI ──────────────────────────────────────────────────────────────
xai = AdvancedXAI(model)
print(f"📐 Model spatial dimensions: {model.spatial_h}x{model.spatial_w}")

# ─── Select 5 Sample Images ──────────────────────────────────────────────────────
print("\n📂 Selecting 5 sample images...")

sample_indices = []
sample_classes = []

# Try to get one from each class first
for cls_idx in range(len(class_names)):
    # Find correctly classified images with high confidence
    mask = (all_p == cls_idx) & (all_l == cls_idx) & (max_probs > 0.7)
    if not mask.any():
        mask = (all_p == cls_idx) & (all_l == cls_idx)
    if not mask.any():
        mask = (all_p == cls_idx)
    
    if mask.any():
        idx = np.where(mask)[0][0]
        sample_indices.append(idx)
        sample_classes.append(cls_idx)
        print(f"   ✅ {class_names[cls_idx]}: confidence {max_probs[idx]:.1%}")

# If we need more samples (less than 5), add from the most confident predictions
if len(sample_indices) < 5:
    # Get additional high-confidence correct predictions
    remaining_mask = ~np.isin(np.arange(len(all_p)), sample_indices)
    remaining_correct = remaining_mask & (all_p == all_l)
    if remaining_correct.any():
        # Sort by confidence
        remaining_indices = np.where(remaining_correct)[0]
        remaining_probs = max_probs[remaining_indices]
        sorted_indices = remaining_indices[np.argsort(remaining_probs)[::-1]]
        
        for idx in sorted_indices:
            if len(sample_indices) >= 5:
                break
            sample_indices.append(idx)
            sample_classes.append(all_l[idx])
            print(f"   ✅ Additional sample: {class_names[all_l[idx]]} (confidence {max_probs[idx]:.1%})")

# If still less than 5, add any predictions
if len(sample_indices) < 5:
    remaining_mask = ~np.isin(np.arange(len(all_p)), sample_indices)
    remaining_indices = np.where(remaining_mask)[0]
    for idx in remaining_indices[:5 - len(sample_indices)]:
        sample_indices.append(idx)
        sample_classes.append(all_l[idx])
        print(f"   ✅ Additional sample: {class_names[all_l[idx]]} (confidence {max_probs[idx]:.1%})")

print(f"\n📊 Selected {len(sample_indices)} sample images")

# ─── Generate 5-Panel Visualization ─────────────────────────────────────────────
titles = ['Original Image', 'Grad-CAM (CNN Focus)', 'Transformer Attention', 'Fused Explanation']

# Create figure with 5 rows (or fewer) and 4 columns
num_samples = len(sample_indices)
fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
fig.suptitle('Model Explainability: Grad-CAM vs. Attention vs. Combined\n(5 Sample Images)', 
             fontsize=18, fontweight='bold', y=1.01)

# If only 1 sample, axes might be 1D
if num_samples == 1:
    axes = axes.reshape(1, -1)

for row_idx, idx in enumerate(sample_indices):
    # Get image path and true class
    img_path = test_s[idx][0]
    true_class = all_l[idx]
    pred_class = all_p[idx]
    confidence = max_probs[idx]
    
    # Generate overlays
    rgb, cam_rgb, attn_rgb, comb_rgb = xai.generate(img_path, true_class)
    
    if rgb is not None:
        imgs = [rgb, cam_rgb, attn_rgb, comb_rgb]
        for col_idx, img in enumerate(imgs):
            axes[row_idx, col_idx].imshow(img)
            
            # Add information
            if col_idx == 0:
                status = "✅ Correct" if pred_class == true_class else "❌ Incorrect"
                axes[row_idx, col_idx].set_title(
                    f"{status}: {class_names[true_class]}\n"
                    f"Pred: {class_names[pred_class]} ({confidence:.1%})", 
                    fontsize=11, fontweight='bold'
                )
            else:
                axes[row_idx, col_idx].set_title(titles[col_idx], fontsize=11, fontweight='bold')
            
            axes[row_idx, col_idx].axis('off')
    else:
        for col_idx in range(4):
            axes[row_idx, col_idx].axis('off')
        axes[row_idx, 0].text(0.5, 0.5, f'Failed to load image', 
                             ha='center', va='center', transform=axes[row_idx, 0].transAxes)

# Remove empty rows if any
if num_samples < 5:
    for row_idx in range(num_samples, 5):
        for col_idx in range(4):
            axes[row_idx, col_idx].axis('off')

plt.tight_layout()
plt.savefig(PLTDIR / '17_xai_5_samples_gradcam_attention.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ 5-Sample explainability plot saved to: {PLTDIR / '17_xai_5_samples_gradcam_attention.png'}")

# ─── Bonus: Create a Grid Summary with All Samples (FIXED) ──────────────────────
print("\n📊 Creating summary grid with all 5 samples...")

num_samples = len(sample_indices)

# Create figure with num_samples + 1 rows (for header) and 4 columns
fig, axes = plt.subplots(num_samples + 1, 4, figsize=(20, 5 * (num_samples + 1)))
fig.suptitle('Model Explainability: Grad-CAM vs. Attention vs. Combined\n(5 Sample Images)', 
             fontsize=20, fontweight='bold', y=0.98)

# Column headers
headers = ['Original Image', 'Grad-CAM (CNN Focus)', 'Transformer Attention', 'Fused Explanation']
for col_idx, header in enumerate(headers):
    axes[0, col_idx].set_title(header, fontsize=14, fontweight='bold', color='navy')
    axes[0, col_idx].axis('off')

# Fill rows
for row_idx, idx in enumerate(sample_indices[:5]):
    img_path = test_s[idx][0]
    true_class = all_l[idx]
    pred_class = all_p[idx]
    confidence = max_probs[idx]
    
    # Generate overlays
    rgb, cam_rgb, attn_rgb, comb_rgb = xai.generate(img_path, true_class)
    
    if rgb is not None:
        imgs = [rgb, cam_rgb, attn_rgb, comb_rgb]
        for col_idx, img in enumerate(imgs):
            ax = axes[row_idx + 1, col_idx]  # +1 for header row
            ax.imshow(img)
            
            if col_idx == 0:
                # Show class label and confidence on the original image
                status = "✅" if pred_class == true_class else "❌"
                ax.set_xlabel(
                    f"{status} True: {class_names[true_class]}\n"
                    f"Pred: {class_names[pred_class]} ({confidence:.1%})",
                    fontsize=11, fontweight='bold'
                )
            ax.axis('off')
    else:
        for col_idx in range(4):
            ax = axes[row_idx + 1, col_idx]
            ax.axis('off')

# Remove any empty rows if less than 5 samples
if num_samples < 5:
    for row_idx in range(num_samples + 1, 6):
        if row_idx < axes.shape[0]:
            for col_idx in range(4):
                axes[row_idx, col_idx].axis('off')

plt.tight_layout()
plt.savefig(PLTDIR / '17_xai_5_samples_summary_grid.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Summary grid saved to: {PLTDIR / '17_xai_5_samples_summary_grid.png'}")

# ─── PLOT 18: Deployment Thresholds ────────────────────────────────────────────
print("\n🚀 Analyzing Confidence Thresholds...")
thresholds = np.arange(0.5, 1.0, 0.05)
metrics_data = []
for thresh in thresholds:
    mask = max_probs >= thresh
    if mask.sum() == 0: continue
    p_thresh, l_thresh = all_p[mask], all_l[mask]
    acc = accuracy_score(l_thresh, p_thresh)
    f1 = f1_score(l_thresh, p_thresh, average='weighted', zero_division=0)
    coverage = mask.sum() / len(all_p)
    metrics_data.append({'Threshold': thresh, 'Accuracy': acc, 'F1-Score': f1, 'Coverage': coverage * 100})

df_thresh = pd.DataFrame(metrics_data)
fig, ax1 = plt.subplots(figsize=(10, 6))
fig.suptitle('Deployment Strategy: Confidence vs. Performance - DR', fontsize=14, fontweight='bold')
ax1.set_xlabel('Minimum Confidence Threshold')
ax1.set_ylabel('Metric Score', color='#3498db')
ax1.plot(df_thresh['Threshold'], df_thresh['Accuracy'], color='#3498db', marker='o', lw=2, label='Accuracy')
ax1.plot(df_thresh['Threshold'], df_thresh['F1-Score'], color='#2ecc71', marker='s', lw=2, label='F1-Score')
ax1.tick_params(axis='y', labelcolor='#3498db')
ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx()
ax2.set_ylabel('Dataset Coverage (%)', color='#e74c3c')
ax2.plot(df_thresh['Threshold'], df_thresh['Coverage'], color='#e74c3c', marker='^', lw=2, linestyle='--', label='Coverage')
ax2.tick_params(axis='y', labelcolor='#e74c3c')
fig.legend(loc='lower left', bbox_to_anchor=(0.1, -0.1), ncol=3)
plt.tight_layout()
plt.savefig(PLTDIR / '18_deployment_thresholds.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 19: Class-Specific Thresholds ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
cls_threshs = np.arange(0.4, 1.0, 0.05)
for cls_idx, cls in enumerate(class_names):
    cls_f1s = []
    for t in cls_threshs:
        mask = (all_p == cls_idx) & (max_probs >= t)
        if mask.sum() > 0:
            cls_f1s.append(f1_score(all_l[mask] == cls_idx, all_p[mask] == cls_idx, zero_division=0))
        else:
            cls_f1s.append(0)
    opt_idx = np.argmax(cls_f1s)
    ax.plot(cls_threshs, cls_f1s, color=colors[cls_idx], marker='o', lw=2, 
            label=f'{cls} (Opt @ {cls_threshs[opt_idx]:.2f})')
ax.set_title('Class-Specific F1 vs. Confidence Threshold - DR', fontsize=14, fontweight='bold')
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('F1-Score')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLTDIR / '19_class_specific_thresholds.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── PLOT 20: Deep Feature Maps ───────────────────────────────────────────────
print("\n🧠 Extracting Deep Feature Maps...")
mask = (all_l == 0) & (all_p == 0)
idx = np.where(mask)[0][0] if mask.any() else 0
img = cv2.imread(test_s[idx][0])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
x_feat = get_tensor(img)

feature_maps = {}
def get_hook(name):
    def hook(module, input, output):
        feature_maps[name] = output.detach().cpu()
    return hook

model.backbone.feature_extractor[1].register_forward_hook(get_hook('Early_Layer'))
model.backbone.feature_extractor[4].register_forward_hook(get_hook('Mid_Layer'))
model.backbone.feature_extractor[7].register_forward_hook(get_hook('Late_Layer'))

with torch.no_grad():
    _ = model(x_feat)

fig, axes = plt.subplots(3, 9, figsize=(20, 7))
fig.suptitle('Hierarchical Feature Extraction - DR Retina', fontsize=16, fontweight='bold')
layers = ['Early_Layer', 'Mid_Layer', 'Late_Layer']
layer_names = ['Early Features (Edges)', 'Mid Features (Anatomy)', 'Late Features (Pathology)']

for row_idx, (layer, title) in enumerate(zip(layers, layer_names)):
    if layer in feature_maps:
        fm = feature_maps[layer].squeeze()
        if len(fm.shape) >= 2:
            n_channels = fm.shape[0]
            channel_idxs = np.linspace(0, n_channels - 1, 8, dtype=int)
            for col_idx, c_idx in enumerate(channel_idxs):
                if c_idx < n_channels:
                    channel_img = fm[c_idx].numpy()
                    channel_img = (channel_img - channel_img.min()) / (channel_img.max() - channel_img.min() + 1e-8)
                    axes[row_idx, col_idx].imshow(channel_img, cmap='magma')
                    axes[row_idx, col_idx].set_title(f'Ch {c_idx}', fontsize=9)
                    axes[row_idx, col_idx].axis('off')
    axes[row_idx, -1].axis('off')
    axes[row_idx, -1].text(1.1, 0.5, title, transform=axes[row_idx, -1].transAxes, 
                          fontsize=12, fontweight='bold', va='center', rotation=270)

plt.tight_layout()
plt.savefig(PLTDIR / '20_feature_maps_hierarchy.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("✅ ALL 20 PLOTS GENERATED SUCCESSFULLY FOR DIABETIC RETINOPATHY!")
print(f"📁 Location: {PLTDIR}")
print("="*60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — Download Entire Kaggle Working Space (FIXED)                        ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

import shutil
from IPython.display import FileLink

def download_workspace(output_name='DR.zip'):
    """Zip the entire /kaggle/working directory and provide download link."""
    
    zip_path = OUTDIR / output_name
    
    # Remove old zip if exists
    if zip_path.exists():
        zip_path.unlink()
    
    print("📦 Scanning /kaggle/working for files to archive...")
    
    total_size = 0
    for item in OUTDIR.rglob('*'):
        if item.is_file() and item != zip_path:
            size_mb = item.stat().st_size / (1024 * 1024)
            total_size += size_mb
            if size_mb > 0.1:
                rel_path = item.relative_to(OUTDIR)
                print(f"      📄 {rel_path}: {size_mb:.2f} MB")
    
    print(f"\n   Total uncompressed size: {total_size:.2f} MB")
    
    # ✅ FIXED: Removed 'with' statement
    print("\n🔒 Compressing... (this may take a minute for large models)")
    archive_name = str(zip_path.with_suffix(''))
    shutil.make_archive(archive_name, 'zip', OUTDIR)
    
    # Re-find the zip in case of naming quirks
    final_zip = Path(archive_name + '.zip')
    final_size = final_zip.stat().st_size / (1024 * 1024)
    
    print(f"✅ Archive created successfully!")
    print(f"   File: {final_zip.name}")
    print(f"   Size: {final_size:.2f} MB")
    
    # Provide download link
    print("\n⬇️ Click the link below to download:")
    display(FileLink(str(final_zip), result_html_prefix="🔗 Download Link: "))
    
    return str(final_zip)

# Run it
zip_file = download_workspace()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Setup, Imports, and Global Configuration (HAM10000)                 ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
!pip install -q timm einops albumentations opencv-python-headless

import os, time, random, copy, warnings, re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
import cv2
from tqdm.auto import tqdm
from scipy.ndimage import zoom

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2
from einops import rearrange

from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, roc_curve, 
                             confusion_matrix, classification_report, precision_recall_curve,
                             average_precision_score, matthews_corrcoef, cohen_kappa_score,
                             log_loss, brier_score_loss)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTDIR = Path('/kaggle/working')
CKDIR = OUTDIR / 'checkpoints'
PLTDIR = OUTDIR / 'plots'
for d in [CKDIR, PLTDIR]: d.mkdir(exist_ok=True)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

# ✅ HAM10000 CONFIG - 7 CLASSES
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
class_full_names = ['Actinic Keratoses', 'Basal Cell Carcinoma', 'Benign Keratosis', 
                    'Dermatofibroma', 'Melanoma', 'Melanocytic Nevi', 'Vascular Lesions']
NUM_CLASSES = 7
COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f1c40f', '#1abc9c', '#e67e22']

print(f"✅ Setup complete on {DEVICE} for HAM10000 (7 Classes)")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Hyperparameters                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
HPARAMS = {
    'img_size': 224, 'batch_size': 32, 'num_epochs': 40,
    'lr_backbone': 1e-5, 'lr_head': 1e-4, 'weight_decay': 1e-4,
    'warmup_epochs': 5, 'freeze_epochs': 8, 'patience': 10,
    'd_model': 384, 'nhead': 6, 'n_layers': 4, 'dim_ffn': 1536,
    'head_dropout': 0.3, 'label_smoothing': 0.05,
    'mixup_alpha': 0.2, 'cutmix_alpha': 1.0, 'mixup_prob': 0.3,
    'backbone': 'convnext_base', 'num_classes': NUM_CLASSES
}

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Data Augmentations (Dermoscopy Specific)                             ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
def get_skin_transforms(split, img_size=224):
    norm = [A.Normalize(mean=MEAN, std=STD), ToTensorV2()]
    if split != 'train':
        return A.Compose([A.Resize(height=img_size, width=img_size), A.CenterCrop(img_size, img_size)] + norm)
    
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5), # Skin lesions are mostly rotation invariant
        A.Rotate(limit=180, p=0.5, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.CoarseDropout(max_holes=8, max_height=16, max_width=16, min_holes=1, p=0.3),
    ] + norm)

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Dataset Class                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
class SkinDataset(Dataset):
    def __init__(self, samples, class_names, transform=None):
        self.samples = samples
        self.transform = transform
        self.class_to_idx = {n: i for i, n in enumerate(class_names)}
        
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        p, l = self.samples[idx]
        l = self.class_to_idx[l]
        img = cv2.imread(p)
        if img is None: img = np.array(Image.open(p).convert('RGB'))
        else: img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform: img = self.transform(image=img)['image']
        return img.float(), l
    
    def class_weights(self):
        labels = [self.class_to_idx[l] for _, l in self.samples]
        counts = torch.bincount(torch.tensor(labels), minlength=len(class_names)).float()
        beta = 0.9999
        effective_num = 1.0 - torch.pow(beta, counts)
        weights = ((1.0 - beta) / (effective_num + 1e-8)) / len(class_names) * len(class_names)
        return weights

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Load HAM10000 Dataset                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
def load_ham10000():
    print("📂 Loading HAM10000 dataset...")
    base_dir = None
    for p in Path('/kaggle/input/datasets/nikitachaulagain/dataasets/skin cancer/skin cancer').rglob('HAM10000_metadata.csv'):
        base_dir = p.parent
        print(f"✅ Found dataset at: {base_dir}")
        break
    if not base_dir: raise ValueError("HAM10000 dataset not found!")
    
    df_meta = pd.read_csv(base_dir / 'HAM10000_metadata.csv')
    img_dirs = [base_dir / 'HAM10000_images_part_1', base_dir / 'HAM10000_images_part_2']
    
    def get_path(img_id):
        for d in img_dirs:
            p = d / f"{img_id}.jpg"
            if p.exists(): return str(p)
        return None
        
    df_meta['path'] = df_meta['image_id'].apply(get_path)
    df_meta = df_meta.dropna(subset=['path'])
    return list(zip(df_meta['path'], df_meta['dx']))

samples = load_ham10000()
print(f"✅ Total images loaded: {len(samples)}")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Train/Val/Test Split                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
labels = [class_names.index(l) for _, l in samples]
train_s, temp_s = train_test_split(samples, test_size=0.3, stratify=labels, random_state=SEED)
val_s, test_s = train_test_split(temp_s, test_size=0.5, stratify=[class_names.index(l) for _, l in temp_s], random_state=SEED)

df_all = pd.DataFrame(samples, columns=['path', 'label'])
print(f"Split -> Train: {len(train_s)}, Val: {len(val_s)}, Test: {len(test_s)}")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Create Dataloaders                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
train_ds = SkinDataset(train_s, class_names, get_skin_transforms('train'))
val_ds = SkinDataset(val_s, class_names, get_skin_transforms('val'))
test_ds = SkinDataset(test_s, class_names, get_skin_transforms('test'))

class_weights = train_ds.class_weights()
sample_weights = [class_weights[class_names.index(l)] for _, l in train_s]

loaders = {
    'train': DataLoader(train_ds, batch_size=HPARAMS['batch_size'], sampler=WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True), num_workers=2, pin_memory=True),
    'val': DataLoader(val_ds, batch_size=HPARAMS['batch_size'], shuffle=False, num_workers=2, pin_memory=True),
    'test': DataLoader(test_ds, batch_size=HPARAMS['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
}

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Model Architecture (ConvNeXt + Hybrid ViT)                           ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0): super().__init__(); self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training: return x
        return x * (torch.rand((x.shape[0], 1, 1), dtype=torch.float32, device=x.device) * float(1.0 - self.drop_prob))

class WindowAttention(nn.Module):
    def __init__(self, d_model, nhead, window_size=7, dropout=0.1):
        super().__init__()
        self.nhead, self.window_size, self.head_dim = nhead, window_size, d_model // nhead
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3); self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.proj_drop, self.attn_weights = nn.Dropout(dropout), nn.Dropout(dropout), None
    def forward(self, x, H, W):
        B, N, C = x.shape; x = x.reshape(B, H, W, C)
        pad_h, pad_w = (self.window_size - H % self.window_size) % self.window_size, (self.window_size - W % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0: x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        Hp, Wp = x.shape[1], x.shape[2]
        x = x.reshape(B, Hp//self.window_size, self.window_size, Wp//self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(-1, self.window_size**2, C)
        qkv = self.qkv(x).reshape(-1, self.window_size**2, 3, self.nhead, self.head_dim).permute(2,0,3,1,4)
        attn = (qkv[0] @ qkv[1].transpose(-2,-1)) * self.scale; attn = attn.softmax(dim=-1); self.attn_weights = attn.detach()
        x = self.proj_drop(self.proj((self.attn_drop(attn) @ qkv[2]).transpose(1,2).reshape(-1, self.window_size**2, C)))
        x = x.reshape(B, Hp//self.window_size, Wp//self.window_size, self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(B, Hp, Wp, C)
        return x[:, :H, :W, :].reshape(B, -1, C) if pad_h > 0 or pad_w > 0 else x.reshape(B, -1, C)

class TransformerBlockWithWindow(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim, window_size=7, attn_drop=0.1, ffn_drop=0.1, drop_path=0.0):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.global_attn = nn.MultiheadAttention(d_model, nhead, dropout=attn_drop, batch_first=True)
        self.window_attn = WindowAttention(d_model, nhead, window_size, attn_drop)
        self.gate = nn.Sequential(nn.Linear(d_model*2, d_model), nn.Sigmoid())
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(ffn_drop), nn.Linear(ffn_dim, d_model), nn.Dropout(ffn_drop))
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.attn_weights, self.window_attn_weights = None, None
    def forward(self, x, H, W):
        x_norm = self.norm1(x)
        global_out, global_attn = self.global_attn(x_norm, x_norm, x_norm, need_weights=True)
        self.attn_weights = global_attn.detach()
        window_out = self.window_attn(x_norm[:, 1:, :], H, W); self.window_attn_weights = self.window_attn.attn_weights
        gate = self.gate(torch.cat([global_out[:, 1:, :], window_out], dim=-1))
        combined = torch.cat([global_out[:, :1, :], gate * global_out[:, 1:, :] + (1-gate) * window_out], dim=1)
        x = x + self.drop_path(combined)
        return x + self.drop_path(self.ffn(self.norm2(x)))

class ImprovedBackbone(nn.Module):
    def __init__(self): super().__init__(); self.backbone = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1); self.feature_extractor = self.backbone.features
    def forward(self, x): return self.feature_extractor(x)

class EnhancedHViT(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536):
        super().__init__(); self.d_model = d_model; self.img_size = img_size; self.backbone = ImprovedBackbone()
        with torch.no_grad(): _, c, h, w = self.backbone(torch.zeros(1, 3, img_size, img_size)).shape
        self.spatial_h, self.spatial_w = h, w
        self.proj = nn.Sequential(nn.Conv2d(c, d_model, 1, bias=False), nn.BatchNorm2d(d_model), nn.GELU(), nn.Conv2d(d_model, d_model, 3, padding=1, bias=False), nn.BatchNorm2d(d_model), nn.GELU())
        self.norm_proj = nn.LayerNorm(d_model); self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, h*w + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02); nn.init.trunc_normal_(self.pos_embed, std=0.02)
        dpr = [x.item() for x in torch.linspace(0, 0.15, n_layers)]
        self.encoder = nn.ModuleList([TransformerBlockWithWindow(d_model, nhead, dim_ffn, min(7, h, w), drop_path=dpr[i]) for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model//2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d_model//2, d_model//4), nn.GELU(), nn.Dropout(0.15), nn.Linear(d_model//4, num_classes))
    def forward(self, x):
        B = x.size(0); tokens = rearrange(self.proj(self.backbone(x)), 'b d h w -> b (h w) d')
        tokens = self.norm_proj(tokens); tokens = torch.cat([self.cls_token.expand(B,-1,-1), tokens], dim=1) + self.pos_embed
        for block in self.encoder: tokens = block(tokens, self.spatial_h, self.spatial_w)
        return self.head(self.norm(tokens)[:, 0])
    def get_attentions(self):
        g, w = [], []
        for b in self.encoder:
            if b.attn_weights is not None: g.append(b.attn_weights)
            if b.window_attn_weights is not None: w.append(b.window_attn_weights)
        return g, w
    def unfreeze_backbone_partial(self, n=3):
        for p in self.backbone.feature_extractor.parameters(): p.requires_grad_(False)
        for child in list(self.backbone.feature_extractor.children())[-n:]:
            for p in child.parameters(): p.requires_grad_(True)

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Training Functions                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
class SmoothCELoss(nn.Module):
    def __init__(self, label_smoothing=0.05): super().__init__(); self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    def forward(self, inputs, targets): return self.criterion(inputs, targets)

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha); idx = torch.randperm(x.size(0)).to(x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha); idx = torch.randperm(x.size(0)).to(x.device)
    W, H = x.size(2), x.size(3); r = np.sqrt(1.0 - lam)
    cw, ch = int(W*r), int(H*r); cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = np.clip(cx - cw//2, 0, W), np.clip(cy - ch//2, 0, H)
    x2, y2 = np.clip(cx + cw//2, 0, W), np.clip(cy + ch//2, 0, H)
    mixed = x.clone(); mixed[:, :, x1:x2, y1:y2] = x[idx, :, x1:x2, y1:y2]
    return mixed, y, y[idx], 1 - (x2-x1)*(y2-y1)/(W*H)

def train_one_epoch(model, loader, optimizer, scaler, criterion, epoch):
    model.train(); total_loss, preds, labs = 0, [], []
    for images, labels in tqdm(loader, desc=f'Epoch {epoch+1}', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE); optimizer.zero_grad(set_to_none=True)
        mixed = False
        if np.random.random() < HPARAMS['mixup_prob']:
            if np.random.random() < 0.5: images, la, lb, lam = mixup_data(images, labels, HPARAMS['mixup_alpha']); mixed = True
            else: images, la, lb, lam = cutmix_data(images, labels, HPARAMS['cutmix_alpha']); mixed = True
        with autocast():
            out = model(images)
            loss = lam * criterion(out, la) + (1-lam) * criterion(out, lb) if mixed else criterion(out, labels)
            if not mixed: preds.extend(out.argmax(1).cpu().numpy()); labs.extend(labels.cpu().numpy())
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0); scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    acc = accuracy_score(labs, preds) if labs else 0; f1 = f1_score(labs, preds, average='weighted', zero_division=0) if labs else 0
    return total_loss/len(loader), acc, f1

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval(); total_loss, preds, labs, probs = 0, [], [], []
    for images, labels in tqdm(loader, desc='Validating', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with autocast(): out = model(images); loss = criterion(out, labels)
        total_loss += loss.item(); p = F.softmax(out, dim=1)
        preds.extend(out.argmax(1).cpu().numpy()); labs.extend(labels.cpu().numpy()); probs.extend(p.cpu().numpy())
    acc = accuracy_score(labs, preds); f1 = f1_score(labs, preds, average='weighted', zero_division=0)
    return total_loss/len(loader), acc, f1, preds, labs, probs

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Main Training Loop                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
print("\n🚀 Starting Training for HAM10000 Skin Cancer...")
model = EnhancedHViT(num_classes=NUM_CLASSES).to(DEVICE)
criterion = SmoothCELoss(label_smoothing=HPARAMS['label_smoothing'])
backbone_params = [p for n, p in model.named_parameters() if 'backbone' in n]
head_params = [p for n, p in model.named_parameters() if 'backbone' not in n]
optimizer = optim.AdamW([{'params': backbone_params, 'lr': HPARAMS['lr_backbone']}, {'params': head_params, 'lr': HPARAMS['lr_head']}], weight_decay=HPARAMS['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=HPARAMS['num_epochs']//3, T_mult=2, eta_min=1e-7)
scaler = GradScaler()

best_f1, patience_counter = 0, 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': [], 'lr': []}

# Freeze backbone initially
for p in model.backbone.feature_extractor.parameters(): p.requires_grad_(False)

for epoch in range(HPARAMS['num_epochs']):
    if epoch == HPARAMS['freeze_epochs']:
        print("\n🔄 Unfreezing last 3 backbone blocks..."); model.unfreeze_backbone_partial(3)
        optimizer.param_groups[0]['lr'] = HPARAMS['lr_backbone']
        
    train_loss, train_acc, train_f1 = train_one_epoch(model, loaders['train'], optimizer, scaler, criterion, epoch)
    val_loss, val_acc, val_f1, _, _, _ = validate(model, loaders['val'], criterion)
    scheduler.step()
    
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc); history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1); history['lr'].append(optimizer.param_groups[0]['lr'])
    
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.3f}, F1={train_f1:.3f} | Val Acc={val_acc:.3f}, F1={val_f1:.3f}")
    
    if val_f1 > best_f1:
        best_f1 = val_f1; patience_counter = 0
        torch.save({'model': model.state_dict(), 'f1': val_f1, 'acc': val_acc, 'epoch': epoch + 1, 'class_names': class_names}, CKDIR / 'best_model.pth')
        print(f"   ✅ Best model saved (Val F1={val_f1:.3f})")
    else:
        patience_counter += 1
        if patience_counter >= HPARAMS['patience']: print(f"⏹️ Early stopping at epoch {epoch+1}"); break

print("\n✅ Training complete!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 13-15 — Interactive Prediction + Batch Predict + Quick Test               ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
from PIL import Image as PILImage

print("📦 Loading Skin Cancer model...")
try: model
except NameError: model = EnhancedHViT(num_classes=7).to(DEVICE)

if (CKDIR / 'best_model.pth').exists():
    model.load_state_dict(torch.load(CKDIR / 'best_model.pth', map_location=DEVICE, weights_only=False)['model'])
    print("✅ Model loaded.")
else: print("❌ Checkpoint not found!")

model.eval()

SEVERITY_MAP = {
    'mel': '🚨 HIGH RISK - Malignant Melanoma', 'bcc': '⚠️ MEDIUM - Basal Cell Carcinoma',
    'akiec': '⚠️ MEDIUM - Pre-cancerous Actinic Keratosis', 'bkl': '✅ LOW - Benign Keratosis',
    'df': '✅ LOW - Benign Dermatofibroma', 'nv': '✅ LOW - Common Mole (Nevi)',
    'vasc': '✅ LOW - Benign Vascular Lesion'
}

def predict_single_image(image_array):
    img_resized = cv2.resize(image_array, (224, 224))
    t = torch.from_numpy(img_resized).float().permute(2, 0, 1) / 255.0
    t = ((t - torch.tensor(MEAN).view(3,1,1)) / torch.tensor(STD).view(3,1,1)).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): 
        with autocast(): logits = model(t); probs = F.softmax(logits, dim=1)
        conf, pred_idx = probs.max(dim=1)
    return pred_idx.item(), conf.item(), probs.cpu().numpy()[0]

def generate_prediction_plot(image_rgb, pred_idx, confidence, probs):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image_rgb)
    color = 'red' if class_names[pred_idx] in ['mel', 'bcc', 'akiec'] else 'green'
    axes[0].set_title(f"{SEVERITY_MAP.get(class_names[pred_idx], '')}\nConfidence: {confidence:.2%}", fontsize=13, fontweight='bold', color=color)
    axes[0].axis('off')
    
    colors_bar = ['#e74c3c' if i == pred_idx else '#95a5a6' for i in range(7)]
    bars = axes[1].barh(class_full_names, probs * 100, color=colors_bar, edgecolor='black', linewidth=0.5)
    axes[1].set_xlabel('Probability (%)'); axes[1].set_title('Class Probabilities', fontweight='bold')
    axes[1].set_xlim(0, 105)
    for bar, prob in zip(bars, probs): axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{prob:.1f}%', va='center', fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3); plt.tight_layout()
    return fig

upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='📤 Upload Skin Image', button_style='info')
path_input = widgets.Text(value='', placeholder='/kaggle/input/.../image.jpg', description='📁 Or path:', layout={'width': '600px'})
predict_button = widgets.Button(description='🔍 Predict', button_style='success', layout={'width': '200px'})
output_area = widgets.Output()

def on_predict_clicked(b):
    with output_area:
        clear_output(wait=True); image_rgb = None
        if upload_widget.value:
            content = list(upload_widget.value.values())[0]['content']
            image_rgb = np.array(PILImage.open(io.BytesIO(content)).convert('RGB'))
        elif path_input.value.strip():
            p = Path(path_input.value.strip())
            if p.exists(): image_rgb = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
            else: print("❌ File not found"); return
        else: print("⚠️ Provide an image"); return
            
        pred_idx, confidence, probs = predict_single_image(image_rgb)
        print(f"\n{'='*50}\n   PREDICTION: {class_names[pred_idx].upper()}\n   {SEVERITY_MAP[class_names[pred_idx]]}\n   CONFIDENCE: {confidence:.2%}\n{'='*50}")
        fig = generate_prediction_plot(image_rgb, pred_idx, confidence, probs)
        plt.show()

predict_button.on_click(on_predict_clicked)
print("╔══════════════════════════════════════════════════════════════╗")
print("║        🔬 HAM10000 SKIN CANCER CLASSIFICATION - INFERENCE   ║")
print("╚══════════════════════════════════════════════════════════════╝\n")
display(widgets.VBox([upload_widget, widgets.Label("── OR ──"), path_input, widgets.Label(""), predict_button, output_area]))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  STANDALONE HEATMAP GENERATOR - HAM10000 (Grad-CAM + Attention)                ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import zoom
import torchvision.models as models
from einops import rearrange
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKDIR = OUTDIR / 'checkpoints'; PLTDIR = OUTDIR / 'plots'; PLTDIR.mkdir(exist_ok=True)
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
class_full_names = ['Actinic Keratoses', 'Basal Cell Carcinoma', 'Benign Keratosis', 'Dermatofibroma', 'Melanoma', 'Melanocytic Nevi', 'Vascular Lesions']

# (Assuming EnhancedHViT is already defined in memory from Cell 8. If running standalone, copy class here)
model = EnhancedHViT(num_classes=7).to(DEVICE)
model.load_state_dict(torch.load(CKDIR / 'best_model.pth', map_location=DEVICE, weights_only=False)['model'])
model.eval()

def get_tensor(img_input):
    if len(img_input.shape) == 2: img_input = np.stack([img_input]*3, axis=-1)
    t = torch.from_numpy(cv2.resize(img_input, (224, 224))).float().permute(2,0,1)/255.0
    return ((t - torch.tensor(MEAN).view(3,1,1)) / torch.tensor(STD).view(3,1,1)).unsqueeze(0).to(DEVICE)

class SkinHeatmapGenerator:
    def __init__(self, model):
        self.model = model; self.model.eval(); self.cnn_f, self.cnn_g = None, None
        last_conv = None
        for m in self.model.backbone.feature_extractor.modules():
            if isinstance(m, nn.Conv2d): last_conv = m
        if last_conv:
            last_conv.register_forward_hook(lambda m, i, o: setattr(self, 'cnn_f', o.detach()))
            last_conv.register_backward_hook(lambda m, gi, go: setattr(self, 'cnn_g', go[0].detach()))

    def get_cnn(self, x, target):
        out = self.model(x); self.model.zero_grad()
        oh = torch.zeros_like(out); oh[0, target] = 1
        out.backward(gradient=oh, retain_graph=True)
        if self.cnn_f is None: return None
        weights = self.cnn_g.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.cnn_f).sum(1)).squeeze().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8) if cam.max() > cam.min() else cam

    def get_attn(self, x):
        with torch.no_grad(): self.model(x)
        g, _ = self.model.get_attentions()
        if not g: return None
        a = g[-1]
        a = a[0, :, 0, 1:].mean(0).cpu().numpy() if a.dim() == 4 else a[0, 0, 1:].cpu().numpy()
        a = a.reshape(self.model.spatial_h, self.model.spatial_w)
        return (a - a.min()) / (a.max() - a.min() + 1e-8) if a.max() > a.min() else a

    def draw(self, img_path, target, save_path):
        rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        sz = rgb.shape[:2]; t = get_tensor(rgb)
        c, a = self.get_cnn(t.clone(), target), self.get_attn(t.clone())
        comb = 0.5 * c + 0.5 * a if c is not None and a is not None else c
        if comb is not None and comb.max() > comb.min(): comb = (comb - comb.min()) / (comb.max() - comb.min() + 1e-8)

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(rgb); axes[0].set_title(f"True: {class_full_names[target]}", fontweight='bold'); axes[0].axis('off')
        for i, (hmap, title) in enumerate(zip([c, a, comb], ['Grad-CAM', 'Attention', 'Combined'])):
            if hmap is not None:
                hr = zoom(hmap, (sz[0]/hmap.shape[0], sz[1]/hmap.shape[1]), order=1)
                colored = (plt.cm.jet(hr)[:,:,:3] * 255).astype(np.uint8)
                axes[i+1].imshow(cv2.addWeighted(rgb, 0.6, colored, 0.4, 0))
            else: axes[i+1].imshow(rgb)
            axes[i+1].set_title(title, fontweight='bold'); axes[i+1].axis('off')
        plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()

# Generate for test set
hm_gen = SkinHeatmapGenerator(model)
print("🔥 Generating Heatmaps for HAM10000...")
for cls_idx in range(7):
    cls_tests = [s for s in test_s if class_names.index(s[1]) == cls_idx]
    if cls_tests:
        print(f"\n📊 {class_full_names[cls_idx]}:")
        for p, _ in cls_tests[:2]: hm_gen.draw(p, cls_idx, PLTDIR / f'hm_{class_names[cls_idx]}_{Path(p).stem}.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  ULTIMATE MEGA CELL: ALL 20 PLOTS + EXACT METRICS (HAM10000 7-Classes)         ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

import os, time, random, warnings, re
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, cv2
from tqdm.auto import tqdm
from scipy.ndimage import zoom
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from sklearn.metrics import (accuracy_score, f1_score, roc_curve, auc, confusion_matrix, 
                             classification_report, precision_recall_curve, average_precision_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss, brier_score_loss)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from torch.cuda.amp import autocast
from einops import rearrange
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTDIR = Path('/kaggle/working')
PLTDIR = OUTDIR / 'plots/ham10000_final'; PLTDIR.mkdir(exist_ok=True, parents=True)
CKDIR = OUTDIR / '/kaggle/input/models/nikitachaulagain/skincaner/transformers/default/1/checkpoints/'
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
class_full_names = ['Actinic Keratoses', 'Basal Cell Carcinoma', 'Benign Keratosis', 
                    'Dermatofibroma', 'Melanoma', 'Melanocytic Nevi', 'Vascular Lesions']
COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f1c40f', '#1abc9c', '#e67e22']

# ─── 1. RE-DEFINE MODEL ARCHITECTURE (Required for loading weights) ──────────────
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0): super().__init__(); self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training: return x
        return x * (torch.rand((x.shape[0], 1, 1), dtype=torch.float32, device=x.device) * float(1.0 - self.drop_prob))

class WindowAttention(nn.Module):
    def __init__(self, d_model, nhead, window_size=7, dropout=0.1):
        super().__init__()
        self.nhead, self.window_size, self.head_dim = nhead, window_size, d_model // nhead
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3); self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.proj_drop, self.attn_weights = nn.Dropout(dropout), nn.Dropout(dropout), None
    def forward(self, x, H, W):
        B, N, C = x.shape; x = x.reshape(B, H, W, C)
        pad_h, pad_w = (self.window_size - H % self.window_size) % self.window_size, (self.window_size - W % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0: x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        Hp, Wp = x.shape[1], x.shape[2]
        x = x.reshape(B, Hp//self.window_size, self.window_size, Wp//self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(-1, self.window_size**2, C)
        qkv = self.qkv(x).reshape(-1, self.window_size**2, 3, self.nhead, self.head_dim).permute(2,0,3,1,4)
        attn = (qkv[0] @ qkv[1].transpose(-2,-1)) * self.scale; attn = attn.softmax(dim=-1); self.attn_weights = attn.detach()
        x = self.proj_drop(self.proj((self.attn_drop(attn) @ qkv[2]).transpose(1,2).reshape(-1, self.window_size**2, C)))
        x = x.reshape(B, Hp//self.window_size, Wp//self.window_size, self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(B, Hp, Wp, C)
        return x[:, :H, :W, :].reshape(B, -1, C) if pad_h > 0 or pad_w > 0 else x.reshape(B, -1, C)

class TransformerBlockWithWindow(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim, window_size=7, attn_drop=0.1, ffn_drop=0.1, drop_path=0.0):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.global_attn = nn.MultiheadAttention(d_model, nhead, dropout=attn_drop, batch_first=True)
        self.window_attn = WindowAttention(d_model, nhead, window_size, attn_drop)
        self.gate = nn.Sequential(nn.Linear(d_model*2, d_model), nn.Sigmoid())
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(ffn_drop), nn.Linear(ffn_dim, d_model), nn.Dropout(ffn_drop))
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.attn_weights, self.window_attn_weights = None, None
    def forward(self, x, H, W):
        x_norm = self.norm1(x)
        global_out, global_attn = self.global_attn(x_norm, x_norm, x_norm, need_weights=True)
        self.attn_weights = global_attn.detach()
        window_out = self.window_attn(x_norm[:, 1:, :], H, W); self.window_attn_weights = self.window_attn.attn_weights
        gate = self.gate(torch.cat([global_out[:, 1:, :], window_out], dim=-1))
        combined = torch.cat([global_out[:, :1, :], gate * global_out[:, 1:, :] + (1-gate) * window_out], dim=1)
        x = x + self.drop_path(combined)
        return x + self.drop_path(self.ffn(self.norm2(x)))

class ImprovedBackbone(nn.Module):
    def __init__(self): super().__init__(); self.backbone = models.convnext_base(weights=None); self.feature_extractor = self.backbone.features
    def forward(self, x): return self.feature_extractor(x)

class EnhancedHViT(nn.Module):
    def __init__(self, num_classes=7, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536):
        super().__init__(); self.d_model = d_model; self.img_size = img_size; self.backbone = ImprovedBackbone()
        with torch.no_grad(): _, c, h, w = self.backbone(torch.zeros(1, 3, img_size, img_size)).shape
        self.spatial_h, self.spatial_w = h, w
        self.proj = nn.Sequential(nn.Conv2d(c, d_model, 1, bias=False), nn.BatchNorm2d(d_model), nn.GELU(), nn.Conv2d(d_model, d_model, 3, padding=1, bias=False), nn.BatchNorm2d(d_model), nn.GELU())
        self.norm_proj = nn.LayerNorm(d_model); self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, h*w + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02); nn.init.trunc_normal_(self.pos_embed, std=0.02)
        dpr = [x.item() for x in torch.linspace(0, 0.15, n_layers)]
        self.encoder = nn.ModuleList([TransformerBlockWithWindow(d_model, nhead, dim_ffn, min(7, h, w), drop_path=dpr[i]) for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model//2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d_model//2, d_model//4), nn.GELU(), nn.Dropout(0.15), nn.Linear(d_model//4, num_classes))
    def forward(self, x):
        B = x.size(0); tokens = rearrange(self.proj(self.backbone(x)), 'b d h w -> b (h w) d')
        tokens = self.norm_proj(tokens); tokens = torch.cat([self.cls_token.expand(B,-1,-1), tokens], dim=1) + self.pos_embed
        for block in self.encoder: tokens = block(tokens, self.spatial_h, self.spatial_w)
        return self.head(self.norm(tokens)[:, 0])
    def get_attentions(self):
        g, w = [], []
        for b in self.encoder:
            if b.attn_weights is not None: g.append(b.attn_weights)
            if b.window_attn_weights is not None: w.append(b.window_attn_weights)
        return g, w

def get_tensor(img):
    if len(img.shape) == 2: img = np.stack([img]*3, axis=-1)
    t = torch.from_numpy(cv2.resize(img, (224, 224))).float().permute(2,0,1)/255.0
    return ((t - torch.tensor(MEAN).view(3,1,1)) / torch.tensor(STD).view(3,1,1)).unsqueeze(0).to(DEVICE)

# ─── 2. LOAD MODEL & TEST DATA ──────────────────────────────────────────────────
print("📦 Loading HAM10000 Model & Test Data...")
model = EnhancedHViT(num_classes=7).to(DEVICE)
model.load_state_dict(torch.load(CKDIR / 'best_model.pth', map_location=DEVICE, weights_only=False)['model'])
model.eval()

base_dir = None
for p in Path('/kaggle/input').rglob('HAM10000_metadata.csv'):
    base_dir = p.parent; break
df_meta = pd.read_csv(base_dir / 'HAM10000_metadata.csv')
img_dirs = [base_dir / 'HAM10000_images_part_1', base_dir / 'HAM10000_images_part_2']
def get_path(img_id):
    for d in img_dirs:
        p = d / f"{img_id}.jpg"
        if p.exists(): return str(p)
    return None
df_meta['path'] = df_meta['image_id'].apply(get_path)
df_meta = df_meta.dropna(subset=['path'])
samples = list(zip(df_meta['path'], df_meta['dx']))

labels_list = [class_names.index(l) for _, l in samples]
_, temp_s = train_test_split(samples, test_size=0.3, stratify=labels_list, random_state=42)
_, test_s = train_test_split(temp_s, test_size=0.5, stratify=[class_names.index(l) for _, l in temp_s], random_state=42)
df_all = pd.DataFrame(samples, columns=['path', 'label'])

# ─── 3. RUN INFERENCE ───────────────────────────────────────────────────────────
print("🧠 Running Inference on Test Set...")
all_p, all_l, all_pr = [], [], []
for p, l in tqdm(test_s, desc="Predicting"):
    rgb = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    with torch.no_grad(): out = model(get_tensor(rgb)); pr = F.softmax(out, dim=1)
    all_p.append(out.argmax(1).item()); all_l.append(class_names.index(l)); all_pr.append(pr.cpu().numpy()[0])

all_p, all_l, all_pr = np.array(all_p), np.array(all_l), np.array(all_pr)
max_probs = all_pr.max(axis=1); correct_mask = (all_p == all_l)
labels_bin = label_binarize(all_l, classes=list(range(7)))
cm = confusion_matrix(all_l, all_p)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 1: TRAINING HISTORY
# ─────────────────────────────────────────────────────────────────────────────────
try:
    history
    if len(history.get('train_loss', [])) > 0:
        fig, ax = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('HAM10000 Model Training Dynamics', fontsize=16, fontweight='bold', y=1.02)
        ax[0,0].plot(history['train_loss'], label='Train', color='#e74c3c', lw=2)
        ax[0,0].plot(history['val_loss'], label='Val', color='#3498db', lw=2)
        ax[0,0].set_title('Loss'); ax[0,0].legend(); ax[0,0].grid(True, alpha=0.3)
        ax[0,1].plot(history['train_acc'], label='Train', color='#e74c3c', lw=2)
        ax[0,1].plot(history['val_acc'], label='Val', color='#3498db', lw=2)
        ax[0,1].set_title('Accuracy'); ax[0,1].set_ylim(0,1.05); ax[0,1].legend(); ax[0,1].grid(True, alpha=0.3)
        ax[1,0].plot(history['val_f1'], label='Val F1', color='#2ecc71', lw=2)
        ax[1,0].set_title('F1 Score'); ax[1,0].set_ylim(0,1.05); ax[1,0].legend(); ax[1,0].grid(True, alpha=0.3)
        ax[1,1].plot(history['lr'], color='#9b59b6', lw=2)
        ax[1,1].set_title('Learning Rate'); ax[1,1].set_yscale('log'); ax[1,1].grid(True, alpha=0.3)
        plt.tight_layout(); plt.savefig(PLTDIR / '1_training_history.png', dpi=300, bbox_inches='tight'); plt.show()
except Exception as e: pass

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 2: CONFUSION MATRICES
# ─────────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Skin Cancer Confusion Matrix (HAM10000)', fontsize=16, fontweight='bold')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0], cbar=False)
axes[0].set_title('Absolute Counts'); axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=axes[1], cbar=False)
axes[1].set_title('Normalized (Recall per Class)'); axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted Label')
plt.tight_layout(); plt.savefig(PLTDIR / '2_confusion_matrices.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 3: ROC CURVES
# ─────────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
for i in range(7):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], all_pr[:, i])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=COLORS[i], lw=2, label=f'{class_full_names[i]} (AUC = {roc_auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k:', lw=1, alpha=0.5)
ax.set_title('ROC Curves - Skin Lesion Classification', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(PLTDIR / '3_roc_curves.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 4: PRECISION-RECALL CURVES
# ─────────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
for i in range(7):
    prec, rec, _ = precision_recall_curve(labels_bin[:, i], all_pr[:, i])
    ap = average_precision_score(labels_bin[:, i], all_pr[:, i])
    ax.plot(rec, prec, color=COLORS[i], lw=2, label=f'{class_names[i]} (AP = {ap:.3f})')
ax.set_title('Precision-Recall Curves - HAM10000', fontsize=14, fontweight='bold')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.3); ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
plt.tight_layout(); plt.savefig(PLTDIR / '4_precision_recall.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 5: PER-CLASS METRICS BARS
# ─────────────────────────────────────────────────────────────────────────────────
report = classification_report(all_l, all_p, target_names=class_names, output_dict=True)
metrics_df = pd.DataFrame(report).T.iloc[:-3, :3]
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(7); width = 0.25
bars1 = ax.bar(x - width, metrics_df['precision'], width, label='Precision', color='#3498db', edgecolor='black')
bars2 = ax.bar(x, metrics_df['recall'], width, label='Recall', color='#2ecc71', edgecolor='black')
bars3 = ax.bar(x + width, metrics_df['f1-score'], width, label='F1-Score', color='#e74c3c', edgecolor='black')
ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(class_names, rotation=0); ax.legend(); ax.set_ylim(0,1.15); ax.grid(axis='y', alpha=0.3)
for bars in [bars1, bars2, bars3]:
    for bar in bars: ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.savefig(PLTDIR / '5_per_class_metrics.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 6: CONFIDENCE DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(max_probs[correct_mask], bins=20, alpha=0.7, label='Correct Predictions', color='#2ecc71', edgecolor='black')
ax.hist(max_probs[~correct_mask], bins=20, alpha=0.7, label='Incorrect Predictions', color='#e74c3c', edgecolor='black')
ax.set_title('Model Confidence Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Maximum Class Probability'); ax.set_ylabel('Count'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(PLTDIR / '6_confidence_distribution.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 7: CALIBRATION CURVE
# ─────────────────────────────────────────────────────────────────────────────────
def calibration_curve(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1); accs, confs = [], []
    for i in range(n_bins):
        mask = (y_prob > bins[i]) & (y_prob <= bins[i+1])
        if i == n_bins - 1: mask = (y_prob >= bins[i]) & (y_prob <= bins[i+1])
        if mask.sum() > 0: accs.append(y_true[mask].mean()); confs.append(y_prob[mask].mean())
    return np.array(confs), np.array(accs)

fig, ax = plt.subplots(figsize=(8, 8))
for i, cls in enumerate(class_names):
    conf, acc = calibration_curve((all_l == i).astype(int), all_pr[:, i])
    if len(conf) > 0: ax.plot(conf, acc, marker='o', color=COLORS[i], lw=2, label=cls, markersize=5)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfectly Calibrated')
ax.set_title('Reliability Diagram', fontsize=14, fontweight='bold')
ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Fraction of Positives')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_xlim([0,1]); ax.set_ylim([0,1])
plt.tight_layout(); plt.savefig(PLTDIR / '7_calibration_curve.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 8: SAMPLE PREDICTIONS
# ─────────────────────────────────────────────────────────────────────────────────
correct_idxs, incorrect_idxs = np.where(correct_mask)[0], np.where(~correct_mask)[0]
np.random.seed(42)
samp_corr = np.random.choice(correct_idxs, min(8, len(correct_idxs)), replace=False)
samp_incorr = np.random.choice(incorrect_idxs, min(8, len(incorrect_idxs)), replace=False)
fig, axes = plt.subplots(2, 8, figsize=(24, 6))
for i, idx in enumerate(samp_corr):
    img = cv2.cvtColor(cv2.imread(test_s[idx][0]), cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img); axes[0, i].set_title(f"T:{class_names[all_l[idx]]}\nP:{class_names[all_p[idx]]}\n{max_probs[idx]:.1%}", fontsize=8, color='green'); axes[0, i].axis('off')
for i, idx in enumerate(samp_incorr):
    img = cv2.cvtColor(cv2.imread(test_s[idx][0]), cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img); axes[1, i].set_title(f"T:{class_names[all_l[idx]]}\nP:{class_names[all_p[idx]]}\n{max_probs[idx]:.1%}", fontsize=8, color='red'); axes[1, i].axis('off')
axes[0, 0].set_ylabel('CORRECT', fontsize=12, fontweight='bold', color='green', rotation=0, labelpad=60, va='center')
axes[1, 0].set_ylabel('INCORRECT', fontsize=12, fontweight='bold', color='red', rotation=0, labelpad=60, va='center')
fig.suptitle('Visual Inspection (T=True, P=Predicted)', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout(); plt.savefig(PLTDIR / '8_sample_predictions.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 9: VIOLIN PLOT PROBABILITIES
# ─────────────────────────────────────────────────────────────────────────────────
plot_data = [{'True Class': class_names[i], 'Assigned Probability': prob} for i in range(7) for prob in all_pr[all_l == i, i]]
df_probs = pd.DataFrame(plot_data)
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df_probs, x='True Class', y='Assigned Probability', palette=COLORS, inner='quartile', ax=ax)
ax.set_title('Probability Density for True Class', fontsize=14, fontweight='bold'); ax.set_ylim(0,1.05); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(PLTDIR / '9_probability_violin.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 10: t-SNE FEATURE VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────────
print("⏳ Extracting t-SNE features (may take a minute)...")
feature_tensor = None
def hook_fn(module, inp, out): global feature_tensor; feature_tensor = out
handle = model.norm.register_forward_hook(hook_fn)
tsne_indices = np.random.choice(len(test_s), min(800, len(test_s)), replace=False)
features_list, labels_tsne = [], []
for idx in tsne_indices:
    rgb = cv2.cvtColor(cv2.imread(test_s[idx][0]), cv2.COLOR_BGR2RGB)
    with torch.no_grad(): _ = model(get_tensor(rgb))
    features_list.append(feature_tensor[:, 0, :].cpu().numpy()[0]); labels_tsne.append(all_l[idx])
handle.remove()
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
tsne_results = tsne.fit_transform(np.array(features_list))
fig, ax = plt.subplots(figsize=(10, 8))
for i, cls in enumerate(class_names):
    mask = np.array(labels_tsne) == i
    ax.scatter(tsne_results[mask, 0], tsne_results[mask, 1], c=COLORS[i], label=cls, alpha=0.6, s=20, edgecolors='white', linewidth=0.5)
ax.set_title('t-SNE Visualization of Deep Features', fontsize=14, fontweight='bold'); ax.legend(markerscale=2); ax.grid(True, alpha=0.2)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(PLTDIR / '10_tsne_features.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 11: ERROR ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────────
errors = [{'True': class_names[all_l[i]], 'Predicted': class_names[all_p[i]]} for i in range(len(all_l)) if all_l[i] != all_p[i]]
if errors:
    error_df = pd.DataFrame(errors)
    error_counts = error_df.groupby(['True', 'Predicted']).size().reset_index(name='Count').sort_values(by='Count', ascending=False).head(10)
    fig, ax = plt.subplots(figsize=(10, 6))
    error_counts['Pair'] = error_counts['True'] + ' → ' + error_counts['Predicted']
    sns.barplot(data=error_counts, x='Count', y='Pair', palette='Reds_r', ax=ax)
    ax.set_title('Top Misclassification Pairs', fontsize=14, fontweight='bold')
    for i, v in enumerate(error_counts['Count']): ax.text(v + 0.5, i, str(v), color='black', va='center', fontweight='bold')
    plt.tight_layout(); plt.savefig(PLTDIR / '11_error_analysis.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 12: CLASS DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('HAM10000 Dataset Class Distribution', fontsize=16, fontweight='bold')
counts = df_all['label'].value_counts().reindex(class_names)
sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette=COLORS, edgecolor='black')
axes[0].set_title('Image Count per Class', fontsize=13)
for i, v in enumerate(counts.values): axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold', fontsize=11)
axes[1].pie(counts.values, labels=class_names, autopct='%1.1f%%', colors=COLORS, startangle=90, wedgeprops={'edgecolor': 'black', 'linewidth': 1.5}, textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title('Percentage Distribution', fontsize=13)
plt.tight_layout(); plt.savefig(PLTDIR / '12_class_distribution.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 13: SPLIT DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────────
labels_list_full = [class_names.index(l) for _, l in samples]
train_s_full, temp_s_full = train_test_split(samples, test_size=0.3, stratify=labels_list_full, random_state=42)
val_s_full, test_s_full = train_test_split(temp_s_full, test_size=0.5, stratify=[class_names.index(l) for _, l in temp_s_full], random_state=42)
train_df = pd.DataFrame(train_s_full, columns=['path', 'label']); train_df['split'] = 'Train'
val_df = pd.DataFrame(val_s_full, columns=['path', 'label']); val_df['split'] = 'Validation'
test_df = pd.DataFrame(test_s_full, columns=['path', 'label']); test_df['split'] = 'Test'
df_splits = pd.concat([train_df, val_df, test_df])
split_counts = df_splits.groupby(['split', 'label']).size().unstack(fill_value=0)[class_names]
fig, ax = plt.subplots(figsize=(10, 6))
split_counts.plot(kind='barh', stacked=True, ax=ax, color=COLORS, edgecolor='black', linewidth=0.5)
ax.set_title('Class Distribution Across Data Splits', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Images'); ax.set_ylabel('')
for i, split in enumerate(split_counts.index):
    ax.text(split_counts.loc[split].sum() + 10, i, f'Total: {split_counts.loc[split].sum()}', va='center', fontweight='bold')
plt.tight_layout(); plt.savefig(PLTDIR / '13_split_distribution.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 14: AVERAGE IMAGES PER CLASS
# ─────────────────────────────────────────────────────────────────────────────────
print("⏳ Computing average images...")
fig, axes = plt.subplots(1, 7, figsize=(21, 4))
fig.suptitle('Average Skin Lesion Image per Class', fontsize=14, fontweight='bold')
for i, cls in enumerate(class_names):
    cls_paths = df_all[df_all['label'] == cls]['path'].sample(n=min(300, len(df_all[df_all['label']==cls])), random_state=42).tolist()
    avg_img = np.zeros((224, 224, 3), dtype=np.float32)
    for p in cls_paths: avg_img += cv2.resize(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB), (224, 224))
    axes[i].imshow((avg_img / len(cls_paths)).astype(np.uint8)); axes[i].set_title(f'{cls}'); axes[i].axis('off')
plt.tight_layout(); plt.savefig(PLTDIR / '14_average_images.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 15: PIXEL INTENSITY DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for i, cls in enumerate(class_names):
    cls_paths = df_all[df_all['label'] == cls]['path'].sample(n=50, random_state=42).tolist()
    pixels = []
    for p in cls_paths:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None: pixels.extend(np.random.choice(img.flatten(), size=1000, replace=False))
    sns.kdeplot(pixels, fill=True, color=COLORS[i], label=cls, ax=ax, alpha=0.3, linewidth=2)
ax.set_title('Pixel Intensity Distribution by Class', fontsize=14, fontweight='bold'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(PLTDIR / '15_pixel_intensity.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 16: IMAGE DIMENSIONS
# ─────────────────────────────────────────────────────────────────────────────────
dim_data = []
for p in df_all['path'].sample(n=500, random_state=42):
    img = cv2.imread(p)
    if img is not None:
        h, w = img.shape[:2]; lbl = df_all.loc[df_all['path']==p, 'label'].values[0]
        dim_data.append({'Height': h, 'Width': w, 'Label': lbl})
dim_df = pd.DataFrame(dim_data)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Image Dimension Analysis', fontsize=14, fontweight='bold')
sns.boxplot(data=dim_df, x='Label', y='Height', palette=COLORS, ax=axes[0]); axes[0].set_title('Image Height')
sns.boxplot(data=dim_df, x='Label', y='Width', palette=COLORS, ax=axes[1]); axes[1].set_title('Image Width')
plt.tight_layout(); plt.savefig(PLTDIR / '16_image_dimensions.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#               PLOT 17: XAI GRAD-CAM + ATTENTION (5 SAMPLES)
# ─────────────────────────────────────────────────────────────────────────────────
print("\n🔥 Generating XAI Grad-CAM + Attention Overlays...")
class AdvancedXAI:
    def __init__(self, model):
        self.model = model; self.model.eval(); self.grads, self.activations = None, None
        last_conv = None
        for m in self.model.backbone.feature_extractor.modules():
            if isinstance(m, nn.Conv2d): last_conv = m
        if last_conv:
            last_conv.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
            last_conv.register_backward_hook(lambda m, gi, go: setattr(self, 'grads', go[0].detach()))

    def generate(self, img_path, target_class):
        rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB); orig_h, orig_w = rgb.shape[:2]
        x = get_tensor(rgb).float().requires_grad_(True)
        
        out = self.model(x); self.model.zero_grad(); oh = torch.zeros_like(out); oh[0, target_class] = 1
        out.backward(gradient=oh, retain_graph=True)
        
        if self.grads is not None and self.activations is not None:
            weights = self.grads.mean(dim=(2, 3), keepdim=True)
            cam = F.relu((weights * self.activations).sum(1)).squeeze().cpu().numpy()
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8) if cam.max() > cam.min() else np.zeros_like(cam)
        else: cam = np.zeros((self.model.spatial_h, self.model.spatial_w))

        with torch.no_grad(): _ = self.model(x)
        g_attn, _ = self.model.get_attentions()
        attn = np.zeros((self.model.spatial_h, self.model.spatial_w))
        if g_attn:
            a = g_attn[-1]
            attn = a[0, :, 0, 1:].mean(0).cpu().numpy() if a.dim() == 4 else a[0, 0, 1:].cpu().numpy()
            attn = attn.reshape(self.model.spatial_h, self.model.spatial_w)
            attn = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8) if attn.max() > attn.min() else np.zeros_like(attn)

        def overlay(hmap):
            if hmap.max() == 0: return rgb.copy()
            hr = cv2.resize(hmap, (orig_w, orig_h)); colored = cv2.applyColorMap((hr * 255).astype(np.uint8), cv2.COLORMAP_JET)
            return cv2.addWeighted(rgb, 0.6, colored, 0.4, 0)

        comb = 0.5 * cam + 0.5 * attn
        comb = (comb - comb.min()) / (comb.max() - comb.min() + 1e-8) if comb.max() > comb.min() else comb
        return rgb, overlay(cam), overlay(attn), overlay(comb)

xai = AdvancedXAI(model)
sample_indices = []
for cls_idx in range(7):
    mask = (all_p == cls_idx) & (all_l == cls_idx)
    if mask.any(): sample_indices.append(np.where(mask)[0][0])
sample_indices = sample_indices[:5] 

fig, axes = plt.subplots(len(sample_indices), 4, figsize=(16, 4 * len(sample_indices)))
fig.suptitle('Model Explainability: Grad-CAM vs Attention vs Combined', fontsize=18, fontweight='bold', y=1.01)
if len(sample_indices) == 1: axes = axes.reshape(1, -1)
for row_idx, idx in enumerate(sample_indices):
    rgb, cam_rgb, attn_rgb, comb_rgb = xai.generate(test_s[idx][0], all_l[idx])
    for col_idx, img in enumerate([rgb, cam_rgb, attn_rgb, comb_rgb]):
        axes[row_idx, col_idx].imshow(img)
        if col_idx == 0: axes[row_idx, col_idx].set_title(f"True: {class_names[all_l[idx]]}\nPred: {class_names[all_p[idx]]} ({max_probs[idx]:.1%})", fontweight='bold')
        else: axes[row_idx, col_idx].set_title(['Original', 'Grad-CAM', 'Attention', 'Fused'][col_idx], fontweight='bold')
        axes[row_idx, col_idx].axis('off')
plt.tight_layout(); plt.savefig(PLTDIR / '17_xai_5_samples.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#               PLOT 18: DEPLOYMENT THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────────
thresholds = np.arange(0.5, 1.0, 0.05); metrics_data = []
for thresh in thresholds:
    mask = max_probs >= thresh
    if mask.sum() == 0: continue
    p_thresh, l_thresh = all_p[mask], all_l[mask]
    metrics_data.append({'Threshold': thresh, 'Accuracy': accuracy_score(l_thresh, p_thresh), 'F1-Score': f1_score(l_thresh, p_thresh, average='weighted', zero_division=0), 'Coverage': mask.sum() / len(all_p) * 100})
df_thresh = pd.DataFrame(metrics_data)
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.set_xlabel('Minimum Confidence Threshold')
ax1.set_ylabel('Metric Score', color='#3498db')
ax1.plot(df_thresh['Threshold'], df_thresh['Accuracy'], color='#3498db', marker='o', lw=2, label='Accuracy')
ax1.plot(df_thresh['Threshold'], df_thresh['F1-Score'], color='#2ecc71', marker='s', lw=2, label='F1-Score')
ax1.tick_params(axis='y', labelcolor='#3498db'); ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx(); ax2.set_ylabel('Dataset Coverage (%)', color='#e74c3c')
ax2.plot(df_thresh['Threshold'], df_thresh['Coverage'], color='#e74c3c', marker='^', lw=2, linestyle='--', label='Coverage')
ax2.tick_params(axis='y', labelcolor='#e74c3c')
fig.legend(loc='lower left', bbox_to_anchor=(0.1, -0.1), ncol=3)
plt.tight_layout(); plt.savefig(PLTDIR / '18_deployment_thresholds.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#               PLOT 19: CLASS-SPECIFIC THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
cls_threshs = np.arange(0.4, 1.0, 0.05)
for cls_idx, cls in enumerate(class_names):
    cls_f1s = []
    for t in cls_threshs:
        mask = (all_p == cls_idx) & (max_probs >= t)
        cls_f1s.append(f1_score(all_l[mask] == cls_idx, all_p[mask] == cls_idx, zero_division=0) if mask.sum() > 0 else 0)
    opt_idx = np.argmax(cls_f1s)
    ax.plot(cls_threshs, cls_f1s, color=COLORS[cls_idx], marker='o', lw=2, label=f'{cls} (Opt @ {cls_threshs[opt_idx]:.2f})')
ax.set_title('Class-Specific F1 vs. Confidence Threshold', fontsize=14, fontweight='bold')
ax.set_xlabel('Confidence Threshold'); ax.set_ylabel('F1-Score'); ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(PLTDIR / '19_class_specific_thresholds.png', dpi=300, bbox_inches='tight'); plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#               PLOT 20: DEEP FEATURE MAPS HIERARCHY
# ─────────────────────────────────────────────────────────────────────────────────
print("\n🧠 Extracting Hierarchical Feature Maps...")
mask_idx = np.where((all_l == 4) & (all_p == 4))[0][0] # Melanoma example
img = cv2.cvtColor(cv2.imread(test_s[mask_idx][0]), cv2.COLOR_BGR2RGB)
x_feat = get_tensor(img)
feature_maps = {}
def get_hook(name):
    def hook(module, input, output): feature_maps[name] = output.detach().cpu()
    return hook
model.backbone.feature_extractor[1].register_forward_hook(get_hook('Early_Layer'))
model.backbone.feature_extractor[4].register_forward_hook(get_hook('Mid_Layer'))
model.backbone.feature_extractor[7].register_forward_hook(get_hook('Late_Layer'))
with torch.no_grad(): _ = model(x_feat)

fig, axes = plt.subplots(3, 8, figsize=(20, 7))
fig.suptitle('Hierarchical Feature Extraction (Melanoma Example)', fontsize=16, fontweight='bold')
for row_idx, (layer, title) in enumerate(zip(['Early_Layer', 'Mid_Layer', 'Late_Layer'], ['Early Features (Edges)', 'Mid Features (Anatomy)', 'Late Features (Pathology)'])):
    if layer in feature_maps:
        fm = feature_maps[layer].squeeze(); n_channels = fm.shape[0]
        channel_idxs = np.linspace(0, n_channels - 1, 8, dtype=int)
        for col_idx, c_idx in enumerate(channel_idxs):
            if c_idx < n_channels:
                ch_img = fm[c_idx].numpy(); ch_img = (ch_img - ch_img.min()) / (ch_img.max() - ch_img.min() + 1e-8)
                axes[row_idx, col_idx].imshow(ch_img, cmap='magma'); axes[row_idx, col_idx].set_title(f'Ch {c_idx}', fontsize=9); axes[row_idx, col_idx].axis('off')
    axes[row_idx, -1].axis('off')
    axes[row_idx, -1].text(1.1, 0.5, title, transform=axes[row_idx, -1].transAxes, fontsize=12, fontweight='bold', va='center', rotation=270)
plt.tight_layout(); plt.savefig(PLTDIR / '20_feature_maps_hierarchy.png', dpi=300, bbox_inches='tight'); plt.show()


# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  EXACT FINAL EVALUATION METRICS BLOCK                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  EXACT FINAL EVALUATION METRICS BLOCK (FIXED)                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

test_acc = accuracy_score(all_l, all_p)
test_f1 = f1_score(all_l, all_p, average='weighted')
test_auc = roc_auc_score(all_l, all_pr, multi_class='ovr', average='weighted')
test_ap = average_precision_score(labels_bin, all_pr, average='weighted')

mcc = matthews_corrcoef(all_l, all_p)
kappa = cohen_kappa_score(all_l, all_p)
ll = log_loss(all_l, all_pr)

# FIXED: Compute Brier score per class and average
brier_per_class = []
for i in range(7):
    brier_per_class.append(brier_score_loss(labels_bin[:, i], all_pr[:, i]))
brier = np.mean(brier_per_class)

# Multi-class Brier score (alternative)
brier_multi = np.mean(np.sum((labels_bin - all_pr)**2, axis=1))

# Calculate Macro metrics over 7 classes
sens_list = [cm[i,i]/(cm[i,:].sum()+1e-8) for i in range(7)]
spec_list = [(cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]) / (cm.sum() - cm[i,:].sum() + 1e-8) for i in range(7)]
prec_list = [cm[i,i]/(cm[:,i].sum()+1e-8) for i in range(7)]
npv_list = [(cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]) / (cm.sum() - cm[:,i].sum() + 1e-8) for i in range(7)]

print("\n============================================================")
print("📊 COMPREHENSIVE EVALUATION METRICS")
print("============================================================")

print(f"\n🎯 PRIMARY METRICS:")
print(f"   Accuracy:  {test_acc:.4f}")
print(f"   F1-Score:  {test_f1:.4f}")
print(f"   AUC-ROC:   {test_auc:.4f}")
print(f"   AP Score:  {test_ap:.4f}")

print(f"\n📈 ADDITIONAL METRICS:")
print(f"   MCC:       {mcc:.4f}")
print(f"   Cohen's Kappa: {kappa:.4f}")
print(f"   Log Loss:  {ll:.4f}")
print(f"   Brier Score (per-class avg): {brier:.4f}")
print(f"   Brier Score (multi-class):   {brier_multi:.4f}")

print(f"\n🔬 PER-CLASS METRICS (HAM10000 7-Classes Macro Avg):")
print(f"   Sensitivity (Recall): {np.mean(sens_list):.4f}")
print(f"   Specificity:          {np.mean(spec_list):.4f}")
print(f"   Precision:            {np.mean(prec_list):.4f}")
print(f"   NPV:                  {np.mean(npv_list):.4f}")
print(f"   FPR:                  {1 - np.mean(spec_list):.4f}")
print(f"   FNR:                  {1 - np.mean(sens_list):.4f}")

print(f"\n📊 CONFUSION MATRIX:")
header = f"{'':>18s}" + "".join([f"{n:>10s}" for n in class_names])
print(header)
for i, cls in enumerate(class_names):
    row = f"{cls:>18s}" + "".join([f"{cm[i,j]:>10d}" for j in range(7)])
    print(row)

print(f"\n📋 CLASSIFICATION REPORT:")
print(classification_report(all_l, all_p, target_names=class_names))

print("="*60)
print(f"✅ ALL 20 PLOTS GENERATED SUCCESSFULLY FOR HAM10000!")
print(f"📁 Location: {PLTDIR}")
print("="*60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Setup, Imports, and Global Configuration                             ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
!pip install -q timm einops albumentations opencv-python-headless

import os, time, random, copy, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
import cv2
from tqdm.auto import tqdm
from scipy.ndimage import zoom

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2
from einops import rearrange

from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, roc_curve, 
                             confusion_matrix, classification_report, precision_recall_curve,
                             average_precision_score, matthews_corrcoef, cohen_kappa_score,
                             log_loss, brier_score_loss)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTDIR = Path('/kaggle/working')
CKDIR = OUTDIR / 'checkpoints'
PLTDIR = OUTDIR / 'plots'
for d in [CKDIR, PLTDIR]: d.mkdir(exist_ok=True)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

# HAM10000 - 7 Classes
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
class_full_names = ['Actinic Keratoses', 'Basal Cell Carcinoma', 'Benign Keratosis', 
                    'Dermatofibroma', 'Melanoma', 'Melanocytic Nevi', 'Vascular Lesions']
NUM_CLASSES = 7
COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f1c40f', '#1abc9c', '#e67e22']

print(f"✅ Setup complete on {DEVICE} | Classes: {class_names}")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Hyperparameters                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
HPARAMS = {
    'img_size': 224, 'batch_size': 32, 'num_epochs': 40,
    'lr_backbone': 1e-5, 'lr_head': 1e-4, 'weight_decay': 1e-4,
    'freeze_epochs': 10, 'patience': 15,
    'd_model': 384, 'nhead': 6, 'n_layers': 4, 'dim_ffn': 1536,
    'label_smoothing': 0.05, 'mixup_alpha': 0.2, 'cutmix_alpha': 1.0, 'mixup_prob': 0.3,
}

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Data Augmentations                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
def get_transforms(split, img_size=224):
    norm = [A.Normalize(mean=MEAN, std=STD), ToTensorV2()]
    if split != 'train':
        return A.Compose([A.Resize(height=img_size, width=img_size)] + norm)
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
    ] + norm)

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Dataset Class                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
class SkinCancerDataset(Dataset):
    def __init__(self, samples, class_names, transform=None):
        self.samples = samples; self.transform = transform
        self.class_to_idx = {n: i for i, n in enumerate(class_names)}
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        p, l = self.samples[idx]; l = self.class_to_idx[l]
        img = cv2.imread(p)
        if img is None: 
            img = np.array(Image.open(p).convert('RGB'))
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform: img = self.transform(image=img)['image']
        return img.float(), l
    def class_weights(self):
        labels = [self.class_to_idx[l] for _, l in self.samples]
        counts = torch.bincount(torch.tensor(labels), minlength=NUM_CLASSES).float()
        beta = 0.9999
        effective_num = 1.0 - torch.pow(beta, counts)
        return ((1.0 - beta) / (effective_num + 1e-8)) / NUM_CLASSES * NUM_CLASSES

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Load HAM10000 Dataset                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

def load_ham10000_dataset():
    print("📂 Loading HAM10000 Skin Cancer Dataset...")
    
    # Find the dataset path
    base = None
    for p in Path('/kaggle/input/datasets/nikitachaulagain/dataasets/skin cancer/skin cancer').rglob('HAM10000_metadata.csv'):
        base = p.parent
        break
    
    if base is None:
        # Try alternative path
        for p in Path('/kaggle/input').rglob('*metadata.csv'):
            if 'ham' in str(p).lower():
                base = p.parent
                break
    
    if base is None:
        raise ValueError("❌ HAM10000 dataset not found!")
    
    print(f"✅ Found dataset at: {base}")
    
    # Load metadata
    df_meta = pd.read_csv(base / 'HAM10000_metadata.csv')
    
    # Find image directories
    img_dirs = []
    for d in base.iterdir():
        if d.is_dir() and 'images' in d.name.lower():
            img_dirs.append(d)
    
    if not img_dirs:
        # Try root directory
        img_dirs = [base]
    
    def get_path(img_id):
        for d in img_dirs:
            for ext in ['.jpg', '.png', '.jpeg']:
                p = d / f"{img_id}{ext}"
                if p.exists():
                    return str(p)
        return None
    
    df_meta['path'] = df_meta['image_id'].apply(get_path)
    df_meta = df_meta.dropna(subset=['path'])
    
    # Create samples
    samples = list(zip(df_meta['path'], df_meta['dx']))
    
    # Stratified split
    labels = [class_names.index(l) for _, l in samples]
    train_s, temp_s = train_test_split(samples, test_size=0.3, stratify=labels, random_state=42)
    temp_labels = [class_names.index(l) for _, l in temp_s]
    val_s, test_s = train_test_split(temp_s, test_size=0.5, stratify=temp_labels, random_state=42)
    
    print(f"\n📊 Dataset Statistics:")
    print(f"   Total Images: {len(samples)}")
    print(f"   Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}")
    
    print("\n   Class Distribution:")
    for cls in class_names:
        count = len(df_meta[df_meta['dx'] == cls])
        print(f"   - {cls:10s}: {count:5d} images ({count/len(df_meta)*100:.1f}%)")
    
    return train_s, val_s, test_s

# Execute loading
train_s, val_s, test_s = load_ham10000_dataset()

if not train_s and not val_s and not test_s:
    raise ValueError("❌ No images loaded!")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Create Dataloaders                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
train_ds = SkinCancerDataset(train_s, class_names, get_transforms('train'))
val_ds = SkinCancerDataset(val_s, class_names, get_transforms('val'))
test_ds = SkinCancerDataset(test_s, class_names, get_transforms('test'))

class_weights = train_ds.class_weights()
sample_weights = [class_weights[class_names.index(l)] for _, l in train_s]

loaders = {
    'train': DataLoader(train_ds, batch_size=HPARAMS['batch_size'], 
                       sampler=WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True),
                       num_workers=2, pin_memory=True),
    'val': DataLoader(val_ds, batch_size=HPARAMS['batch_size'], shuffle=False, num_workers=2, pin_memory=True),
    'test': DataLoader(test_ds, batch_size=HPARAMS['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
}

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Model Architecture (ConvNeXt-Base + Hybrid ViT)                      ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0): super().__init__(); self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training: return x
        return x * (torch.rand((x.shape[0], 1, 1), dtype=torch.float32, device=x.device) * float(1.0 - self.drop_prob))

class WindowAttention(nn.Module):
    def __init__(self, d_model, nhead, window_size=7, dropout=0.1):
        super().__init__()
        self.nhead, self.window_size, self.head_dim = nhead, window_size, d_model // nhead
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3); self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.proj_drop, self.attn_weights = nn.Dropout(dropout), nn.Dropout(dropout), None
    def forward(self, x, H, W):
        B, N, C = x.shape; x = x.reshape(B, H, W, C)
        pad_h, pad_w = (self.window_size - H % self.window_size) % self.window_size, (self.window_size - W % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0: x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        Hp, Wp = x.shape[1], x.shape[2]
        x = x.reshape(B, Hp//self.window_size, self.window_size, Wp//self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(-1, self.window_size**2, C)
        qkv = self.qkv(x).reshape(-1, self.window_size**2, 3, self.nhead, self.head_dim).permute(2,0,3,1,4)
        attn = (qkv[0] @ qkv[1].transpose(-2,-1)) * self.scale; attn = attn.softmax(dim=-1); self.attn_weights = attn.detach()
        x = self.proj_drop(self.proj((self.attn_drop(attn) @ qkv[2]).transpose(1,2).reshape(-1, self.window_size**2, C)))
        x = x.reshape(B, Hp//self.window_size, Wp//self.window_size, self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(B, Hp, Wp, C)
        return x[:, :H, :W, :].reshape(B, -1, C) if pad_h > 0 or pad_w > 0 else x.reshape(B, -1, C)

class TransformerBlockWithWindow(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim, window_size=7, attn_drop=0.1, ffn_drop=0.1, drop_path=0.0):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.global_attn = nn.MultiheadAttention(d_model, nhead, dropout=attn_drop, batch_first=True)
        self.window_attn = WindowAttention(d_model, nhead, window_size, attn_drop)
        self.gate = nn.Sequential(nn.Linear(d_model*2, d_model), nn.Sigmoid())
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(ffn_drop), nn.Linear(ffn_dim, d_model), nn.Dropout(ffn_drop))
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.attn_weights, self.window_attn_weights = None, None
    def forward(self, x, H, W):
        x_norm = self.norm1(x)
        global_out, global_attn = self.global_attn(x_norm, x_norm, x_norm, need_weights=True)
        self.attn_weights = global_attn.detach()
        window_out = self.window_attn(x_norm[:, 1:, :], H, W); self.window_attn_weights = self.window_attn.attn_weights
        gate = self.gate(torch.cat([global_out[:, 1:, :], window_out], dim=-1))
        combined = torch.cat([global_out[:, :1, :], gate * global_out[:, 1:, :] + (1-gate) * window_out], dim=1)
        x = x + self.drop_path(combined)
        return x + self.drop_path(self.ffn(self.norm2(x)))

class ImprovedBackbone(nn.Module):
    def __init__(self): 
        super().__init__(); 
        self.backbone = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
        self.feature_extractor = self.backbone.features
    def forward(self, x): return self.feature_extractor(x)

class EnhancedHViT(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536):
        super().__init__(); self.d_model = d_model; self.img_size = img_size; self.backbone = ImprovedBackbone()
        with torch.no_grad(): _, c, h, w = self.backbone(torch.zeros(1, 3, img_size, img_size)).shape
        self.spatial_h, self.spatial_w = h, w
        self.proj = nn.Sequential(
            nn.Conv2d(c, d_model, 1, bias=False), 
            nn.BatchNorm2d(d_model), 
            nn.GELU(), 
            nn.Conv2d(d_model, d_model, 3, padding=1, bias=False), 
            nn.BatchNorm2d(d_model), 
            nn.GELU()
        )
        self.norm_proj = nn.LayerNorm(d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, h*w + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        dpr = [x.item() for x in torch.linspace(0, 0.15, n_layers)]
        self.encoder = nn.ModuleList([
            TransformerBlockWithWindow(d_model, nhead, dim_ffn, min(7, h, w), drop_path=dpr[i]) 
            for i in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), 
            nn.Linear(d_model, d_model//2), 
            nn.GELU(), 
            nn.Dropout(0.3), 
            nn.Linear(d_model//2, d_model//4), 
            nn.GELU(), 
            nn.Dropout(0.15), 
            nn.Linear(d_model//4, num_classes)
        )
    def forward(self, x):
        B = x.size(0)
        tokens = rearrange(self.proj(self.backbone(x)), 'b d h w -> b (h w) d')
        tokens = self.norm_proj(tokens)
        tokens = torch.cat([self.cls_token.expand(B,-1,-1), tokens], dim=1) + self.pos_embed
        for block in self.encoder:
            tokens = block(tokens, self.spatial_h, self.spatial_w)
        return self.head(self.norm(tokens)[:, 0])
    def get_attentions(self):
        g, w = [], []
        for b in self.encoder:
            if b.attn_weights is not None: g.append(b.attn_weights)
            if b.window_attn_weights is not None: w.append(b.window_attn_weights)
        return g, w
    def unfreeze_backbone_partial(self, n=3):
        for p in self.backbone.feature_extractor.parameters(): p.requires_grad_(False)
        for child in list(self.backbone.feature_extractor.children())[-n:]:
            for p in child.parameters(): p.requires_grad_(True)

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Training Functions                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
class SmoothCELoss(nn.Module):
    def __init__(self, label_smoothing=0.05): 
        super().__init__(); 
        self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    def forward(self, inputs, targets): 
        return self.criterion(inputs, targets)

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    W, H = x.size(2), x.size(3)
    r = np.sqrt(1.0 - lam)
    cw, ch = int(W*r), int(H*r)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = np.clip(cx - cw//2, 0, W), np.clip(cy - ch//2, 0, H)
    x2, y2 = np.clip(cx + cw//2, 0, W), np.clip(cy + ch//2, 0, H)
    mixed = x.clone()
    mixed[:, :, x1:x2, y1:y2] = x[idx, :, x1:x2, y1:y2]
    return mixed, y, y[idx], 1 - (x2-x1)*(y2-y1)/(W*H)

def train_one_epoch(model, loader, optimizer, scaler, criterion, epoch):
    model.train()
    total_loss, preds, labs = 0, [], []
    for images, labels in tqdm(loader, desc=f'Epoch {epoch+1}', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        mixed = False
        if np.random.random() < HPARAMS['mixup_prob']:
            if np.random.random() < 0.5:
                images, la, lb, lam = mixup_data(images, labels, HPARAMS['mixup_alpha'])
                mixed = True
            else:
                images, la, lb, lam = cutmix_data(images, labels, HPARAMS['cutmix_alpha'])
                mixed = True
        with autocast():
            out = model(images)
            loss = lam * criterion(out, la) + (1-lam) * criterion(out, lb) if mixed else criterion(out, labels)
            if not mixed:
                preds.extend(out.argmax(1).cpu().numpy())
                labs.extend(labels.cpu().numpy())
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss/len(loader), accuracy_score(labs, preds) if labs else 0, f1_score(labs, preds, average='weighted', zero_division=0) if labs else 0

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, preds, labs, probs = 0, [], [], []
    for images, labels in tqdm(loader, desc='Validating', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with autocast():
            out = model(images)
            loss = criterion(out, labels)
        total_loss += loss.item()
        p = F.softmax(out, dim=1)
        preds.extend(out.argmax(1).cpu().numpy())
        labs.extend(labels.cpu().numpy())
        probs.extend(p.cpu().numpy())
    return total_loss/len(loader), accuracy_score(labs, preds), f1_score(labs, preds, average='weighted', zero_division=0), preds, labs, probs

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Main Training Loop                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
print("\n🚀 Starting Training for HAM10000 Skin Cancer Classification...")
model = EnhancedHViT(num_classes=NUM_CLASSES).to(DEVICE)
criterion = SmoothCELoss(label_smoothing=HPARAMS['label_smoothing'])
optimizer = optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if 'backbone' in n], 'lr': HPARAMS['lr_backbone']},
    {'params': [p for n, p in model.named_parameters() if 'backbone' not in n], 'lr': HPARAMS['lr_head']}
], weight_decay=HPARAMS['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=HPARAMS['num_epochs']//3, T_mult=2, eta_min=1e-7)
scaler = GradScaler()

for p in model.backbone.feature_extractor.parameters():
    p.requires_grad_(False)  # Freeze initially

best_f1, patience_counter = 0, 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': [], 'lr': []}

for epoch in range(HPARAMS['num_epochs']):
    if epoch == HPARAMS['freeze_epochs']:
        print("\n🔄 Unfreezing backbone...")
        model.unfreeze_backbone_partial(3)
    
    train_loss, train_acc, train_f1 = train_one_epoch(model, loaders['train'], optimizer, scaler, criterion, epoch)
    val_loss, val_acc, val_f1, _, _, _ = validate(model, loaders['val'], criterion)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.3f} | Val Acc={val_acc:.3f}, Val F1={val_f1:.3f}")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save({
            'model': model.state_dict(), 
            'f1': val_f1, 
            'class_names': class_names
        }, CKDIR / 'best_model.pth')
        print(f"   ✅ Best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= HPARAMS['patience']:
            print("⏹️ Early stopping")
            break

print("\n✅ Training complete!")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Test Set Evaluation                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
print("\n📊 Evaluating on Test Set...")
model.load_state_dict(torch.load(CKDIR / 'best_model.pth', map_location=DEVICE, weights_only=False)['model'])
model.eval()

all_p, all_l, all_pr = [], [], []
for images, labels in tqdm(loaders['test'], desc="Testing"):
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    with torch.no_grad():
        with autocast():
            out = model(images)
            pr = F.softmax(out, dim=1)
    all_p.extend(out.argmax(1).cpu().numpy())
    all_l.extend(labels.cpu().numpy())
    all_pr.extend(pr.cpu().numpy())

all_p, all_l, all_pr = np.array(all_p), np.array(all_l), np.array(all_pr)
max_probs = all_pr.max(axis=1)
correct_mask = (all_p == all_l)
labels_bin = label_binarize(all_l, classes=list(range(NUM_CLASSES)))
cm = confusion_matrix(all_l, all_p)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

print(f"\n✅ Test Set Size: {len(all_p)} images")
print(f"   Accuracy: {accuracy_score(all_l, all_p):.4f}")
print(f"   Weighted F1: {f1_score(all_l, all_p, average='weighted'):.4f}")

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Interactive Prediction Interface                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
from PIL import Image as PILImage

print("📦 Loading HAM10000 Skin Cancer Classification Model...")
model.load_state_dict(torch.load(CKDIR / 'best_model.pth', map_location=DEVICE, weights_only=False)['model'])
model.eval()

def predict_single_image(image_array):
    if len(image_array.shape) == 2:
        image_array = np.stack([image_array]*3, axis=-1)
    img_resized = cv2.resize(image_array, (224, 224))
    t = torch.from_numpy(img_resized).float().permute(2, 0, 1) / 255.0
    t = ((t - torch.tensor(MEAN).view(3,1,1)) / torch.tensor(STD).view(3,1,1)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        with autocast():
            logits = model(t)
            probs = F.softmax(logits, dim=1)
            conf, pred_idx = probs.max(dim=1)
    return pred_idx.item(), conf.item(), probs.cpu().numpy()[0]

upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='📤 Upload Skin Image', button_style='info')
path_input = widgets.Text(value='', placeholder='/kaggle/input/.../skin_lesion.jpg', description='📁 Or path:', layout={'width': '600px'})
predict_button = widgets.Button(description='🔍 Predict', button_style='success', layout={'width': '200px'})
output_area = widgets.Output()

def on_predict_clicked(b):
    with output_area:
        clear_output(wait=True)
        image_rgb = None
        if upload_widget.value:
            content = list(upload_widget.value.values())[0]['content']
            image_rgb = np.array(PILImage.open(io.BytesIO(content)).convert('RGB'))
        elif path_input.value.strip():
            p = Path(path_input.value.strip())
            if p.exists():
                image_rgb = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
            else:
                print("❌ File not found")
                return
        else:
            print("⚠️ Please provide an image")
            return
            
        pred_idx, confidence, probs = predict_single_image(image_rgb)
        
        print(f"\n{'='*50}")
        print(f"   🏥 PREDICTION: {class_full_names[pred_idx]}")
        print(f"   📊 Class: {class_names[pred_idx]}")
        print(f"   🎯 Confidence: {confidence:.2%}")
        print(f"{'='*50}")
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(image_rgb)
        axes[0].set_title(f"{class_full_names[pred_idx]}\n{confidence:.2%}", 
                         fontsize=14, fontweight='bold', color=COLORS[pred_idx])
        axes[0].axis('off')
        
        colors_bar = [COLORS[i] if i == pred_idx else '#95a5a6' for i in range(NUM_CLASSES)]
        bars = axes[1].barh(class_full_names, probs * 100, color=colors_bar, edgecolor='black')
        axes[1].set_xlim(0, 105)
        axes[1].set_title('Class Probabilities', fontweight='bold')
        for bar, prob in zip(bars, probs):
            axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
                        f'{prob:.1f}%', va='center', fontweight='bold')
        axes[1].grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()

predict_button.on_click(on_predict_clicked)

print("╔══════════════════════════════════════════════════════════════╗")
print("║        🏥 SKIN CANCER CLASSIFICATION (HAM10000)             ║")
print("╚══════════════════════════════════════════════════════════════╝\n")
display(widgets.VBox([upload_widget, widgets.Label("── OR ──"), path_input, predict_button, output_area]))

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Standalone Heatmap Generator                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝
class XAIHeatmapGenerator:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.cnn_f, self.cnn_g = None, None
        last_conv = None
        for m in self.model.backbone.feature_extractor.modules():
            if isinstance(m, nn.Conv2d):
                last_conv = m
        if last_conv:
            last_conv.register_forward_hook(lambda m, i, o: setattr(self, 'cnn_f', o.detach()))
            last_conv.register_backward_hook(lambda m, gi, go: setattr(self, 'cnn_g', go[0].detach()))

    def get_cnn(self, x, target):
        out = self.model(x)
        self.model.zero_grad()
        oh = torch.zeros_like(out)
        oh[0, target] = 1
        out.backward(gradient=oh, retain_graph=True)
        if self.cnn_f is None:
            return None
        weights = self.cnn_g.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.cnn_f).sum(1)).squeeze().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8) if cam.max() > cam.min() else cam

    def get_attn(self, x):
        with torch.no_grad():
            self.model(x)
        g, _ = self.model.get_attentions()
        if not g:
            return None
        a = g[-1]
        a = a[0, :, 0, 1:].mean(0).cpu().numpy() if a.dim() == 4 else a[0, 0, 1:].cpu().numpy()
        a = a.reshape(self.model.spatial_h, self.model.spatial_w)
        return (a - a.min()) / (a.max() - a.min() + 1e-8) if a.max() > a.min() else a

    def draw(self, img_path, target, save_path):
        img = cv2.imread(str(img_path))
        if img is None:
            return
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        sz = rgb.shape[:2]
        t = get_tensor(rgb)
        c = self.get_cnn(t.clone(), target)
        a = self.get_attn(t.clone())
        comb = 0.5 * c + 0.5 * a if c is not None and a is not None else c
        if comb is not None and comb.max() > comb.min():
            comb = (comb - comb.min()) / (comb.max() - comb.min() + 1e-8)

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(rgb)
        axes[0].set_title(f"True: {class_full_names[target]}", fontweight='bold')
        axes[0].axis('off')
        
        for i, (hmap, title) in enumerate(zip([c, a, comb], ['Grad-CAM', 'Attention', 'Combined'])):
            if hmap is not None:
                hr = zoom(hmap, (sz[0]/hmap.shape[0], sz[1]/hmap.shape[1]), order=1)
                colored = (plt.cm.jet(hr)[:,:,:3] * 255).astype(np.uint8)
                axes[i+1].imshow(cv2.addWeighted(rgb, 0.6, colored, 0.4, 0))
            else:
                axes[i+1].imshow(rgb)
            axes[i+1].set_title(title, fontweight='bold')
            axes[i+1].axis('off')
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

def get_tensor(img):
    if len(img.shape) == 2:
        img = np.stack([img]*3, axis=-1)
    t = torch.from_numpy(cv2.resize(img, (224, 224))).float().permute(2,0,1)/255.0
    return ((t - torch.tensor(MEAN).view(3,1,1)) / torch.tensor(STD).view(3,1,1)).unsqueeze(0).to(DEVICE)

hm_gen = XAIHeatmapGenerator(model)
print("🔥 Generating Heatmaps for Skin Cancer Images...")
for cls_idx in range(NUM_CLASSES):
    cls_tests = [s for s in test_s if class_names.index(s[1]) == cls_idx]
    if cls_tests:
        print(f"\n📊 {class_full_names[cls_idx]}:")
        for p, _ in cls_tests[:2]:
            hm_gen.draw(p, cls_idx, PLTDIR / f'hm_{class_names[cls_idx]}_{Path(p).stem}.png')

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  ULTIMATE MEGA CELL: ALL 20 PLOTS + EXACT METRICS (HAM10000 7-Classes)         ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

import os, time, random, warnings, re
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, cv2
from tqdm.auto import tqdm
from scipy.ndimage import zoom
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from sklearn.metrics import (accuracy_score, f1_score, roc_curve, auc, confusion_matrix, 
                             classification_report, precision_recall_curve, average_precision_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss, brier_score_loss)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from torch.cuda.amp import autocast
from einops import rearrange
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTDIR = Path('/kaggle/working')
PLTDIR = OUTDIR / 'plots/ham10000_final'
PLTDIR.mkdir(exist_ok=True, parents=True)
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
class_full_names = ['Actinic Keratoses', 'Basal Cell Carcinoma', 'Benign Keratosis', 
                    'Dermatofibroma', 'Melanoma', 'Melanocytic Nevi', 'Vascular Lesions']
COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f1c40f', '#1abc9c', '#e67e22']

# ─── 1. RE-DEFINE MODEL ARCHITECTURE (Required for loading weights) ──────────────
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0): super().__init__(); self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training: return x
        return x * (torch.rand((x.shape[0], 1, 1), dtype=torch.float32, device=x.device) * float(1.0 - self.drop_prob))

class WindowAttention(nn.Module):
    def __init__(self, d_model, nhead, window_size=7, dropout=0.1):
        super().__init__()
        self.nhead, self.window_size, self.head_dim = nhead, window_size, d_model // nhead
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3); self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.proj_drop, self.attn_weights = nn.Dropout(dropout), nn.Dropout(dropout), None
    def forward(self, x, H, W):
        B, N, C = x.shape; x = x.reshape(B, H, W, C)
        pad_h, pad_w = (self.window_size - H % self.window_size) % self.window_size, (self.window_size - W % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0: x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        Hp, Wp = x.shape[1], x.shape[2]
        x = x.reshape(B, Hp//self.window_size, self.window_size, Wp//self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(-1, self.window_size**2, C)
        qkv = self.qkv(x).reshape(-1, self.window_size**2, 3, self.nhead, self.head_dim).permute(2,0,3,1,4)
        attn = (qkv[0] @ qkv[1].transpose(-2,-1)) * self.scale; attn = attn.softmax(dim=-1); self.attn_weights = attn.detach()
        x = self.proj_drop(self.proj((self.attn_drop(attn) @ qkv[2]).transpose(1,2).reshape(-1, self.window_size**2, C)))
        x = x.reshape(B, Hp//self.window_size, Wp//self.window_size, self.window_size, self.window_size, C).permute(0,1,3,2,4,5).reshape(B, Hp, Wp, C)
        return x[:, :H, :W, :].reshape(B, -1, C) if pad_h > 0 or pad_w > 0 else x.reshape(B, -1, C)

class TransformerBlockWithWindow(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim, window_size=7, attn_drop=0.1, ffn_drop=0.1, drop_path=0.0):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.global_attn = nn.MultiheadAttention(d_model, nhead, dropout=attn_drop, batch_first=True)
        self.window_attn = WindowAttention(d_model, nhead, window_size, attn_drop)
        self.gate = nn.Sequential(nn.Linear(d_model*2, d_model), nn.Sigmoid())
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(ffn_drop), nn.Linear(ffn_dim, d_model), nn.Dropout(ffn_drop))
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.attn_weights, self.window_attn_weights = None, None
    def forward(self, x, H, W):
        x_norm = self.norm1(x)
        global_out, global_attn = self.global_attn(x_norm, x_norm, x_norm, need_weights=True)
        self.attn_weights = global_attn.detach()
        window_out = self.window_attn(x_norm[:, 1:, :], H, W); self.window_attn_weights = self.window_attn.attn_weights
        gate = self.gate(torch.cat([global_out[:, 1:, :], window_out], dim=-1))
        combined = torch.cat([global_out[:, :1, :], gate * global_out[:, 1:, :] + (1-gate) * window_out], dim=1)
        x = x + self.drop_path(combined)
        return x + self.drop_path(self.ffn(self.norm2(x)))

class ImprovedBackbone(nn.Module):
    def __init__(self): super().__init__(); self.backbone = models.convnext_base(weights=None); self.feature_extractor = self.backbone.features
    def forward(self, x): return self.feature_extractor(x)

class EnhancedHViT(nn.Module):
    def __init__(self, num_classes=7, img_size=224, d_model=384, nhead=6, n_layers=4, dim_ffn=1536):
        super().__init__(); self.d_model = d_model; self.img_size = img_size; self.backbone = ImprovedBackbone()
        with torch.no_grad(): _, c, h, w = self.backbone(torch.zeros(1, 3, img_size, img_size)).shape
        self.spatial_h, self.spatial_w = h, w
        self.proj = nn.Sequential(
            nn.Conv2d(c, d_model, 1, bias=False), nn.BatchNorm2d(d_model), nn.GELU(),
            nn.Conv2d(d_model, d_model, 3, padding=1, bias=False), nn.BatchNorm2d(d_model), nn.GELU()
        )
        self.norm_proj = nn.LayerNorm(d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, h*w + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02); nn.init.trunc_normal_(self.pos_embed, std=0.02)
        dpr = [x.item() for x in torch.linspace(0, 0.15, n_layers)]
        self.encoder = nn.ModuleList([TransformerBlockWithWindow(d_model, nhead, dim_ffn, min(7, h, w), drop_path=dpr[i]) for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model//2), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(d_model//2, d_model//4), nn.GELU(), nn.Dropout(0.15), nn.Linear(d_model//4, num_classes)
        )
    def forward(self, x):
        B = x.size(0)
        tokens = rearrange(self.proj(self.backbone(x)), 'b d h w -> b (h w) d')
        tokens = self.norm_proj(tokens)
        tokens = torch.cat([self.cls_token.expand(B,-1,-1), tokens], dim=1) + self.pos_embed
        for block in self.encoder:
            tokens = block(tokens, self.spatial_h, self.spatial_w)
        return self.head(self.norm(tokens)[:, 0])
    def get_attentions(self):
        g, w = [], []
        for b in self.encoder:
            if b.attn_weights is not None: g.append(b.attn_weights)
            if b.window_attn_weights is not None: w.append(b.window_attn_weights)
        return g, w

def get_tensor(img):
    if len(img.shape) == 2: img = np.stack([img]*3, axis=-1)
    t = torch.from_numpy(cv2.resize(img, (224, 224))).float().permute(2,0,1)/255.0
    return ((t - torch.tensor(MEAN).view(3,1,1)) / torch.tensor(STD).view(3,1,1)).unsqueeze(0).to(DEVICE)

# ─── 2. LOAD MODEL & TEST DATA ──────────────────────────────────────────────────
print("📦 Loading HAM10000 Model & Test Data...")
model = EnhancedHViT(num_classes=7).to(DEVICE)
model.load_state_dict(torch.load(CKDIR / 'best_model.pth', map_location=DEVICE, weights_only=False)['model'])
model.eval()

# ─── 3. RUN INFERENCE ───────────────────────────────────────────────────────────
print("🧠 Running Inference on Test Set...")
all_p, all_l, all_pr = [], [], []
for images, labels in tqdm(loaders['test'], desc="Predicting"):
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    with torch.no_grad():
        with autocast():
            out = model(images)
            pr = F.softmax(out, dim=1)
    all_p.extend(out.argmax(1).cpu().numpy())
    all_l.extend(labels.cpu().numpy())
    all_pr.extend(pr.cpu().numpy())

all_p, all_l, all_pr = np.array(all_p), np.array(all_l), np.array(all_pr)
max_probs = all_pr.max(axis=1)
correct_mask = (all_p == all_l)
labels_bin = label_binarize(all_l, classes=list(range(7)))
cm = confusion_matrix(all_l, all_p)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 1: TRAINING HISTORY
# ─────────────────────────────────────────────────────────────────────────────────
try:
    if len(history.get('train_loss', [])) > 0:
        fig, ax = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('HAM10000 Model Training Dynamics', fontsize=16, fontweight='bold', y=1.02)
        ax[0,0].plot(history['train_loss'], label='Train', color='#e74c3c', lw=2)
        ax[0,0].plot(history['val_loss'], label='Val', color='#3498db', lw=2)
        ax[0,0].set_title('Loss'); ax[0,0].legend(); ax[0,0].grid(True, alpha=0.3)
        ax[0,1].plot(history['train_acc'], label='Train', color='#e74c3c', lw=2)
        ax[0,1].plot(history['val_acc'], label='Val', color='#3498db', lw=2)
        ax[0,1].set_title('Accuracy'); ax[0,1].set_ylim(0,1.05); ax[0,1].legend(); ax[0,1].grid(True, alpha=0.3)
        ax[1,0].plot(history['val_f1'], label='Val F1', color='#2ecc71', lw=2)
        ax[1,0].set_title('F1 Score'); ax[1,0].set_ylim(0,1.05); ax[1,0].legend(); ax[1,0].grid(True, alpha=0.3)
        ax[1,1].plot(history['lr'], color='#9b59b6', lw=2)
        ax[1,1].set_title('Learning Rate'); ax[1,1].set_yscale('log'); ax[1,1].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(PLTDIR / '1_training_history.png', dpi=300, bbox_inches='tight')
        plt.show()
except Exception as e:
    print(f"Plot 1 error: {e}")
    pass

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 2: CONFUSION MATRICES
# ─────────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Skin Cancer Confusion Matrix (HAM10000)', fontsize=16, fontweight='bold')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0], cbar=False)
axes[0].set_title('Absolute Counts'); axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=axes[1], cbar=False)
axes[1].set_title('Normalized (Recall per Class)'); axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(PLTDIR / '2_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 3: ROC CURVES
# ─────────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
for i in range(7):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], all_pr[:, i])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=COLORS[i], lw=2, label=f'{class_full_names[i]} (AUC = {roc_auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k:', lw=1, alpha=0.5)
ax.set_title('ROC Curves - Skin Lesion Classification', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLTDIR / '3_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 4: PRECISION-RECALL CURVES
# ─────────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
for i in range(7):
    prec, rec, _ = precision_recall_curve(labels_bin[:, i], all_pr[:, i])
    ap = average_precision_score(labels_bin[:, i], all_pr[:, i])
    ax.plot(rec, prec, color=COLORS[i], lw=2, label=f'{class_names[i]} (AP = {ap:.3f})')
ax.set_title('Precision-Recall Curves - HAM10000', fontsize=14, fontweight='bold')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.3); ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
plt.tight_layout()
plt.savefig(PLTDIR / '4_precision_recall.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 5: PER-CLASS METRICS BARS
# ─────────────────────────────────────────────────────────────────────────────────
report = classification_report(all_l, all_p, target_names=class_names, output_dict=True)
metrics_df = pd.DataFrame(report).T.iloc[:-3, :3]
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(7); width = 0.25
bars1 = ax.bar(x - width, metrics_df['precision'], width, label='Precision', color='#3498db', edgecolor='black')
bars2 = ax.bar(x, metrics_df['recall'], width, label='Recall', color='#2ecc71', edgecolor='black')
bars3 = ax.bar(x + width, metrics_df['f1-score'], width, label='F1-Score', color='#e74c3c', edgecolor='black')
ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(class_names, rotation=0)
ax.legend(); ax.set_ylim(0,1.15); ax.grid(axis='y', alpha=0.3)
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(PLTDIR / '5_per_class_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 6: CONFIDENCE DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(max_probs[correct_mask], bins=20, alpha=0.7, label='Correct Predictions', color='#2ecc71', edgecolor='black')
ax.hist(max_probs[~correct_mask], bins=20, alpha=0.7, label='Incorrect Predictions', color='#e74c3c', edgecolor='black')
ax.set_title('Model Confidence Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Maximum Class Probability'); ax.set_ylabel('Count')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLTDIR / '6_confidence_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 7: CALIBRATION CURVE
# ─────────────────────────────────────────────────────────────────────────────────
def calibration_curve(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    accs, confs = [], []
    for i in range(n_bins):
        mask = (y_prob > bins[i]) & (y_prob <= bins[i+1])
        if i == n_bins - 1:
            mask = (y_prob >= bins[i]) & (y_prob <= bins[i+1])
        if mask.sum() > 0:
            accs.append(y_true[mask].mean())
            confs.append(y_prob[mask].mean())
    return np.array(confs), np.array(accs)

fig, ax = plt.subplots(figsize=(8, 8))
for i, cls in enumerate(class_names):
    conf, acc = calibration_curve((all_l == i).astype(int), all_pr[:, i])
    if len(conf) > 0:
        ax.plot(conf, acc, marker='o', color=COLORS[i], lw=2, label=cls, markersize=5)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfectly Calibrated')
ax.set_title('Reliability Diagram', fontsize=14, fontweight='bold')
ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Fraction of Positives')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_xlim([0,1]); ax.set_ylim([0,1])
plt.tight_layout()
plt.savefig(PLTDIR / '7_calibration_curve.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 8: SAMPLE PREDICTIONS
# ─────────────────────────────────────────────────────────────────────────────────
correct_idxs, incorrect_idxs = np.where(correct_mask)[0], np.where(~correct_mask)[0]
np.random.seed(42)
samp_corr = np.random.choice(correct_idxs, min(8, len(correct_idxs)), replace=False)
samp_incorr = np.random.choice(incorrect_idxs, min(8, len(incorrect_idxs)), replace=False)

fig, axes = plt.subplots(2, 8, figsize=(24, 6))
for i, idx in enumerate(samp_corr):
    img_path = test_s[idx][0]
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"T:{class_names[all_l[idx]]}\nP:{class_names[all_p[idx]]}\n{max_probs[idx]:.1%}",
                         fontsize=8, color='green')
    axes[0, i].axis('off')

for i, idx in enumerate(samp_incorr):
    img_path = test_s[idx][0]
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"T:{class_names[all_l[idx]]}\nP:{class_names[all_p[idx]]}\n{max_probs[idx]:.1%}",
                         fontsize=8, color='red')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('CORRECT', fontsize=12, fontweight='bold', color='green', rotation=0, labelpad=60, va='center')
axes[1, 0].set_ylabel('INCORRECT', fontsize=12, fontweight='bold', color='red', rotation=0, labelpad=60, va='center')
fig.suptitle('Visual Inspection (T=True, P=Predicted)', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(PLTDIR / '8_sample_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 9: VIOLIN PLOT PROBABILITIES
# ─────────────────────────────────────────────────────────────────────────────────
plot_data = [{'True Class': class_names[i], 'Assigned Probability': prob}
             for i in range(7) for prob in all_pr[all_l == i, i]]
df_probs = pd.DataFrame(plot_data)
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df_probs, x='True Class', y='Assigned Probability', palette=COLORS, inner='quartile', ax=ax)
ax.set_title('Probability Density for True Class', fontsize=14, fontweight='bold')
ax.set_ylim(0,1.05); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLTDIR / '9_probability_violin.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 10: t-SNE FEATURE VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────────
print("⏳ Extracting t-SNE features (may take a minute)...")
feature_tensor = None
def hook_fn(module, inp, out):
    global feature_tensor
    feature_tensor = out
handle = model.norm.register_forward_hook(hook_fn)

tsne_indices = np.random.choice(len(test_s), min(800, len(test_s)), replace=False)
features_list, labels_tsne = [], []
for idx in tsne_indices:
    img_path = test_s[idx][0]
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        _ = model(get_tensor(img))
    features_list.append(feature_tensor[:, 0, :].cpu().numpy()[0])
    labels_tsne.append(all_l[idx])
handle.remove()

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
tsne_results = tsne.fit_transform(np.array(features_list))

fig, ax = plt.subplots(figsize=(10, 8))
for i, cls in enumerate(class_names):
    mask = np.array(labels_tsne) == i
    ax.scatter(tsne_results[mask, 0], tsne_results[mask, 1],
              c=COLORS[i], label=cls, alpha=0.6, s=20, edgecolors='white', linewidth=0.5)
ax.set_title('t-SNE Visualization of Deep Features', fontsize=14, fontweight='bold')
ax.legend(markerscale=2); ax.grid(True, alpha=0.2)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(PLTDIR / '10_tsne_features.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 11: ERROR ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────────
errors = [{'True': class_names[all_l[i]], 'Predicted': class_names[all_p[i]]}
          for i in range(len(all_l)) if all_l[i] != all_p[i]]
if errors:
    error_df = pd.DataFrame(errors)
    error_counts = error_df.groupby(['True', 'Predicted']).size().reset_index(name='Count')
    error_counts = error_counts.sort_values(by='Count', ascending=False).head(10)
    fig, ax = plt.subplots(figsize=(10, 6))
    error_counts['Pair'] = error_counts['True'] + ' → ' + error_counts['Predicted']
    sns.barplot(data=error_counts, x='Count', y='Pair', palette='Reds_r', ax=ax)
    ax.set_title('Top Misclassification Pairs', fontsize=14, fontweight='bold')
    for i, v in enumerate(error_counts['Count']):
        ax.text(v + 0.5, i, str(v), color='black', va='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(PLTDIR / '11_error_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 12: CLASS DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────────
all_samples = train_s + val_s + test_s
df_all = pd.DataFrame(all_samples, columns=['path', 'label'])
counts = df_all['label'].value_counts().reindex(class_names).fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('HAM10000 Dataset Class Distribution', fontsize=16, fontweight='bold')
sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette=COLORS, edgecolor='black')
axes[0].set_title('Image Count per Class', fontsize=13)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold', fontsize=11)
axes[1].pie(counts.values, labels=class_names, autopct='%1.1f%%', colors=COLORS,
           startangle=90, wedgeprops={'edgecolor': 'black', 'linewidth': 1.5},
           textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title('Percentage Distribution', fontsize=13)
plt.tight_layout()
plt.savefig(PLTDIR / '12_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 13: SPLIT DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────────
split_data = []
for split_name, split_samples in [('Train', train_s), ('Validation', val_s), ('Test', test_s)]:
    for p, l in split_samples:
        split_data.append({'split': split_name, 'label': l})
df_splits = pd.DataFrame(split_data)
split_counts = df_splits.groupby(['split', 'label']).size().unstack(fill_value=0)[class_names]

fig, ax = plt.subplots(figsize=(10, 6))
split_counts.plot(kind='barh', stacked=True, ax=ax, color=COLORS, edgecolor='black', linewidth=0.5)
ax.set_title('Class Distribution Across Data Splits', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Images'); ax.set_ylabel('')
for i, split in enumerate(split_counts.index):
    ax.text(split_counts.loc[split].sum() + 10, i, f'Total: {split_counts.loc[split].sum()}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig(PLTDIR / '13_split_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 14: AVERAGE IMAGES PER CLASS
# ─────────────────────────────────────────────────────────────────────────────────
print("⏳ Computing average images...")
fig, axes = plt.subplots(1, 7, figsize=(21, 4))
fig.suptitle('Average Skin Lesion Image per Class', fontsize=14, fontweight='bold')
for i, cls in enumerate(class_names):
    cls_paths = df_all[df_all['label'] == cls]['path'].sample(n=min(300, len(df_all[df_all['label']==cls])), random_state=42).tolist()
    avg_img = np.zeros((224, 224, 3), dtype=np.float32)
    valid_count = 0
    for p in cls_paths:
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        if img is not None:
            avg_img += cv2.resize(img, (224, 224))
            valid_count += 1
    if valid_count > 0:
        axes[i].imshow((avg_img / valid_count).astype(np.uint8))
    axes[i].set_title(f'{cls}'); axes[i].axis('off')
plt.tight_layout()
plt.savefig(PLTDIR / '14_average_images.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 15: PIXEL INTENSITY DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for i, cls in enumerate(class_names):
    cls_paths = df_all[df_all['label'] == cls]['path'].sample(n=50, random_state=42).tolist()
    if not cls_paths: continue
    pixels = []
    for p in cls_paths:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            pixels.extend(np.random.choice(img.flatten(), size=1000, replace=False))
    if pixels:
        sns.kdeplot(pixels, fill=True, color=COLORS[i], label=cls, ax=ax, alpha=0.3, linewidth=2)
ax.set_title('Pixel Intensity Distribution by Class', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLTDIR / '15_pixel_intensity.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#                          PLOT 16: IMAGE DIMENSIONS
# ─────────────────────────────────────────────────────────────────────────────────
dim_data = []
for p in df_all['path'].sample(n=500, random_state=42):
    img = cv2.imread(p)
    if img is not None:
        h, w = img.shape[:2]
        lbl = df_all.loc[df_all['path']==p, 'label'].values[0]
        dim_data.append({'Height': h, 'Width': w, 'Label': lbl})
if dim_data:
    dim_df = pd.DataFrame(dim_data)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Image Dimension Analysis', fontsize=14, fontweight='bold')
    sns.boxplot(data=dim_df, x='Label', y='Height', palette=COLORS, ax=axes[0])
    axes[0].set_title('Image Height')
    sns.boxplot(data=dim_df, x='Label', y='Width', palette=COLORS, ax=axes[1])
    axes[1].set_title('Image Width')
    plt.tight_layout()
    plt.savefig(PLTDIR / '16_image_dimensions.png', dpi=300, bbox_inches='tight')
    plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#               PLOT 17: XAI GRAD-CAM + ATTENTION (5 SAMPLES)
# ─────────────────────────────────────────────────────────────────────────────────
print("\n🔥 Generating XAI Grad-CAM + Attention Overlays...")
class AdvancedXAI:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.grads, self.activations = None, None
        last_conv = None
        for m in self.model.backbone.feature_extractor.modules():
            if isinstance(m, nn.Conv2d):
                last_conv = m
        if last_conv:
            last_conv.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
            last_conv.register_backward_hook(lambda m, gi, go: setattr(self, 'grads', go[0].detach()))

    def generate(self, img_path, target_class):
        rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        orig_h, orig_w = rgb.shape[:2]
        x = get_tensor(rgb).float().requires_grad_(True)
        
        out = self.model(x)
        self.model.zero_grad()
        oh = torch.zeros_like(out)
        oh[0, target_class] = 1
        out.backward(gradient=oh, retain_graph=True)
        
        if self.grads is not None and self.activations is not None:
            weights = self.grads.mean(dim=(2, 3), keepdim=True)
            cam = F.relu((weights * self.activations).sum(1)).squeeze().cpu().numpy()
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8) if cam.max() > cam.min() else np.zeros_like(cam)
        else:
            cam = np.zeros((self.model.spatial_h, self.model.spatial_w))

        with torch.no_grad():
            _ = self.model(x)
        g_attn, _ = self.model.get_attentions()
        attn = np.zeros((self.model.spatial_h, self.model.spatial_w))
        if g_attn:
            a = g_attn[-1]
            attn = a[0, :, 0, 1:].mean(0).cpu().numpy() if a.dim() == 4 else a[0, 0, 1:].cpu().numpy()
            attn = attn.reshape(self.model.spatial_h, self.model.spatial_w)
            attn = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8) if attn.max() > attn.min() else np.zeros_like(attn)

        def overlay(hmap):
            if hmap.max() == 0:
                return rgb.copy()
            hr = cv2.resize(hmap, (orig_w, orig_h))
            colored = cv2.applyColorMap((hr * 255).astype(np.uint8), cv2.COLORMAP_JET)
            return cv2.addWeighted(rgb, 0.6, colored, 0.4, 0)

        comb = 0.5 * cam + 0.5 * attn
        comb = (comb - comb.min()) / (comb.max() - comb.min() + 1e-8) if comb.max() > comb.min() else comb
        return rgb, overlay(cam), overlay(attn), overlay(comb)

xai = AdvancedXAI(model)

# Select sample indices (one correct prediction per class if possible)
sample_indices = []
for cls_idx in range(7):
    mask = (all_p == cls_idx) & (all_l == cls_idx)
    if mask.any():
        sample_indices.append(np.where(mask)[0][0])
sample_indices = sample_indices[:5]

if sample_indices:
    fig, axes = plt.subplots(len(sample_indices), 4, figsize=(16, 4 * len(sample_indices)))
    fig.suptitle('Model Explainability: Grad-CAM vs Attention vs Combined', fontsize=18, fontweight='bold', y=1.01)
    if len(sample_indices) == 1:
        axes = axes.reshape(1, -1)
    for row_idx, idx in enumerate(sample_indices):
        rgb, cam_rgb, attn_rgb, comb_rgb = xai.generate(test_s[idx][0], all_l[idx])
        for col_idx, img in enumerate([rgb, cam_rgb, attn_rgb, comb_rgb]):
            axes[row_idx, col_idx].imshow(img)
            if col_idx == 0:
                axes[row_idx, col_idx].set_title(f"True: {class_names[all_l[idx]]}\nPred: {class_names[all_p[idx]]} ({max_probs[idx]:.1%})", fontweight='bold')
            else:
                axes[row_idx, col_idx].set_title(['Original', 'Grad-CAM', 'Attention', 'Fused'][col_idx], fontweight='bold')
            axes[row_idx, col_idx].axis('off')
    plt.tight_layout()
    plt.savefig(PLTDIR / '17_xai_5_samples.png', dpi=300, bbox_inches='tight')
    plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#               PLOT 18: DEPLOYMENT THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────────
thresholds = np.arange(0.5, 1.0, 0.05)
metrics_data = []
for thresh in thresholds:
    mask = max_probs >= thresh
    if mask.sum() == 0:
        continue
    p_thresh, l_thresh = all_p[mask], all_l[mask]
    metrics_data.append({
        'Threshold': thresh,
        'Accuracy': accuracy_score(l_thresh, p_thresh),
        'F1-Score': f1_score(l_thresh, p_thresh, average='weighted', zero_division=0),
        'Coverage': mask.sum() / len(all_p) * 100
    })
df_thresh = pd.DataFrame(metrics_data)

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.set_xlabel('Minimum Confidence Threshold')
ax1.set_ylabel('Metric Score', color='#3498db')
ax1.plot(df_thresh['Threshold'], df_thresh['Accuracy'], color='#3498db', marker='o', lw=2, label='Accuracy')
ax1.plot(df_thresh['Threshold'], df_thresh['F1-Score'], color='#2ecc71', marker='s', lw=2, label='F1-Score')
ax1.tick_params(axis='y', labelcolor='#3498db')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.set_ylabel('Dataset Coverage (%)', color='#e74c3c')
ax2.plot(df_thresh['Threshold'], df_thresh['Coverage'], color='#e74c3c', marker='^', lw=2, linestyle='--', label='Coverage')
ax2.tick_params(axis='y', labelcolor='#e74c3c')

fig.legend(loc='lower left', bbox_to_anchor=(0.1, -0.1), ncol=3)
plt.tight_layout()
plt.savefig(PLTDIR / '18_deployment_thresholds.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#               PLOT 19: CLASS-SPECIFIC THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
cls_threshs = np.arange(0.4, 1.0, 0.05)
for cls_idx, cls in enumerate(class_names):
    cls_f1s = []
    for t in cls_threshs:
        mask = (all_p == cls_idx) & (max_probs >= t)
        cls_f1s.append(f1_score(all_l[mask] == cls_idx, all_p[mask] == cls_idx, zero_division=0) if mask.sum() > 0 else 0)
    opt_idx = np.argmax(cls_f1s)
    ax.plot(cls_threshs, cls_f1s, color=COLORS[cls_idx], marker='o', lw=2,
            label=f'{cls} (Opt @ {cls_threshs[opt_idx]:.2f})')
ax.set_title('Class-Specific F1 vs. Confidence Threshold', fontsize=14, fontweight='bold')
ax.set_xlabel('Confidence Threshold'); ax.set_ylabel('F1-Score')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLTDIR / '19_class_specific_thresholds.png', dpi=300, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────────────────────
#               PLOT 20: DEEP FEATURE MAPS HIERARCHY
# ─────────────────────────────────────────────────────────────────────────────────
print("\n🧠 Extracting Hierarchical Feature Maps...")
mask_idx = np.where((all_l == 4) & (all_p == 4))[0]  # Melanoma example
if len(mask_idx) == 0:
    mask_idx = [0]

img_path = test_s[mask_idx[0]][0]
img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
x_feat = get_tensor(img)

feature_maps = {}
def get_hook(name):
    def hook(module, input, output):
        feature_maps[name] = output.detach().cpu()
    return hook

model.backbone.feature_extractor[1].register_forward_hook(get_hook('Early_Layer'))
model.backbone.feature_extractor[4].register_forward_hook(get_hook('Mid_Layer'))
model.backbone.feature_extractor[7].register_forward_hook(get_hook('Late_Layer'))

with torch.no_grad():
    _ = model(x_feat)

fig, axes = plt.subplots(3, 8, figsize=(20, 7))
fig.suptitle('Hierarchical Feature Extraction (Melanoma Example)', fontsize=16, fontweight='bold')

for row_idx, (layer, title) in enumerate(zip(['Early_Layer', 'Mid_Layer', 'Late_Layer'],
                                             ['Early Features (Edges)', 'Mid Features (Anatomy)', 'Late Features (Pathology)'])):
    if layer in feature_maps:
        fm = feature_maps[layer].squeeze()
        n_channels = fm.shape[0]
        channel_idxs = np.linspace(0, n_channels - 1, 8, dtype=int)
        for col_idx, c_idx in enumerate(channel_idxs):
            if c_idx < n_channels:
                ch_img = fm[c_idx].numpy()
                ch_img = (ch_img - ch_img.min()) / (ch_img.max() - ch_img.min() + 1e-8)
                axes[row_idx, col_idx].imshow(ch_img, cmap='magma')
                axes[row_idx, col_idx].set_title(f'Ch {c_idx}', fontsize=9)
                axes[row_idx, col_idx].axis('off')
    axes[row_idx, -1].axis('off')
    axes[row_idx, -1].text(1.1, 0.5, title, transform=axes[row_idx, -1].transAxes,
                          fontsize=12, fontweight='bold', va='center', rotation=270)

plt.tight_layout()
plt.savefig(PLTDIR / '20_feature_maps_hierarchy.png', dpi=300, bbox_inches='tight')
plt.show()

# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  EXACT FINAL EVALUATION METRICS BLOCK                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

test_acc = accuracy_score(all_l, all_p)
test_f1 = f1_score(all_l, all_p, average='weighted')
test_auc = roc_auc_score(all_l, all_pr, multi_class='ovr', average='weighted')
test_ap = average_precision_score(labels_bin, all_pr, average='weighted')

mcc = matthews_corrcoef(all_l, all_p)
kappa = cohen_kappa_score(all_l, all_p)
ll = log_loss(all_l, all_pr)

# Fix: Compute Brier score per class and average
brier_per_class = []
for i in range(7):
    brier_per_class.append(brier_score_loss(labels_bin[:, i], all_pr[:, i]))
brier = np.mean(brier_per_class)

# Multi-class Brier score (alternative)
brier_multi = np.mean(np.sum((labels_bin - all_pr)**2, axis=1))

# Calculate Macro metrics over 7 classes
sens_list = [cm[i,i]/(cm[i,:].sum()+1e-8) for i in range(7)]
spec_list = [(cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]) / (cm.sum() - cm[i,:].sum() + 1e-8) for i in range(7)]
prec_list = [cm[i,i]/(cm[:,i].sum()+1e-8) for i in range(7)]
npv_list = [(cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]) / (cm.sum() - cm[:,i].sum() + 1e-8) for i in range(7)]

print("\n============================================================")
print("📊 COMPREHENSIVE EVALUATION METRICS")
print("============================================================")

print(f"\n🎯 PRIMARY METRICS:")
print(f"   Accuracy:  {test_acc:.4f}")
print(f"   F1-Score:  {test_f1:.4f}")
print(f"   AUC-ROC:   {test_auc:.4f}")
print(f"   AP Score:  {test_ap:.4f}")

print(f"\n📈 ADDITIONAL METRICS:")
print(f"   MCC:       {mcc:.4f}")
print(f"   Cohen's Kappa: {kappa:.4f}")
print(f"   Log Loss:  {ll:.4f}")
print(f"   Brier Score (per-class avg): {brier:.4f}")
print(f"   Brier Score (multi-class):   {brier_multi:.4f}")

print(f"\n🔬 PER-CLASS METRICS (HAM10000 7-Classes Macro Avg):")
print(f"   Sensitivity (Recall): {np.mean(sens_list):.4f}")
print(f"   Specificity:          {np.mean(spec_list):.4f}")
print(f"   Precision:            {np.mean(prec_list):.4f}")
print(f"   NPV:                  {np.mean(npv_list):.4f}")
print(f"   FPR:                  {1 - np.mean(spec_list):.4f}")
print(f"   FNR:                  {1 - np.mean(sens_list):.4f}")

print(f"\n📊 CONFUSION MATRIX:")
header = f"{'':>18s}" + "".join([f"{n:>10s}" for n in class_names])
print(header)
for i, cls in enumerate(class_names):
    row = f"{cls:>18s}" + "".join([f"{cm[i,j]:>10d}" for j in range(7)])
    print(row)

print(f"\n📋 CLASSIFICATION REPORT:")
print(classification_report(all_l, all_p, target_names=class_names))

print("="*60)
print(f"✅ ALL 20 PLOTS GENERATED SUCCESSFULLY FOR HAM10000!")
print(f"📁 Location: {PLTDIR}")
print("="*60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — Download Entire Kaggle Working Space (FIXED)                        ║
# ╚══════════════════════════════════════════════════════════════════════════════════╝

import shutil
from IPython.display import FileLink

def download_workspace(output_name='skincancer.zip'):
    """Zip the entire /kaggle/working directory and provide download link."""
    
    zip_path = OUTDIR / output_name
    
    # Remove old zip if exists
    if zip_path.exists():
        zip_path.unlink()
    
    print("📦 Scanning /kaggle/working for files to archive...")
    
    total_size = 0
    for item in OUTDIR.rglob('*'):
        if item.is_file() and item != zip_path:
            size_mb = item.stat().st_size / (1024 * 1024)
            total_size += size_mb
            if size_mb > 0.1:
                rel_path = item.relative_to(OUTDIR)
                print(f"      📄 {rel_path}: {size_mb:.2f} MB")
    
    print(f"\n   Total uncompressed size: {total_size:.2f} MB")
    
    # ✅ FIXED: Removed 'with' statement
    print("\n🔒 Compressing... (this may take a minute for large models)")
    archive_name = str(zip_path.with_suffix(''))
    shutil.make_archive(archive_name, 'zip', OUTDIR)
    
    # Re-find the zip in case of naming quirks
    final_zip = Path(archive_name + '.zip')
    final_size = final_zip.stat().st_size / (1024 * 1024)
    
    print(f"✅ Archive created successfully!")
    print(f"   File: {final_zip.name}")
    print(f"   Size: {final_size:.2f} MB")
    
    # Provide download link
    print("\n⬇️ Click the link below to download:")
    display(FileLink(str(final_zip), result_html_prefix="🔗 Download Link: "))
    
    return str(final_zip)

# Run it
zip_file = download_workspace()